<a href="https://colab.research.google.com/drive/1q6r7tg8RoyLBaMS3b7EONwkPCBzEM2c1?usp=sharing" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


---
# **VLSI 2026 Code-a-Chip Competition**  
---

# **$R_{on}/g_m$ Based Design methodology for Dynamic Amplifiers**

### A Proof of Concept for $R_{on}/g_m$ based design methodology for Dynamic Amplifier

 ---

## Team Overview

| Name | Affiliation | IEEE Member | SSCS Member | Email |
|:--|:--|:--:|:--:|:--|
| **Nithin P** | B.Tech, VTU | Yes | Yes | [nithinpurushothama@gmail.com](mailto:nithinpurushothama@gmail.com) |
| **Pramoda S R** | B.Tech, VTU | No | No | [pramoda9.2.2004@gmail.com](mailto:pramoda9.2.2004@gmail.com) |
| **Praveen Kumar Venkatacahala** | PhD, Oregon State University | Yes | Yes | - |
| **Runpeng Gao** | PhD, Oregon State University | Yes | Yes |  [rppgao@gmail.com](mailto:rppgao@gmail.com) |
| **S Suyajnaa Jagannath Gowda** | B.Tech, VTU | No | No | [suyajnaa@gmail.com](mailto:suyajnaa@gmail.com) |

---

This work is Licensed under **Apache 2.0**

---


## **How to Run This Notebook**

---

### **Recommended: Run in Google Colab (Zero Setup Required)**

**The reviewers are highly encouraged to use the Google Colab version of this work. It requires no setup and is very user-friendly.**

**Step 1: Open in Colab**  
Click the **"Open in Colab"** badge at the very top of this notebook.  
This opens the notebook in Google's free cloud environment, no installation needed.

**Step 2: Sign in**  
Sign in with any Google account if prompted. A free account is sufficient.

**Step 3: Run Everything at Once**  
In the top menu, click **Runtime → Run all** (or press `Ctrl+F9`).  
Colab will ask *"Warning: this notebook was not authored by Google"*, click **Run Anyway**.

**Step 4: Wait (~10-12 minutes total)**  
The notebook builds ngspice from source and runs 15 SPICE simulations automatically.  
A progress table prints live as each simulation finishes. **Do not click anything while it runs.** Please read through the contents while things run in the background.

| Phase | What happens | Time |
|:--|:--|:--|
| Steps 1–3 | Install build tools, download model files, compile PSP model | ~2 min |
| Step 4 | Build ngspice 45.2 with OSDI support | ~9 min |
| Steps 5–6 | Generate netlists, run 15 parametric SPICE simulations | ~1 min |
| Steps 7+ | Load data, generate design plots, interactive dashboards | ~30 s |

**Step 5: Interact with the Results**  
Once complete, scroll down to explore:
- **6 Ron/gm Design Plots** - hover over traces for detailed values
- **Gm/Id Design Helper** - drag sliders to explore the design space
- **Ron/Gm Design Helper** - select device, corner, and bias; see the best match highlighted in green

---

### Local / Jupyter Setup

Install dependencies first:
```bash
pip install plotly pandas numpy scipy gdown requests kaleido ipywidgets
```
Then run cells top-to-bottom. An internet connection is required for model files and simulation data.  
Update `data_path` in necessary cells to point to your local CSV directory if not using Colab.

---
**Google Colab Link**: https://colab.research.google.com/drive/1q6r7tg8RoyLBaMS3b7EONwkPCBzEM2c1?usp=sharing

---


 # **Abstract**
 ---
 This work presents an $R_{on}/g_m$ based design methodology for Inverter based dynamic amplifiers(IBA), addressing a fundamental gap in existing approaches where the large-signal RC settling phase governed by the final stage device ON resistance $R_{on}$ remains uncharacterized until post-simulation. Unlike the conventional $g_m/I_D$ methodology, which targets only the small-signal transconductance, the proposed approach simultaneously co-designs both settling phases through pre-characterized device look-up tables (LUTs) derived from parametric SPICE simulations in the IHP SG13G2 130 nm BiCMOS process. These LUTs take into consideration of $R_{on}/g_m$ as a function of device geometry, bias, and process corner, making worst case corner behavior and valid bias deadzone boundaries directly readable at the design entry stage without iterative simulation. A head to head comparison with the $g_m/I_D$ methodology confirms that the $R_{on}/g_m$ approach achieves equivalent settling accuracy while substantially reducing design cycles and providing more useful information regarding deadzone bias requirement by surfacing process-corner sensitivity upfront.

 ---

## Table of Contents
---

1. Introduction
2. Dynamic Amplifier
   - 2.1 Understanding Dynamic Amplifiers
   - 2.2 Error Behavior of Dynamic Amplifiers
   - 2.3 Ron/gm Based Design Methodology
3. Simulation Environment Setup
   - 3.1 Automated SPICE Netlist Generation
   - 3.2 Step 1 - Build Ngspice 45.2
   - 3.3 Step 2 - Download IHP SG13G2 Models
   - 3.4 Step 3 - Compile PSP 103.6 NQS
   - 3.5 Step 4 - Write .spiceinit
   - 3.6 Step 5 - Simulation Configuration
   - 3.7 Step 6 - Run All 15 Simulations
4. Design Plots for the Ron/gm Methodology
   - 4.1 $V_{bias}$ vs log($R_{on}$/$g_m$)
   - 4.2 $V_{bias}$ vs Width
   - 4.3 log($I_{peak}$) vs log($R_{on}$/$g_m$)
   - 4.4 $g_{m,bias}$ vs log($R_{on}$/$g_m$)
   - 4.5 $g_m$/$I_d$ vs log($R_{on}$/$g_m$)
   - 4.6 $V_{swing}$ vs log($R_{on}$/$g_m$)
5. IBA Design: gm/ID Methodology
   - 5.1 Gm/ID Characterisation and Design Helper
   - 5.2 Results from the IBA designed using gm/ID
   - 5.3 Summary and Motivation for RAMPA
6. IBA Design: Ron/gm Methodology
   - 6.1 Ron/gm Design Helper
   - 6.2 Results from the IBA designed using Ron/gm
7. Comparative Analysis
8. References

---

# **1. Introduction**
---

Modern electronic systems from smartphones to medical imaging to high-speed communications rely on <b>Analog-to-Digital Converters (ADCs)</b> to bridge the continuous physical world and the discrete digital domain.
At the heart of almost every high-performance ADC sits an **amplifier**: it is the component responsible for accurately transferring a sampled charge from one capacitor to another within a tight time budget (one clock half-cycle).

In **switched-capacitor circuits** the dominant implementation style for precision ADCs - the amplifier must:

* Drive a capacitive load $C_L$ to a precise output voltage $V_{out}$,
* **Settle to within the required accuracy** (e.g., $< 0.1\,\text{LSB}$) before the next clock edge,
* Do so while consuming as little power as possible.

The settling speed is set by the amplifier's **unity-gain bandwidth** $f_u \approx g_m / (2\pi C_L)$, where $g_m$ is the transconductance. More $g_m$ means faster settling, but also more bias current and therefore more power. This fundamental **speed–power trade-off** is at the core of every amplifier design.

The classical solution is an **Operational Transconductance Amplifier (OTA)** a differential pair with a current mirror load. The OTA works by converting an input voltage difference into an output current through its transconductance $g_m$.

$$\boxed{I_{out} = g_m \cdot \Delta V_{in} }$$

then integrating that current onto the load capacitor:

$$\boxed{V_{out}(t) = \frac{1}{C_L}\int I_{out}\, dt}$$

Together these produce the familiar **exponential settling**:

$$\boxed{V_{out}(t) = V_{final}\left(1 - e^{-t/\tau}\right), \quad \tau = \frac{C_L}{g_m}}$$

This is **purely small-signal, exponential settling**, the output creeps toward its final value, and the designer must wait many time constants $\tau$ to reach the required accuracy.

As CMOS technologies scale to smaller nodes (130 nm, 65 nm, 28 nm and below), the OTA runs into fundamental walls:

| OTA Limitation | Root Cause | Impact |
|:--|:--|:--|
| **Lower intrinsic gain** $g_m r_o$ | Shorter $L$ → lower $r_o$ | Worse closed-loop accuracy |
| **Reduced voltage headroom** | Lower $V_{DD}$ at scaled nodes | Restricted output swing |
| **Poor process scalability** | Shorter $L$ degrades $g_m r_o$ faster than $f_T$ improves **[5]** | Performance worsens with each technology node |
| **Poor power efficiency** | Must burn static bias current at all times | Energy wasted even during idle phases |

**OTAs were designed for an era of high supply voltages and long channel devices. They do not scale to modern process nodes.**

---

### The Classic Amplifier Trade-off

In practice, the speed-power tradeoff is only one face of a four-dimensional constraint that governs every analog amplifier design. The four axes **Speed/Bandwidth**, **Power**, **Noise/Accuracy**, and **Area** are mutually coupled in a linear amplifier: every transistor must be sized and biased to simultaneously satisfy the needed requirements, and the costs compound. The OTA sits **high on all four axes at once** it always draws the full static bias current (paying the power cost), requires large devices for low noise (paying the area cost), and is still bandwidth-limited by the same $g_m$ to current ratio (paying the speed cost). In a conventional OTA, all four axes are rigidly coupled push any one and the others move with it, with no degree of freedom to break the classic tradeoff **[6]**.

In the Classic Linear settling based Amplifier:

* **Speed vs. Power**: Faster settling demands more $g_m$ → more bias current → more static power.
* **Noise vs. Area**: Lower thermal noise requires larger transistors ($\text{noise} \propto kT/C$) → more area **[7]**.
* **Speed vs. Area**: Wider transistors add parasitic capacitance → need even more $g_m$ to compensate → back to more power.
* **Power vs. Noise**: Higher $I_D$ lowers the noise floor - but only by burning more power.

<div align="center">
  <img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/all_figs/Four_way_amp_tradeoff.png" width="900">

  **Figure 1:** Classic Amplifier Tradeoff Quadrilateral.

</div>

In a conventional OTA, the designer cannot escape this coupling. The design space is **bounded and high-cost on every axis simultaneously**.

> **How dynamic amplifiers change this:** Dynamic amplifiers **achieve lower power, lower noise, smaller area, and better speed** by **shrinking all four axes of the trade-off quadrilateral simultaneously**, rather than trading one against another. The key insight is that a dynamic amplifier **does not pay the static bias current cost at all times**. An OTA is always on always burning current, always generating noise, always occupying area tuned for worst-case signal conditions. A dynamic amplifier is quiescent until a clock edge fires it. It then draws current only during the settling transient, uses the **large-signal slewing phase** (high speed, very efficient per unit charge delivered) to do most of the work, and then settles precisely in the short small-signal phase. Because the large-signal phase is inherently nonlinear and event-driven, it is decoupled from the $g_m$–current–noise–area coupling that governs linear operation. The net result: dynamic amplifiers achieve **lower power, lower noise, smaller area, and better speed than an OTA at the same specification**. This is exactly why dynamic amplifiers become more attractive as CMOS scales since inverter-based structures inherently improve with every process node, unlike OTAs which degrade.
---

# **2. Dynamic Amplifier**
---
## **2.1 Understanding Dynamic Amplifiers**
---
Dynamic amplifiers takes a different approach. Instead of biasing transistors into a constant small-signal operation, they **exploit the full large-signal drive capability of the transistor** during the settling transients.

The core idea is to: **start with maximum current to charge the load capacitor and then taper off.**

Consider an inverter (a PMOS and NMOS in series) driving a capacitive load. If the input is near the trip point $V_{tp}$:

<div align="center">

<img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/ckt_images/Fig_4_sch_of_IBA.jpg" width="350">

**Figure 2:** Schematic of the Inverter-Based Amplifier (IBA) **[8]**.

</div>

- Both transistors are partially on, but neither is in deep saturation.
- A small input change causes a **large differential(among PMOS and NMOS) output current** → this is the large-signal phase.
- As $V_{out}$ approaches its final settling value, $V_x$ settles to the optimal bias point where PMOS and NMOS currents balance, the net output current driving the load capacitor goes to zero, and the circuit enters **small-signal mode** where a quiescent current flows through both devices → this is the small-signal settling phase.

---

### The Deadzone Voltage

The transition from large-signal to small-signal settling is governed by the **deadzone voltages** $V_{DZP}$ and $V_{DZN}$. These are the gate bias voltages applied to the output-stage PMOS and NMOS respectively, chosen such that both transistors sit at the edge of saturation with balanced drain currents the net output current $I_{out} = 0$ and the output impedance $R_{out}$ is maximised. This defines the **deadzone**: the input-referred window around which the amplifier behaves as a high-gain small-signal stage with a dominant output pole.

$$\boxed{I_{out} = 0, \quad R_{out} \to \text{max} \quad \Longleftrightarrow \quad V_{in} \in [V_{DZN},\, V_{DZP}]}$$

Outside the deadzone (during the large-signal phase), both transistors are driven hard by the large input overdrive and deliver maximum charging current to $C_L$. Once $V_{out}$ enters the deadzone, the circuit self-quenches into small-signal mode and precision settling begins. The width of the deadzone therefore directly controls the **accuracy–speed trade-off**: a narrower deadzone gives higher output impedance and better accuracy but less margin for process variation.

> In the IBA(Inverter based amplifier), the deadzone arises implicitly from the inverter trip point, the same node sets both the large-signal drive and the small-signal bias, leaving no independent handle on $V_{DZP}$/$V_{DZN}$. The Ring Amplifier (Section 2.1.2) resolves this by using dedicated driver stages to set the deadzone voltages explicitly and independently.

---

<div align="center">

<img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/ckt_images/Fig_5_Transient_response_of_IBA.png" width="600">

**Figure 3:** Transient Response of the Inverter-Based Amplifier (IBA) **[2]**.
</div>





In [ ]:
# Interactive Visualization: IBA Transient Response
# RAMPA - Ron/gm-based Design Methodology · CAC 2025
#
# Fetches the animation HTML directly from GitHub - no inline code needed.

import requests
from IPython.display import HTML, display

RAW_URL  = "https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/iba_transient_v6.html"
TOTAL_H  = 665   # total height in px  ← change this to resize

MAIN_H = TOTAL_H - 100   # 48px header + 52px controls = 100px overhead

# Fetch HTML from GitHub
response = requests.get(RAW_URL, timeout=10)
response.raise_for_status()
html = response.text

# Patch standalone (100vh) CSS → fixed Colab-compatible heights
html = html.replace(
    'html,body{width:100%;height:100%;overflow:hidden;background:var(--bg);}',
    f'html,body{{width:100%;height:{TOTAL_H}px;background:var(--bg);}}'
)
html = html.replace(
    'body{font-family:\'Rajdhani\',sans-serif;color:#c8e0f8;display:flex;flex-direction:column;height:100vh;}',
    f'body{{font-family:\'Rajdhani\',sans-serif;color:#c8e0f8;display:flex;flex-direction:column;height:{TOTAL_H}px;}}'
)
html = html.replace(
    '#main{flex:1;display:flex;min-height:0;overflow:hidden;}',
    f'#main{{flex:1;display:flex;min-height:0;overflow:hidden;height:{MAIN_H}px;}}'
)

display(HTML(html), metadata=dict(isolated=True))

<div align="center">

**Figure 4:** Animation of the Transient Response of the Inverter-Based Amplifier (IBA) with Step Input $V_X$ and deadzone of $V_{DZP}$(0.4V), $V_{DZN}$(0.4V). We observe $V_{out}$ and $I_{out}$ during Linear and Non Linear settling phase of the Dynamic amplifier(Behavioural model).

</div>

This two-phase behavior means the output **reaches closer to its final value in the first phase, then settles precisely in the second**, achieving both speed and accuracy that a **pure OTA cannot provide at the same power budget**.


---
  ## 2.2 Error Behavior of Dynamic Amplifiers
---

The dynamic behavior of amplifiers can be better understood by analyzing the **dynamic settling error over time**.


The dynamic error is defined as:

$$
\epsilon_{\text{dynamic}} = \frac{V_{out}(t) - V_{out,\text{final}}}{V_{out,\text{final}}}
$$

and is given by:

$$
\epsilon_{\text{dynamic}} = e^{-t/\tau}
$$

We can see the clear difference in settling behaviour with Log scale plot of Dynamic error. Traditional OTAs exhibit a **single-pole dynamic error response**, resulting in a uniform exponential decay as the system settles toward its final value.

In contrast, **dynamic amplifiers** demonstrate a **two-phase error decay** due to their hybrid **large-signal(Faster)** and **small-signal(Slower)** operation:

* **Fast decay** dominated by non-linear, large-signal dynamics that rapidly drive the output toward the final value.
* **Slow decay** governed by linear, small-signal settling that settles the output to final value.

<br>

<div align="center">

<img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/ckt_images/fig_3_dynamic_settling_error_NL_Linear_settling_behaviour.png" width="900">

**Figure 5a :** Modelled dynamic settling error of a Dynamic and linear amplifier, illustrative simulation using analytical two-phase and single-pole models respectively.

</div>

The dynamic amplifier exhibits a two-phase settling behavior. In the initial non-linear (large-signal) phase, the dynamic error decays rapidly, enabling fast convergence toward the target output. This is followed by a linear (small-signal) phase, where the remaining error is reduced more gradually, ensuring accurate final settling.

As shown in **Fig. 5a**, the dynamic amplifier achieves significantly faster settling (≈ 2.113 µs) compared to a purely linear settling amplifier such as an OTA (≈ 3.445 µs) under the same power budget. This improvement arises from the multi-phase nature of dynamic operation: the large-signal phase accelerates the bulk of the transition, while the small-signal phase refines the output to its final value.

Overall, this hybrid settling mechanism allows dynamic amplifiers to combine speed and precision more effectively than traditional linear OTAs.


In [ ]:
# Interactive Visualization: Amplifier Settling Behaviour
# Dynamic Amplifier vs. Linear Amplifier · CAC 2025
# IHP SG13G2 130nm BiCMOS · ngspice
#
# Fetches the animation HTML directly from GitHub.
# Push settling_analyser_CAC2025.html to your repo and update RAW_URL below.

import requests
from IPython.display import HTML, display

RAW_URL = "https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/Amplifier_Settling_Behaviour_Analyser.html"
TOTAL_H = 540   # total height in px  ← change to resize
# ──────────────────────────────────────────────────────────

MAIN_H = TOTAL_H - 80   # header (~40px) + footer (~12px) + transport bar (~28px)

# Fetch HTML from GitHub
response = requests.get(RAW_URL, timeout=10)
response.raise_for_status()
html = response.text

# Patch standalone CSS → fixed Colab-compatible heights
html = html.replace(
    'html,body{width:100%;height:100%;overflow:hidden;',
    f'html,body{{width:100%;height:{TOTAL_H}px;overflow:hidden;'
)
html = html.replace(
    '#app{display:flex;flex-direction:column;height:100vh;overflow:hidden}',
    f'#app{{display:flex;flex-direction:column;height:{TOTAL_H}px;overflow:hidden}}'
)

display(HTML(html), metadata=dict(isolated=True))

<div align="center">

  **Figure 5b:** Modelled Settling Behaviour of Dynamic and Linear Amplifier

</div>

Dynamic amplifier architectures that exploit this principle include:

* **Inverter-Based Amplifiers (IBAs)** - the simplest form, a CMOS inverter with capacitive feedback **[1]**.
* **Ring Amplifiers (RAMPs)** - a cascaded ring of inverter-like stages with controlled deadzone bias voltages $V_{GP}$, $V_{GN}$ to set the small-signal settling window **[2]**.
* **Zero-Crossing Detector Amplifiers (ZCDs)** - detect the moment the output crosses a threshold and switch off the driving current at that instant **[3]**.

<div align="center">

<img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/ckt_images/fig_1_Ron_Gm_IBA_RAMP_ZCD.png" width="1100">

**Figure 6:** (i) Inverter-Based Amplifier **[1]**, (ii) Ring Amplifier **[2]**, and (iii) Zero-Crossing Detector Amplifier **[3]**.

</div>

---

### The Ring Amplifiers

---

Among dynamic amplifier architectures, the **Ring Amplifier (RAMP)** introduced by Hershberg et al. **[1]** is one of the most widely adopted dynamic amplifier topologies. A RAMP consists of a cascade of 3 inverter-like stages, where the first two stages drive the gates ($V_{GP}$ and $V_{GN}$) of the output-stage transistors. The bias voltages $V_{DZP}$ and $V_{DZN}$ applied to the output stage define the **dead-zone**: the input-referred window around which the net output current charging the load capacitor is zero. Within this window, both the PMOS and NMOS of the output stage carry a quiescent current $I_Q$, establishing a high output impedance and placing the dominant pole at the output of the RAMP. This dead-zone is the key mechanism that governs both **stability** and **settling accuracy**.

As described in Praveen Kumar et al. **[2]**, the overall transient response of a Ring Amplifier is divided into **three distinct dynamic settling phases**:

* **Initial RC settling** - for a large-signal step input, the output of the second stage ($V_{GP}$) is driven close to rail-to-rail. This places a large overdrive on the output-stage PMOS, whose gate-source voltage remains approximately fixed while the drain-source voltage changes. The output current during this phase is governed by the $R_{ON}$ of the output-stage PMOS, charging the load capacitance with an impulse-like current.

* **Large-signal settling** - as feedback from the output reaches back to $V_{GP}$, the gate-source voltage begins to change while the drain-source voltage has nearly settled. The output current during this phase is a function of the large-signal $G_m$ of the ring amplifier, determined by the combined gain of the first two stages and the non-linear $G_m$ of the output stage.

* **Small-signal settling** - as $V_{out}$ approaches its final settling value, $V_x$ settles to the optimal bias point where PMOS and NMOS currents balance. The net output current driving the load capacitor goes to zero and the circuit enters small-signal operation, with a quiescent current $I_Q$ flowing through both output-stage devices. The settling during this phase is governed by the small-signal $g_m$.

For the same average current consumption as a conventional OTA, the initial RC settling and intermediate large-signal settling phases together boost the speed of the Ring Amplifier significantly. Equivalently, for the same settling time as an OTA, these two phases reduce the average current consumption of the RAMP directly improving the power-speed trade-off.

---

### Prior Work: Optimizer-Based Design of Dynamic/Ring Amplifiers

---

Despite the advantages of dynamic amplifiers, their design has historically demanded time-consuming iterative transient simulations, since conventional AC-based design criteria have limited applicability given the large-signal nature of dynamic amplifier operation. Conrad et al. **[4]** address this challenge by introducing an **optimizer-driven design methodology** for Ring Amplifiers, structured around two test conditions and a cost function that together automate transistor sizing.

The methodology proceeds as follows. A **unit inverter** is first designed shared across stages $A_1$, $A_2$, and $A_3$ with $L_{NMOS} = L_{min}$ for $A_1$ and $A_2$, and a longer $L$ for $A_3$ to ensure large output impedance . The NMOS width is initialized to a manually chosen reasonable value, $V_{DD}$ is fixed manually, and a target $V_{cm}$ is set. This unit inverter is then replicated with multipliers to scale the stages.

Two test conditions drive the optimization loop:

* **Test (A):** $V_{in}$ and $V_{out}$ are both shorted and forced to the trip point and held fixed. The small-signal parameters of the ring amplifier are extracted and checked against user-specified targets (gain, bandwidth, phase margin).
* **Test (B):** Only $V_{in} = V_{cm}$ is applied. The optimizer checks whether $V_{out} = V_{cm}$, confirming that the amplifier is biased at the desired common-mode operating point.

<div align="center">
  <img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/ckt_images/conrad_et_al_tests.png" width="600"/>
  <br>
  <em>Fig. 7: Test conditions used in the Conrad et al. optimizer-based Ring Amplifier design methodology [4]. (A) Both $V_{in}$ and $V_{out}$ shorted to trip point for small-signal parameter extraction. (B) Only $V_{in} = V_{cm}$ applied to verify common-mode output. (C) Maximum slew current evaluation to account for $I_{max,P} \neq I_{max,N}$ asymmetry.</em>
</div>

The cost function sweeps $V_{cm}$, $V_{DZP}$, $V_{DZN}$ (with $V_{DZP} = V_{DZN}$), and the widths of the PMOS ($M_P$) and NMOS ($M_N$), checking whether the resulting sizes are close to an integer multiplier of the unit cell and whether the small-signal parameters at Test (A) match the user specifications. The candidate sizes are then applied to Test (B): if $V_{out} = V_{cm}$ is satisfied, the sizes are accepted; otherwise the optimizer iterates. A third condition **(C)** additionally evaluates the maximum slew current of the PMOS and NMOS output devices and accounts for the asymmetry when $I_{max,P} \neq I_{max,N}$.

The key limitation of this approach is that the entire methodology is fundamentally iterative cycling through three test conditions, sweeping parameters, and parsing simulation data without any direct connection to the underlying three dynamic settling phases. The optimizer treats the ring amplifier as a black box: it checks whether small-signal parameters pass a cost function, but provides no pre-simulation insight into $R_{ON}$, the RC settling phase, the large-signal $G_m$ phase, or how these behave across process corners. Critically, even after the optimizer converges on a solution, the designer must still run and iterate on full transient simulations to verify actual settling performance, the same manual iterative loop that existed before, just with an automated parameter sweep layer on top. The root problem remains unsolved: there is no quantitative, first-principles handle that connects transistor sizing decisions directly to dynamic settling behavior before simulation.

**Our work addresses this gap.** Rather than relying on an iterative simulation-based optimizer that is dependant on an empirical starting point, we propose a **pre-simulation design methodology** using the **$R_{on}/g_m$** parameter. This approach gives the designer quantitative insight into both the RC settling phase (governed by $R_{on}$) and the small-signal settling phase (governed by $g_m$), **enabling corner-aware transistor sizing directly from lookup tables *without any iterative SPICE runs***. The $R_{on}/g_m$ curves provides a starting point that [4] had to approximate, and in doing so render the optimizer loop itself unnecessary for the sizing step.

---



## 2.3 $R_{on}/g_m$ Based Design Methodology

---

In Dynamic amplifiers, the settling behavior is influenced by both **non-linear** and **linear** settling phases. Each of these contributes significantly to the overall settling performance of the Dynamic amplifier and must be accounted for during the design process as well.

---

### Non-Linear Settling Phase

During the initial transient, an impulse current drives $V_{out}$ rapidly toward its Final Voltage ($V_{out,final}$) by charging the load capacitor through the transistor's on-resistance $R_{on}$. This is modeled as **RC-type settling**, where R is signal-dependent.

- The current during this phase is dictated by the **on-resistance** ($R_{on}$) of the transistor. The $R_{on}$ is Non-Linear in nature due to its signal dependance.
- This phase is critical for fast initial movement of $V_{out}$ toward the final output voltage.

### Linear (Small-Signal) Settling Phase

- Once the output is near its final value, **$G_m C$-type small signal settling** takes over. Transconductance $G_m$ determines the fine settling accuracy and closed-loop bandwidth.

- This determines final settling residue and bandwidth
- This is well-captured by classical $g_m/I_D$ methodology

---

### Design Motivation

The classic **$g_m/I_D$ based methodology** is powerful, but it comes with limitations:
- It does **not account for the signal swing requirements**, which results in incorrect operating point assumptions.
- It does **not provide direct information about DC bias ranges**.
- It’s primarily tuned for small-signal operation and overlooks large-signal contributions during design.
- The Variation of $R_{on}$ across isnt captured by the $g_m/I_D$ methodology which can silently degrade Slow-corner settling.

> **Key insight:** A circuit with well-sized $g_m$ can still exhibit slow settling if $R_{on}$ is not explicitly constrained, this failure is invisible in a $g_m/I_D$ based flow.

To overcome this, we propose using the **$R_{on}/g_m$** ratio as a design-driven metric.

- The proposed **$R_{on}/g_m$** based methodology should provide information regarding the voltage step that needs to be provided.
- The proposed **$R_{on}/g_m$** should provide additional information about Non-linear settling and DC bias(Deadzone ranges).

---
### The $R_{on}/g_m$ Metric

The ratio $R_{on}/g_m$ simultaneously captures:

- **Large-signal settling speed** via $R_{on}$
- **Small-signal bandwidth** via $g_m$

This enables early, pre-simulation verification of settling behavior across process corners.

---

### $R_{on}/g_m$ Based Design Flow

#### **Step 1**: Size the Unit Cell

  - Make a **unit cell** that is independent of C<sub>load</sub> design, but based on **voltage swing requirements**, **Large-signal speed**, **Small-signal requirement** and **Current Consumption**.

#### **Step 2**: Pick $R_{on}$ First

- Decide what **fraction of total settling time** $T_{settle}$ is allocated to the **non-linear (large-signal)** phase vs. the **small-signal (linear)** phase. This split is the key design knob. Here $\alpha$ denotes the fraction reserved for the small-signal phase, so $(1-\alpha)$ is the non-linear budget.

- The empirical time constants are:
  - Non-linear phase: $\tau_{NL} = R_{on} C_{load}$ → set by the switch resistance
  - Small-signal phase: $\tau_{SS} = C_{load}/g_m$ → set by the transconductance

- **Size $R_{on}$** from the non-linear settling budget:

$$R_{on} = \frac{(1-\alpha) \cdot T_{settle}}{5 \cdot C_{load}}$$

  where $(1-\alpha)$ is the time fraction allocated to non-linear settling (factor of 5 for ~99% settling in that phase). The required $R_{on}$ for a given $C_{load}$ then determines the **width scaling multiplier $M$**.

- **Size $g_m$** to satisfy **both** constraints — take the larger of the two:
  - From small-signal BW spec (application requirement): $g_{m,BW} = 2\pi\,f_{BW}\,C_{load}$
  - From settling budget (hyper-optimized lower bound): $g_{m,min} = \dfrac{5\,C_{load}}{\alpha \cdot T_{settle}}$

$$g_m = \max\,(g_{m,min},\; g_{m,BW})$$

  Designing only for $\alpha \cdot T_{settle}$ gives the minimum $g_m$ that just completes settling (hyper-optimized), but most applications impose a small-signal BW requirement that demands a larger $g_m$. The final $g_m$ is set by whichever constraint is tighter.

  > *Example*: $T_{settle} = 1\,\mu\text{s}$, $\alpha = 0.5$, $C_{load} = 1\,\text{pF}$, $f_{BW} = 10\,\text{MHz}$.
  > $R_{on} = \frac{0.5\,\mu\text{s}}{5 \times 1\,\text{pF}} = 100\,\text{k}\Omega$.
  > $g_{m,min} = \frac{5 \times 1\,\text{pF}}{0.5\,\mu\text{s}} = 10\,\mu\text{S}$, $\quad g_{m,BW} = 2\pi \times 10\,\text{MHz} \times 1\,\text{pF} \approx 63\,\mu\text{S}$ → use $63\,\mu\text{S}$.
  
#### **Step 3**: Scale the Unit Cell by $M$


<div align="center">

<img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/ckt_images/unit_cell.png" width="350">

</div>


- Scaling the unit cell by multiplier (M) gives:
  - $R_{on} \rightarrow R_{unit}/M$
  - $g_m \rightarrow g_{m,unit} \times M$
  - $I_D \rightarrow I_{D,unit} \times M$

#### **Step 4**: Tune $g_m$ via the Replica Path if needed

- The *replica path* which is used to Bias the signal path devices uses:
  - $R_{unit}$, $I_{unit}$, $W_{unit}$
  - Provides *replica tuning flexibility* through $I_{unit}$
  - *$g_m$ tuning* can be achieved by adjusting $I_{unit}$ and $W_{unit}$ across corners, without changing the main signal path device width $W \times M$

This decouples $g_m$ tuning from the fixed $R_{on}$ constraint, preserving large-signal settling while correcting small-signal bandwidth.

---

### Key Insights

- **$R_{on}$ is a great design knob**, rather than a post-simulation tuning.
- With fixed $g_m/I_D$, $R_{on}$ can be significantly worse in the slow corner affecting large-signal settling even when $g_m$ is nominally correct.
- The same $g_m$ can be achieved across corners in subthreshold with fixed $I_D$, but $R_{on}$ will diverge this is only visible with the $R_{on}/g_m$ metric.
- The replica path provides a tuning handle for $g_m$ that is orthogonal to the main path sizing, enabling corner-robust design without re-sizing the signal transistors.
- The $R_{on}/g_m$ flow enables **pre-simulation corner verification**, a capability absent from the classical $g_m/I_D$ methodology.
---

# 3. Simulation Environment Setup

---

## 3.1 Automated SPICE Netlist Generation
---
This section demonstrates *automated generation of multiple SPICE netlists*
to explore design variations across **device lengths (L)** and **process corners (TT, SS, FF)**.

---

### Objective
In many simulations, we often need to simulate the same circuit under:
- Different **channel lengths (L)**, and
- Different **process corners** (TT, SS, FF).

Doing this manually in Ngspice is time-consuming and error-prone.  
This Python script automates:
1. Updating transistor length parameters.
2. Substituting the appropriate process corner libraries.
3. Changing the output file name for each sweep case.
4. Saving each unique `.spice` netlist into a folder.

---

### Key Concept
Instead of editing 30 netlists manually, we:
- Keep **one base netlist template**.
- Replace certain parameters programmatically.
- Generate 30 variations (10 lengths × 3 corners).

Each output netlist can be run individually in Ngspice, producing
a `.csv` output containing the sweep data.

---

## `.control` Block Explanation and Local Simulation Guide

Here we explain the SPICE `.control` block and the guide for local simulation.

-----

### The `.control` Block in a SPICE Netlist

The **`.control`** section in a SPICE netlist, often used in simulators like **Ngspice**, defines the **sequence of commands and simulation steps** that the program will execute. Essentially, it's the simulation's scripting environment.

The Base `.spice` Netlist looks like this:

```
** sch_path: /foss/designs/cac/stg3/new_plots/CAC_ONRES_PAPER_Ibias_try1.sch
**.subckt CAC_ONRES_PAPER_Ibias_try1
V2 VG GND 1.65
XMN_B Vbias Vbias GND GND sg13_lv_nmos w={wx} l={lx} ng=1 m=1
I0 VDD Vbias 1u
V1 VDD GND 1.65
V6 VD GND {vdsx}
XMN_RON_B VD VG GND GND sg13_lv_nmos w={wx} l={lx} ng=1 m=1
V7 net1 GND {vout}
XMN_GM_B net1 Vbias GND GND sg13_lv_nmos w={wx} l={lx} ng=1 m=1

**** begin user architecture code

.lib cornerMOSlv.lib mos_tt
.lib cornerMOShv.lib mos_tt
.lib cornerHBT.lib hbt_typ
.lib cornerRES.lib res_typ
.lib cornerCAP.lib cap_typ

.param wx=1u lx=0.50u mx=1 vdsx=50m vout=0.825
.options savecurrents

.control
  unset appendwrite
  set wr_singlescale
  set wr_vecnames

  let start_w = 1u
  let start_vdsx = 0.4125

  foreach w_val 1u 2u 3u 4u 5u 6u 7u 8u 9u 10u
      foreach vds_val 1.65 1.6 1.2375 0.825 0.4125 0.05
         print 'Simulating with W=' $w_val ' 'VDS=' $vds_val

         alterparam wx = $w_val
         alterparam vdsx = $vds_val

         reset
         save all

         dc I0 5u 30u 5u V2 0.4125 1.65 0.4125

         let Ron_unit = @n.xmn_ron_b.nsg13_lv_nmos[vds]/@n.xmn_ron_b.nsg13_lv_nmos[ids]
         let Ipeak_unit = @n.xmn_ron_b.nsg13_lv_nmos[ids]
         let gmron_unit = @n.xmn_ron_b.nsg13_lv_nmos[gm]
         let Ibias_unit = @n.xmn_gm_b.nsg13_lv_nmos[ids]
         let gmbias_unit = @n.xmn_gm_b.nsg13_lv_nmos[gm]
         let gmrp = @n.xmn_b.nsg13_lv_nmos[gm]
         let width = @n.xmn_ron_b.nsg13_lv_nmos[w]
         let length = @n.xmn_ron_b.nsg13_lv_nmos[l]
         let Ron_Gm_unit = Ron_unit/gmbias_unit

         if (($w_val eq start_w) & ($vds_val eq start_vdsx))
           wrdata /foss/designs/cac/stg3/new_plots/LV_NMOS/paper_plots/large_plots/CAC_ONRES_PAPER_LV_NMOS_L_0_50_Ibias_tt.csv v(VG) v(VD) v(Vbias) gmrp Ipeak_unit gmron_unit Ibias_unit gmbias_unit Ron_Gm_unit width length
         else
           set appendwrite
           wrdata /foss/designs/cac/stg3/new_plots/LV_NMOS/paper_plots/large_plots/CAC_ONRES_PAPER_LV_NMOS_L_0_50_Ibias_tt.csv v(VG) v(VD) v(Vbias) gmrp Ipeak_unit gmron_unit Ibias_unit gmbias_unit Ron_Gm_unit width length
         end
      end
  end

  print 'Multi-parameter sweep finished.'
.endc

.GLOBAL VDD
.GLOBAL GND
.end
```

| Section | Command/Code Snippet | Purpose |
| :--- | :--- | :--- |
| **1. Initialization** | `unset appendwrite` | Ensures the data file is *overwritten* initially, not appended to. |
| | `set wr_singlescale` | Writes only the scale for vectors (like the DC sweep variable) once per file. |
| | `set wr_vecnames` | Writes the vector names (e.g., `v(VG)`, `Ron_unit`) as a header line. |
| **2. Loop Over Parameters** | `foreach w_val 1u 2u ... 10u` | Starts the outer loop, iterating the variable **`w_val`** (transistor width, $W$). |
| | `foreach vds_val 1.65 1.6 ... 0.05` | Starts the inner loop, iterating the variable **`vds_val`** (drain-source voltage, $V_{DS}$). |
| **3. Parameter Updates** | `alterparam wx = $w_val` | **Dynamically updates** the circuit parameter `wx` with the current width value from the loop. |
| | `alterparam vdsx = $vds_val` | **Dynamically updates** the circuit parameter `vdsx` with the current $V_{DS}$ value. |
| **4. Run DC Simulation** | `dc I0 5u 30u 5u V2 0.4125 1.65 0.4125` | Executes a **nested DC sweep**. The first sweep is on parameter `I0` (source/collector current) from $5\mu A$ to $30\mu A$ with a step of $5\mu A$ . The second, inner sweep is on voltage source `V2` (the gate voltage, $V_G$) from $0.4125 V$ to $1.65 V$ with a step of $0.4125 V$. |
| **5. Compute Key Quantities** | `let Ron_unit = @n.xmn_...[vds]/@n.xmn_...[ids]` | Uses the **`let`** command to calculate **derived parameters** like the transistor's on-resistance ($R_{on}$), which is defined as $V_{DS}/I_{DS}$. |
| | `let gmron_unit = @n.xmn_...[gm]` | Calculates the transconductance ($g_m$). |
| **6. Export Results** | `wrdata path/to/output.csv ...` | **Writes** the calculated and measured quantities (voltages, currents, $R_{on}$, $g_m$, etc.) to a `.csv` file. |
| **7. Post-Export** | `set appendwrite` | **Crucially**, after the first write, this command ensures that all **subsequent sweeps** (from the loops) will **append** data to the same `.csv` file, building the full dataset without overwriting. |

-----


### `.control` Block - Visual Flow Diagram
---

The table above describes each command; this diagram shows how they connect sequentially:

<table align="center">
  <tr>
    <td align="center">
      <img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/control_block_annotated_flow_v3.svg" width="500"/>
      <br>
      <h4 style="margin:4px 0; font-weight:normal;">
        <em>Fig. 8 .control block execution flow for the CAC on-resistance sweep</em>
      </h4>
    </td>
  </tr>
</table>

> **Each (L, corner) combination produces one CSV file** 15 files total (5 lengths × 3 corners).  
> Each file contains 13 W × 6 VDS × 24 (Ibias × VG) = **1872 data rows**.


---
## Simulation for Look up Table and Plots Generation
---

### This cell does the following

| Step | Task | Est. Time |
|------|------|-----------|
| 1 | Build **Ngspice 45.2** from source with `--enable-osdi` | ~9 min |
| 2 | Download all **IHP SG13G2 model files** (29 `.lib` files) from GitHub | ~30 s |
| 3 | Compile **PSP 103.6 NQS Verilog-A** → `psp103_nqs.osdi` via OpenVAF | ~2 min |
| 4 | Write **`.spiceinit`** so ngspice auto-loads the OSDI on every run | instant |
| 5 | Set simulation **parameters** (lengths, corners, output paths) | instant |
| 6 | Generate netlists and **run all 15 simulations** → CSV output | ~1 min |

> **IHP130 has Verilog-A models and we need to build ngspice from source**  
> The `apt` version on Colab is v36, too old for OSDI. The IHP `sg13_lv_nmos`
> uses the PSP 103.6 NQS compact model (`pspnqs103va`), which must be compiled
> from Verilog-A into a shared `.osdi` library. Both steps require ngspice 45.2
> built with `--enable-osdi`.

---
## 3.2 Build Ngspice 45.2 from Source

The `apt` ngspice on Colab is **v36** and predates OSDI entirely.
We download the 45.2 source tarball from SourceForge and compile with three critical flags:

- `--enable-osdi` - **the key flag**: enables runtime loading of `.osdi` shared libraries
- `--enable-predictor` - required companion flag for `--enable-osdi`
- `--enable-xspice` - extra SPICE elements used by IHP models

 **This step takes ~9 minutes, So have Patience!!.** Build output is sent to log files to keep output clean.

In [ ]:
# Ngspice Build and Installation with OSDI Support
# This section compiles ngspice (v45.2) from source within the Colab environment
# to enable OSDI-based model loading required by the IHP SG13G2 PDK.
#
# The default package does not support this feature, hence a custom build
# is performed to ensure compatibility with Verilog-A compact models.
# Rationale:
# The IHP SG13G2 PDK relies on the PSP 103.6 NQS compact model implemented
# in Verilog-A. This requires runtime loading via OSDI, which is not enabled
# in standard ngspice distributions. Therefore, ngspice is compiled with
# the required flags to support this functionality.
#
# Helper Function: run(cmd, desc, cwd)
# Executes shell commands required during the build process, provides clear
# status updates, and stops execution if any step fails.
#
# Execute command and report status
#
# Build Procedure:
# 1. Install required system dependencies (compilers, libraries, autotools)
# 2. Download ngspice source from the project repository
# 3. Extract source files and locate build directory
# 4. Generate configure script (required for GitHub archives)
# 5. Configure build with OSDI and related features enabled
# 6. Compile source using parallel execution
# 7. Install binaries to system path
# 8. Verify successful installation
#
#
%%time
import subprocess, sys, os

def run(cmd, desc="", cwd=None):
    # Run a shell command, print a one-line status.
    print(f"   {desc}...", end=" ", flush=True)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    if r.returncode == 0:
        print("✓")
    else:
        print(f"\n{r.stderr[-800:]}")
        raise RuntimeError(f"Failed: {desc}")
    return r

# Install system packages needed to compile ngspice from source
run("apt-get update -qq", "apt update")
run("apt-get install -y -qq git build-essential autoconf automake libtool "
    "flex bison gfortran libx11-dev libxaw7-dev libxmu-dev libxext-dev wget",
    "install build deps")

# Download ngspice 45.2 source from our repo
# The tarball is committed at the repo root as ngspice-ngspice-45.2.tar.gz.
# raw.githubusercontent.com serves it directly — no CDN, no mirrors, 100% reliable.
# Note: GitHub source archives use the "reponame-tagname" naming convention,
# hence the double "ngspice-ngspice-" prefix.
TARBALL = "/content/ngspice-ngspice-45.2.tar.gz"
OUR_URL = (
    "https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/main/"
    "ngspice-ngspice-45.2.tar.gz"
)

print("  download ngspice 45.2 (from repo)...", end=" ", flush=True)
r = subprocess.run(
    f"wget -q --timeout=60 --tries=3 '{OUR_URL}' -O {TARBALL}",
    shell=True, capture_output=True, text=True
)

# Verify that the downloaded file is valid before extraction
chk = subprocess.run(f"file {TARBALL}", shell=True, capture_output=True, text=True)
if r.returncode == 0 and "gzip" in chk.stdout:
    print("✓")
else:
    if os.path.exists(TARBALL): os.remove(TARBALL)
    raise RuntimeError(
        f" Download failed: {chk.stdout.strip()}\n"
        "Check that ngspice-ngspice-45.2.tar.gz is committed to the repo root at:\n"
        "https://github.com/chennakeshavadasa/gmid_IHP130/blob/main/ngspice-ngspice-45.2.tar.gz"
    )

# Extract
# GitHub source archives extract to a folder named <repo>-<tag>, i.e. ngspice-ngspice-45.2/
run(f"tar -xzf {TARBALL} -C /content", "extract tarball")

# Confirm the extracted directory name
r_ls = subprocess.run(
    "ls -d /content/ngspice-ngspice-45.2 2>/dev/null || ls -d /content/ngspice-45.2 2>/dev/null",
    shell=True, capture_output=True, text=True
)

# Identify extracted source directory (handles naming variations)
SRC_DIR = r_ls.stdout.strip().split("\n")[0]
if not SRC_DIR or not os.path.isdir(SRC_DIR):
    raise RuntimeError("Could not find extracted ngspice source directory under /content/")
print(f"  Source directory: {SRC_DIR}")

# Bootstrap ./configure (GitHub source archives omit it)
# The SourceForge release tarball ships a pre-generated ./configure.
# GitHub source archives do not — autoreconf generates it from configure.ac.
if not os.path.exists(f"{SRC_DIR}/configure"):
    run(f"cd {SRC_DIR} && autoreconf -fi > /content/ngspice_autoreconf.log 2>&1",
        "bootstrap (autoreconf)")

# Configure with OSDI support
run(f"cd {SRC_DIR} && ./configure "
    "--enable-xspice "        # Extra SPICE elements used by IHP models
    "--enable-osdi "          # THE KEY FLAG — enables .osdi runtime loading
    "--enable-predictor "     # Required companion flag for --enable-osdi
    "--with-readline=no "     # No readline needed in batch/Colab
    "--disable-debug "        # Faster build, smaller binary
    "--enable-shared "        # Build shared libraries
    "--prefix=/usr/local "    # Install to standard system path
    "> /content/ngspice_configure.log 2>&1",
    "configure (with --enable-osdi)")

# Parallel build using all available CPU cores (~8 min on Colab)
run(f"cd {SRC_DIR} && make -j$(nproc) > /content/ngspice_make.log 2>&1",
    "make (this takes ~8 min)")
run(f"cd {SRC_DIR} && make install > /dev/null 2>&1", "make install")

# Verify
r = subprocess.run("ngspice --version 2>&1 | head -5", shell=True,
                   capture_output=True, text=True)
print("\n Ngspice installed:")
print(r.stdout)


## 3.3 Download IHP SG13G2 Model Files

All `.lib` files are fetched from `chennakeshavadasa/gmid_IHP130` the same
repo used in the original working attempt. Files land in `/content/models/` so
all relative cross-includes inside the corner files resolve correctly.

In [ ]:
# IHP SG13G2 Model Library Download
# This section retrieves all required SPICE model library files associated
# with the SG13G2 PDK. These files define device behaviour across process
# corners and are necessary for accurate circuit-level simulation.
# DOWNLOAD ALL IHP SG13G2 PDK MODEL LIBRARY FILES
# PURPOSE:
#   Downloads 29 .lib model files for the IHP SG13G2 130 nm BiCMOS PDK
#   from GitHub and places them flat in /content/ (NOT a subdirectory).
#
# WHY FLAT IN /content/?
#   cornerMOSlv.lib contains lines like:
#       .include "sg13g2_moslv_mod.lib"
#   ngspice resolves relative paths from CWD = /content/ (set when running
#   simulations).  A subdirectory would break all relative includes.
#
# FILE CATEGORIES:
#   cornerMOSlv.lib / cornerMOShv.lib → LV and HV MOSFET process corners
#   cornerHBT.lib / cornerRES.lib / cornerCAP.lib → BJT, resistor, cap corners
#   sg13g2_moslv_mod.lib etc. → sub-libraries included by corner files
#   sg13g2_moslv_stat.lib etc. → statistical/mismatch model variants
#
# DOWNLOAD STRATEGY:
#   requests.get() with 15 s timeout per file from raw.githubusercontent.com.
#   ok/fail counters accumulate results;
#
import os, requests
from pathlib import Path

# All 29 model files land flat in /content/ — NOT a subdirectory.
# This is critical: cornerMOSlv.lib does .include "sg13g2_moslv_mod.lib"
# (relative, no path prefix). ngspice resolves this from CWD = /content/.
CONTENT = Path("/content")

# Complete list of all IHP SG13G2 model library files
LIB_FILES = [
    "capacitors_mod.lib", "capacitors_mod_mismatch.lib", "capacitors_stat.lib",
    "cornerCAP.lib", "cornerHBT.lib", "cornerMOShv.lib", "cornerMOSlv.lib",
    "cornerRES.lib", "diodes.lib", "resistors_mod.lib", "resistors_mod_mismatch.lib",
    "resistors_stat.lib", "sg13g2_bondpad.lib", "sg13g2_esd.lib",
    "sg13g2_hbt_mod.lib", "sg13g2_hbt_mod_mismatch.lib", "sg13g2_hbt_stat.lib",
    "sg13g2_moshv_mismatch.lib", "sg13g2_moshv_mod.lib",
    "sg13g2_moshv_mod_mismatch.lib", "sg13g2_moshv_parm.lib",
    "sg13g2_moshv_stat.lib", "sg13g2_moslv_mismatch.lib", "sg13g2_moslv_mod.lib",
    "sg13g2_moslv_mod_mismatch.lib", "sg13g2_moslv_parm.lib",
    "sg13g2_moslv_stat.lib", "sg13g2_svaricaphv_mod.lib",
    "sg13g2_svaricaphv_mod_mismatch.lib",
]

# Raw GitHub URL base — files served directly without git LFS complications
BASE_URL = "https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/main/model_files"

ok, fail = 0, 0
# Iterate through each file and perform download
for f in LIB_FILES:
    dest = CONTENT / f
    # Attempt to download file with timeout protection
    try:
        r = requests.get(f"{BASE_URL}/{f}", timeout=15)
        # Confirm successful HTTP response before saving file
        if r.status_code == 200:
            dest.write_bytes(r.content)
            ok += 1
        else:
            print(f"  HTTP {r.status_code}: {f}")   # File missing on server
            fail += 1
    except Exception as e:
        print(f"  ERROR {f}: {e}")    # Network timeout or connection error
        fail += 1

print(f"Downloaded {ok}/{len(LIB_FILES)} model files to /content/")
if fail:
    print(f"WARNING: {fail} file(s) failed")

# Verify the five corner files the netlists reference directly
for key in ["cornerMOSlv.lib", "cornerMOShv.lib", "cornerHBT.lib",
            "cornerRES.lib", "cornerCAP.lib"]:
    p = CONTENT / key
    print(f"  {chr(9989) if p.exists() else chr(10060) + chr(32) + chr(77) + chr(73) + chr(83) + chr(83) + chr(73) + chr(78) + chr(71)}  {key}")


---
## 3.4 Compile PSP 103.6 NQS Model → `psp103_nqs.osdi`

`sg13_lv_nmos` uses the **PSP 103.6 NQS** compact model (`pspnqs103va`).
ngspice does not have this built-in, it must be compiled from Verilog-A
source into a shared OSDI library and loaded at runtime.

This step does three things:

**A.** Download **OpenVAF** - the Verilog-A → OSDI compiler  
*(binary from `fides.fe.uni-lj.si`, OSDI 0.3 build - the DigitalOcean CDN is dead)*

**B.** Download **PSP 103.6 NQS Verilog-A source** (`psp103_nqs.va` + all includes)  
*from `dwarning/VA-Models` via GitHub API (bypasses git LFS)*

**C.** Compile `psp103_nqs.va` → `psp103_nqs.osdi` using OpenVAF

> **Note on patchelf:** On some Colab instances the OpenVAF binary needs its ELF
> interpreter patched to use conda's newer glibc. This is handled automatically.
> If conda's glibc isn't found the binary is tried as-is, it works either way.

In [ ]:
# COMPILE PSP 103.6 NQS VERILOG-A MODEL → .osdi
# THREE SUBSTEPS:
#
#  A: INSTALL OPENVAF
#   OpenVAF converts Verilog-A (.va) → OSDI shared library (.osdi).
#   We use the osdi_0.3 build (compatible with ngspice 45.2's OSDI 0.3 ABI).
#   Install sequence:
#     1. Miniforge3 → provides newer glibc via conda-forge (some Colab VMs
#        have glibc 2.35 but OpenVAF needs 2.38+).
#     2. Download tarball from fides.fe.uni-lj.si; extract the ELF binary
#        in-memory; validate with ELF magic (b'\x7fELF') before writing.
#     3. patchelf rewrites the ELF interpreter pointer to use conda's ld
#        so glibc 2.38 is resolved at runtime.
#     4. Symlink to /usr/local/bin/openvaf for PATH availability.
#
#  B: DOWNLOAD PSP 103.6 NQS VERILOG-A SOURCE
#   Source: dwarning/VA-Models (code/psp103/vacode103p6/).
#   Uses GitHub API download_url to bypass git LFS pointer files.
#   All .va and .include dependency files land in VA_DIR = /content/psp_va/.
#
#  C: COMPILE psp103_nqs.va → psp103_nqs.osdi
#   Command: openvaf -o /content/psp103_nqs.osdi psp103_nqs.va  (in VA_DIR)
#   Output validated: returncode==0, file exists, first 4 bytes == ELF magic.
#   Result: ~982 KB ELF shared library loaded by ngspice at startup.
#
import subprocess, stat, tarfile, io, requests
from pathlib import Path

OSDI_PATH = Path("/content/psp103_nqs.osdi")  # Final output: compiled PSP model
ELF_MAGIC  = b"\x7fELF"                       # First 4 bytes of any valid ELF binary
VA_DIR     = Path("/content/psp_va")          # Working directory for Verilog-A source
VA_DIR.mkdir(exist_ok=True)

# A: Get OpenVAF
# OpenVAF is the compiler that turns Verilog-A (.va) into OSDI (.osdi) shared libs.
# We use the osdi_0.3 release — compatible with ngspice 45.2's OSDI 0.3 API.
# The old DigitalOcean CDN (openva.fra1.cdn.digitaloceanspaces.com) returns 403.
print("=== A: Installing OpenVAF ===")

# Install miniforge + patchelf so we can fix the ELF interpreter if needed.
# Colab ships with glibc 2.35; some OpenVAF builds require 2.38+.
# patchelf rewrites the binary's interpreter pointer to use conda's loader.
subprocess.run(
    "if ! command -v conda &>/dev/null; then "
    "wget -q https://github.com/conda-forge/miniforge/releases/latest/download/"
    "Miniforge3-Linux-x86_64.sh -O /tmp/miniforge.sh && "
    "bash /tmp/miniforge.sh -b -p /opt/conda 2>&1 | tail -2; fi && "
    "/opt/conda/bin/conda install -y -q -c conda-forge patchelf 2>&1 | tail -2",
    shell=True, executable="/bin/bash", timeout=300
)

# Ensure /opt/conda/bin/ exists before we write the binary into it
OPENVAF_BIN = Path("/opt/conda/bin/openvaf")
OPENVAF_BIN.parent.mkdir(parents=True, exist_ok=True)

# Download the tarball and extract the openvaf ELF binary directly into memory
OPENVAF_URL = ("https://fides.fe.uni-lj.si/openvaf/download/"
               "openvaf-reloaded-osdi_0.3-linux_x64.tar.gz")
print("Downloading OpenVAF...", end=" ", flush=True)
r = requests.get(OPENVAF_URL, timeout=60)
if r.status_code == 200:
    with tarfile.open(fileobj=io.BytesIO(r.content), mode="r:gz") as tar:
        for m in tar.getmembers():
            if m.isfile() and "openvaf" in m.name.lower():
                data = tar.extractfile(m).read()
                if data[:4] == ELF_MAGIC:   # Confirm it's a real ELF, not a text file
                    OPENVAF_BIN.write_bytes(data)
                    OPENVAF_BIN.chmod(OPENVAF_BIN.stat().st_mode | stat.S_IEXEC)
                    print(f"OK ({len(data)//1024} KB)")
                    break
else:
    print(f"FAILED HTTP {r.status_code}")

# Attempt to patch the ELF interpreter to use conda's glibc.
# If conda's ld/libc aren't found, skip — the binary may still work as-is.
r2 = subprocess.run(r"""
    export PATH=/opt/conda/bin:$PATH
    CONDA_LD=$(find /opt/conda -name "ld-linux-x86-64.so.2" 2>/dev/null | head -1)
    CONDA_LIBDIR=$(find /opt/conda -name "libc.so.6" 2>/dev/null | head -1 | xargs -I{} dirname {})
    if [ -n "$CONDA_LD" ] && [ -n "$CONDA_LIBDIR" ]; then
        patchelf --set-interpreter "$CONDA_LD" --set-rpath "$CONDA_LIBDIR" /opt/conda/bin/openvaf
        echo "Patched: ld=$CONDA_LD"
    else
        echo "WARNING: conda ld/libc not found, skipping patchelf"
    fi
    /opt/conda/bin/openvaf --version 2>&1
""", shell=True, executable="/bin/bash", capture_output=True, text=True)
print(r2.stdout.strip())

# Symlink to /usr/local/bin so plain "openvaf" works without a full path
subprocess.run("ln -sf /opt/conda/bin/openvaf /usr/local/bin/openvaf", shell=True)

openvaf_ok = subprocess.run(
    "openvaf --version 2>&1", shell=True, capture_output=True, text=True
).returncode == 0
print(f"OpenVAF ready: {openvaf_ok}")

# B: Download PSP 103.6 NQS Verilog-A source
# Source: dwarning/VA-Models — a curated collection of open Verilog-A models.
# We use the GitHub API download_url to get direct file URLs, which bypasses
# git LFS (raw.githubusercontent.com would return an LFS pointer, not the file).
print("\n=== B: Downloading psp103_nqs.va ===")
import json, urllib.request
api_url = ("https://api.github.com/repos/dwarning/VA-Models"
           "/contents/code/psp103/vacode103p6")
try:
    with urllib.request.urlopen(api_url, timeout=15) as resp:
        items = json.load(resp)
    # Download all files in the directory — psp103_nqs.va plus all its .include deps
    for item in items:
        if item["type"] == "file":
            dest = VA_DIR / item["name"]
            rv = requests.get(item["download_url"], timeout=20)
            if rv.status_code == 200:
                dest.write_bytes(rv.content)
    va_files = sorted(VA_DIR.glob("*.va"))
    print(f"VA files: {[f.name for f in va_files]}")
except Exception as e:
    print(f"GitHub API error: {e}")

# C: Compile psp103_nqs.va → psp103_nqs.osdi
# OpenVAF compiles the entry point file and automatically resolves all
# .include statements relative to VA_DIR.
# Output is a shared ELF library (~982 KB) that ngspice loads at runtime.
print("\n=== C: Compiling psp103_nqs.va ===")
if not openvaf_ok:
    print("ERROR: OpenVAF not ready")
else:
    res = subprocess.run(
        f"cd {VA_DIR} && openvaf -o {OSDI_PATH} psp103_nqs.va",
        shell=True, capture_output=True, text=True, timeout=180
    )
    if res.returncode == 0 and OSDI_PATH.exists():
        print(f"OK ({OSDI_PATH.stat().st_size/1024:.0f} KB)")
    else:
        print(f"FAILED\n{res.stderr[-300:]}")

# Confirm the output is a valid ELF shared library
print()
if OSDI_PATH.exists() and OSDI_PATH.read_bytes()[:4] == ELF_MAGIC:
    print(f"SUCCESS: {OSDI_PATH} ({OSDI_PATH.stat().st_size/1024:.0f} KB) ELF OK")
else:
    print("FAILED")

---
## 3.5 Write `.spiceinit` and Verify Readiness

`.spiceinit` is ngspice's startup config file - read automatically every time
ngspice launches. We write it to both `/content/` and `/root/` so it is found
regardless of working directory.

The critical line is:
```
osdi /content/psp103_nqs.osdi
```
This tells ngspice to load the PSP 103.6 NQS shared library **before parsing
any netlist**. Without it, `sg13_lv_nmos` is an unknown model type and every
simulation fails immediately.

A readiness check runs at the end of this cell all four items should show ✅
before proceeding to simulations.

In [ ]:
# WRITE .spiceinit AND VERIFY FULL READINESS
# WHAT IS .spiceinit?
#   ngspice reads .spiceinit automatically at startup before any netlist.
#   It is the ONLY correct place to load OSDI plugins in batch mode
#   ('pre_osdi' in the netlist only works interactively and causes
#    'Error on line 4' in batch -b mode).
#
# KEY SETTINGS:
#   set ngbehavior=hsa    → IHP models require High-Speed Analog mode
#   set ng_nomodcheck     → disables parameter range checks that PSP triggers
#   set wr_singlescale    → wrdata writes ONE shared X-axis column in CSV
#   set wr_vecnames       → wrdata writes column names as CSV header row
#   unset appendwrite     → each wrdata call starts fresh (per-simulation)
#   set sourcepath / searchpath = ( /content ) → .lib/.include resolution
#   osdi /content/psp103_nqs.osdi → CRITICAL: loads PSP model before netlists
#     Without this: 'pspnqs103va: unknown model type' on every simulation.
#
# FILE PLACEMENT:
#   Written to BOTH /content/.spiceinit AND /root/.spiceinit so ngspice
#   finds it regardless of CWD or $HOME.
#
# READINESS CHECK (all 4 must be ✅):
#   1. psp103_nqs.osdi exists on disk  (Step 3)
#   2. .spiceinit contains 'psp103_nqs' directive
#   3. cornerMOSlv.lib exists          (Step 2)
#   4. ngspice --version succeeds      (Step 1)
#
from pathlib import Path

# Path to the compiled PSP OSDI library produced in Step 3
OSDI_PATH = Path("/content/psp103_nqs.osdi")

# Build the .spiceinit lines — these apply to every ngspice session
lines = [
    "* Ngspice init file -- IHP SG13G2 Colab",
    "set ngbehavior=hsa",       # HSA mode: required for correct IHP model behaviour
    "set ng_nomodcheck",        # Skip parameter range checks (needed for PSP model)
    "set wr_singlescale",       # Write a single shared X-axis column in CSV output
    "set wr_vecnames",          # Write vector names as the CSV header row
    "unset appendwrite",        # Each wrdata call starts fresh by default
    "set sourcepath = ( /content )",  # Where ngspice looks for .lib files
    "set searchpath = ( /content )",  # Where ngspice resolves .include statements
]

# Add the OSDI load directive — this is what actually loads the PSP model.
# Without this line ngspice sees "pspnqs103va: unknown model type".
if OSDI_PATH.exists():
    lines.append(f"osdi {OSDI_PATH}")
else:
    print(f"ERROR: {OSDI_PATH} not found -- re-run Step 3")

# Write to both locations for robustness
spiceinit = "\n".join(lines) + "\n"
for p in [Path("/content/.spiceinit"), Path("/root/.spiceinit")]:
    p.write_text(spiceinit)

# Readiness check
# Runs immediately after writing so .spiceinit always exists at check time.
# All four items must show ✅ before running simulations.
spiceinit_p = Path("/content/.spiceinit")
checks = {
    "psp103_nqs.osdi exists"  : OSDI_PATH.exists(),
    ".spiceinit has correct osdi" : "psp103_nqs" in spiceinit_p.read_text() if spiceinit_p.exists() else False,
    "cornerMOSlv.lib exists"  : Path("/content/cornerMOSlv.lib").exists(),
    "ngspice installed"       : __import__('subprocess').run(
                                    "ngspice --version 2>&1 | head -1",
                                    shell=True, capture_output=True, text=True
                                ).stdout.strip(),
}

print()
all_good = True
for k, v in checks.items():
    status = "✅" if v else "❌"
    print(f"  {status}  {k}: {v}")
    if v is False:
        all_good = False

print()
print("✅ READY TO SIMULATE" if all_good else "❌ Fix the items above first")

---
## 3.6 Simulation Configuration

Define sweep parameters and verify all required files exist before launching.

- **5 channel lengths:** 0.2, 0.5, 1, 4, 10 µm
- **3 process corners:** `tt` (typical-typical), `ss` (slow-slow), `ff` (fast-fast)
- **15 total simulations** - each sweeps width (1-10 µm) and V_DS (6 values)
  over a DC bias current sweep (5-30 µA in 5 µA steps)

In [ ]:
# SIMULATION CONFIGURATION AND PRE-FLIGHT CHECKS
# PURPOSE: Defines all paths and sweep parameters. No simulations run here.
#
# DIRECTORY STRUCTURE:
#   OUT_DIR = /content/spice_generated/    ← generated .spice netlist files
#   CSV_DIR = /content/simulation_results/ ← ngspice wrdata CSV outputs
#   LOG_DIR = /content/logs/               ← full ngspice stdout/stderr logs
#
# SWEEP SPACE: 5 lengths × 3 corners = 15 total simulations
#   LENGTHS = [0.2, 0.5, 1, 4, 10] µm
#   CORNERS = ['tt', 'ss', 'ff']
#   Within each simulation: W ∈ {1–10 µm}, VDS ∈ {6 values}, Ibias swept.
#
# LIB dict: absolute paths to corner .lib files.
#   Absolute paths are required because netlists live in OUT_DIR
#   (/content/spice_generated/), not in /content/ where the libs are.
#   Relative paths would fail with 'cannot open file'.
#
# OUTPUT: prints ✅/❌ for each library and OSDI path as a pre-flight check.
#
from pathlib import Path

CONTENT = Path("/content")

# Output directories — created here if they don't already exist
OUT_DIR = Path("/content/spice_generated")     # Generated .spice netlist files
CSV_DIR = Path("/content/simulation_results")  # wrdata CSV output files
LOG_DIR = Path("/content/logs")                # Full ngspice stdout/stderr logs

for d in [OUT_DIR, CSV_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Absolute paths to corner library files — used in .lib statements in each netlist
LIB = {
    "moslv" : CONTENT / "cornerMOSlv.lib",  # LV MOSFET corners (sg13_lv_nmos)
    "moshv" : CONTENT / "cornerMOShv.lib",  # HV MOSFET corners
    "hbt"   : CONTENT / "cornerHBT.lib",    # HBT transistor corners
    "res"   : CONTENT / "cornerRES.lib",    # Resistor corners
    "cap"   : CONTENT / "cornerCAP.lib",    # Capacitor corners
}
OSDI_PATH = CONTENT / "psp103_nqs.osdi"  # Compiled PSP model (from Step 3)

# Sweep parameters: 5 lengths × 3 corners = 15 simulations
LENGTHS = [0.2, 0.5, 1, 4, 10]    # Transistor channel lengths in µm
CORNERS = ["tt", "ss", "ff"]      # Process corners

print("Configuration:")
print(f"  Lengths  : {LENGTHS} um")
print(f"  Corners  : {CORNERS}")
print(f"  Total    : {len(LENGTHS)*len(CORNERS)} simulations")
osdi_status = ("✅ " + str(OSDI_PATH)) if OSDI_PATH.exists() else "❌ MISSING"
print(f"  OSDI     : {osdi_status}")
for k, v in LIB.items():
    lib_status = "✅" if v.exists() else "❌ MISSING"
    print(f"  .lib {k:<6}: {lib_status}")


---
## 3.7 Generate Netlists and Run All 15 Simulations

For each (length, corner) combination, a SPICE netlist is generated from
the verified local template and run with ngspice in batch mode.

**Key design decisions in the netlist:**
- **No `pre_osdi`** in the netlist - OSDI is loaded via `.spiceinit` only
  (`pre_osdi` is interactive-mode only and causes "Error on line 4" in batch mode)
- **Absolute `.lib` paths** (`/content/cornerMOSlv.lib`) - relative paths fail
  because the netlist file lives in `/content/spice_generated/`, not `/content/`
- **`cwd="/content/"`** when invoking ngspice ensures `.spiceinit` is found
  and relative `.include` statements inside the `.lib` files resolve correctly

**Circuit:** Three `sg13_lv_nmos` instances measuring Ron, Gm, and Ron×Gm  
**Sweep:** $W$ = 1-10 µm, $V_{DS}$ = 6 values, $I_{bias}$ = 5-30 µA at each ($W$, $V_{DS}$) point  
**Output:** One CSV per (length, corner) in `/content/simulation_results/`

In [ ]:

# GENERATE NETLISTS AND RUN ALL 15 NGSPICE SIMULATIONS
#
# len_to_tag(v):
#   Converts float length (e.g. 0.5) to a filename-safe string ('0_50').
#   Format: f'{v:.2f}'.replace('.','_') — ensures consistent 2 decimal places.
#
# make_netlist(lx_val, corner):
#   Generates the complete SPICE netlist for one (L, corner) combination.
#   THREE sg13_lv_nmos instances:
#     XMN_B      — diode-connected, biased by I0 (1µA). Sets Vbias.
#     XMN_RON_B  — gate at VDD (fully on), VDS swept. Ron = VDS/IDS.
#     XMN_GM_B   — gate at Vbias, drain at Vout=0.825V. IDS → gm.
#   .control block:
#     OUTER foreach: wx in {1u..10u}  (10 device widths)
#     INNER foreach: vds in {1.65, 1.6, 1.2375, 0.825, 0.4125, 0.05}  (6 bias pts)
#       dc I0 5u 30u 5u  V2 0.4125 1.65 0.4125  → 2D DC sweep per (W, VDS)
#       Computes: Ron_unit, Ipeak_unit, gmron_unit, gmbias_unit, Ron_Gm_unit
#       First iteration: wrdata creates fresh CSV; all others: appendwrite=True
#
# run_sim(lx_val, corner):
#   1. make_netlist() → text + csv_path
#   2. Write .spice file to OUT_DIR
#   3. subprocess: ngspice -b <file>  with cwd='/content'
#      cwd='/content' CRITICAL: .spiceinit + all .lib files are here
#   4. Check CSV exists and is non-empty
#   5. On failure: scan stdout+stderr for error/fatal/unknown/osdi/model lines
#   6. Return status dict: {status, elapsed, csv_kb} or {status, errors, log}
#
# MAIN LOOP:
#   Iterates all 15 (lx, corner) pairs; prints live ASCII progress table.
#   Final summary shows success/fail counts and total wall time.
#
from pathlib import Path
import subprocess, time

# Re-declare paths so this cell is self-contained and can be re-run independently
CONTENT = Path("/content")
OUT_DIR = Path("/content/spice_generated")
CSV_DIR = Path("/content/simulation_results")
LOG_DIR = Path("/content/logs")
for d in [OUT_DIR, CSV_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

LENGTHS = [0.2, 0.5, 1, 4, 10]   # Channel lengths in µm
CORNERS = ["tt", "ss", "ff"]     # Process corners

def len_to_tag(v):
    """Convert float length to a filename-safe string. E.g. 0.5 -> '0_50'"""
    return f"{v:.2f}".replace(".", "_")

def make_netlist(lx_val, corner):
    """
    Generate a complete SPICE netlist for one (length, corner) pair.

    This is the verified local netlist with two minimal changes for Colab:
      1. .lib paths are absolute (/content/cornerMOSlv.lib not relative)
      2. wrdata path points to /content/simulation_results/ not local FOSS path
    Everything else — circuit, .control block, sweep structure — is unchanged.
    """
    len_tag  = len_to_tag(lx_val)
    csv_path = CSV_DIR / f"CAC_ONRES_LV_NMOS_L_{len_tag}_{corner}.csv"

    # Identical to the verified local netlist — only wrdata path changed
    return f"""** sch_path: /foss/designs/cac/stg3/new_plots/CAC_ONRES_PAPER_Ibias_try1.sch
**.subckt CAC_ONRES_PAPER_Ibias_try1
V2 VG GND 1.65
XMN_B Vbias Vbias GND GND sg13_lv_nmos w={{wx}} l={{lx}} ng=1 m=1
I0 VDD Vbias 1u
V1 VDD GND 1.65
V6 VD GND {{vdsx}}
XMN_RON_B VD VG GND GND sg13_lv_nmos w={{wx}} l={{lx}} ng=1 m=1
V7 net1 GND {{vout}}
XMN_GM_B net1 Vbias GND GND sg13_lv_nmos w={{wx}} l={{lx}} ng=1 m=1

**** begin user architecture code

.lib /content/cornerMOSlv.lib mos_{corner}
.lib /content/cornerMOShv.lib mos_{corner}
.lib /content/cornerHBT.lib hbt_typ
.lib /content/cornerRES.lib res_typ
.lib /content/cornerCAP.lib cap_typ

.param wx=1u lx={lx_val}u mx=1 vdsx=50m vout=0.825
.options savecurrents

.control
  unset appendwrite
  set wr_singlescale
  set wr_vecnames

  let start_w = 1u
  let start_vdsx = 0.4125

  foreach w_val 1u 2u 3u 4u 5u 6u 7u 8u 9u 10u
      foreach vds_val 1.65 1.6 1.2375 0.825 0.4125 0.05
         alterparam wx = $w_val
         alterparam vdsx = $vds_val

         reset
         save all

         dc I0 5u 30u 5u V2 0.4125 1.65 0.4125

         let Ron_unit = @n.xmn_ron_b.nsg13_lv_nmos[vds]/@n.xmn_ron_b.nsg13_lv_nmos[ids]
         let Ipeak_unit = @n.xmn_ron_b.nsg13_lv_nmos[ids]
         let gmron_unit = @n.xmn_ron_b.nsg13_lv_nmos[gm]
         let Ibias_unit = @n.xmn_gm_b.nsg13_lv_nmos[ids]
         let gmbias_unit = @n.xmn_gm_b.nsg13_lv_nmos[gm]
         let gmrp = @n.xmn_b.nsg13_lv_nmos[gm]
         let width = @n.xmn_ron_b.nsg13_lv_nmos[w]
         let length = @n.xmn_ron_b.nsg13_lv_nmos[l]
         let Ron_Gm_unit = Ron_unit/gmbias_unit

         if (($w_val eq start_w) & ($vds_val eq start_vdsx))
           wrdata {csv_path} v(VG) v(VD) v(Vbias) gmrp Ipeak_unit gmron_unit Ibias_unit gmbias_unit Ron_Gm_unit width length
         else
           set appendwrite
           wrdata {csv_path} v(VG) v(VD) v(Vbias) gmrp Ipeak_unit gmron_unit Ibias_unit gmbias_unit Ron_Gm_unit width length
         end
      end
  end

  print 'Multi-parameter sweep finished.'
.endc

.GLOBAL VDD
.GLOBAL GND
.end
""", csv_path

def run_sim(lx_val, corner):
    """
    Write the netlist to disk and invoke ngspice in batch mode (-b).
    Returns a dict: {status, elapsed, csv_kb} on success,
                    {status, elapsed, errors, log} on failure.

    cwd="/content/" is critical — ngspice reads .spiceinit from its CWD,
    and all .lib files live there.
    """
    netlist, csv_path = make_netlist(lx_val, corner)
    len_tag = len_to_tag(lx_val)
    spice_file = OUT_DIR / f"CAC_ONRES_L_{len_tag}_{corner}.spice"
    log_file   = LOG_DIR / f"CAC_ONRES_L_{len_tag}_{corner}.log"
    spice_file.write_text(netlist)  # Write netlist to disk
    t0 = time.time()
    try:
        proc = subprocess.run(
            ["ngspice", "-b", str(spice_file)],   # -b = batch mode, no GUI
            capture_output=True, text=True,
            cwd="/content",   # CRITICAL: .spiceinit + all .lib files are here
            timeout=700,
        )
        log_file.write_text(proc.stdout + "\n" + proc.stderr)   # Always save full log
        elapsed = time.time() - t0
        if proc.returncode == 0:
            csv_kb = csv_path.stat().st_size/1024 if csv_path.exists() else 0
            return {"status":"success", "elapsed":elapsed, "csv_kb":csv_kb}
        else:
            # Filter log for known error keywords to surface the root cause quickly
            all_out = proc.stdout + proc.stderr
            errs = [l for l in all_out.splitlines()
                    if any(k in l.lower() for k in
                           ["error","fatal","unknown","cannot open","psp","osdi","model"])]
            return {"status":"failed", "elapsed":elapsed,
                    "errors":errs[:6], "log":str(log_file)}

    except subprocess.TimeoutExpired:
        return {"status":"timeout", "elapsed":700}  # Simulation hung
    except FileNotFoundError:
        return {"status":"ngspice_missing", "elapsed":0}  # ngspice not in PATH

# Run all 15 simulations and print a live progress table
results = []
total = len(LENGTHS) * len(CORNERS)
count = 0
grand_start = time.time()

print(f"{'#':<5}{'L (µm)':<10}{'Corner':<8}{'Status':<14}{'Time':>7}  Notes")
print("─" * 65)

for lx in LENGTHS:
    for corner in CORNERS:
        count += 1
        r = run_sim(lx, corner)
        r.update({"lx": lx, "corner": corner})
        results.append(r)
        icon = {"success":"✅","failed":"❌","timeout":"⏰"}.get(r["status"],"❓")
        note = f"CSV {r.get('csv_kb',0):.1f} KB" if r["status"] == "success" \
               else (r.get("errors",["(see log)"])[0][:55] if r.get("errors") else "(see log)")
        print(f"[{count:>2}/{total}] {lx:<10}{corner:<8}{icon} {r['status']:<14}"
              f"{r['elapsed']:>6.1f}s  {note}")

wall = time.time() - grand_start
ok   = sum(1 for r in results if r["status"] == "success")
fail = sum(1 for r in results if r["status"] == "failed")
print("\n" + "═"*65)
print(f"  ✅ Success : {ok}/{total}")
print(f"  ❌ Failed  : {fail}/{total}")
print(f"  🕐 Wall time: {wall/60:.1f} min")

### Netlist Generation Complete

The cell above generated **15 SPICE netlists** (5 channel lengths x 3 process corners: TT, SS, FF), saved to `/content/spice_generated/`. Each netlist sweeps transistor width (W = 1-10 um) and V_DS across 6 values over a DC bias current range (5-30 uA).

**Connection to the design flow:** These netlists are the simulation inputs that produce the look-up table (LUT) data for the Ron/gm methodology plots in the next section. Without this sweep, there is no data to read off $R_{on}$/$g_m$ vs $V_{bias}$, $g_m$/$I_d$, or voltage swing. Each resulting CSV represents one (L, corner) characterisation point in the design space.


In [ ]:
!ls /content/spice_generated

### Simulations Complete - CSV Data Extracted

All 15 simulations ran successfully. Each CSV in `/content/simulation_results/` contains the swept $R_{on}$, $g_m$, $V_{bias}$, width, and length data for one (L, corner) combination.



In [ ]:
!ls /content/simulation_results

---
# 4. Plot Generation for $R_{on}/g_m$ Methodology Based Design
---

> **Visualization of design plots used in the $R_{on}/g_m$ methodology**

In this section, we generate and analyze the plots required for circuit design using the **$R_{on}/g_m$ methodology**.  
The device characterization data used for these plots was generated during the previous simulation step.

For convenience and reproducibility, we load the **locally extracted simulation data** corresponding to the **LV\_NMOS** and **LV\_PMOS** devices from **IHP 130 PDK**.

These plots provide deeper design insight and enable additional design intuition that is **not readily available in the traditional $g_m/I_D$ methodology**.

Using this framework, we generate **eight key design plots**:

1. **$V_{bias}$ vs $R_{on}/g_m$**  
2. **$V_{bias}$ vs Width**  
3. **$I_{peak}$ vs $R_{on}/g_m$**   
4. **$g_{m,bias}$ vs $R_{on}/g_m$**  
5. **$g_m/I_D$ vs $R_{on}/g_m$**  
6. **$V_{swing}$ vs $R_{on}/g_m$**


These plots form the **core design framework of the $R_{on}/g_m$ methodology**, enabling improved trade-off analysis between biasing, device sizing, current consumption, and achievable signal swing.

---

### How to Read These Plots

Each plot is generated by `plot_data()` using data from the simulation CSVs. The table below maps each plot to the design question it answers:

| # | Plot | X-axis | Y-axis | Design Question Answered | How to Use It |
|:--:|:--|:--|:--|:--|:--|
| 1 | **$V_{bias}$ vs $R_{on}/g_m$** | $R_{on}/g_m$ (log) | Gate bias voltage $V_{bias}$ | What bias voltage do I need to hit my target $R_{on}/g_m$? | Read off $V_{DZP}$ / $V_{DZN}$ deadzone voltages directly. Corner spread shows process sensitivity. |
| 2 | **$V_{bias}$ vs Width** | Device width $W$ | Gate bias voltage $V_{bias}$ | What device width corresponds to my chosen bias point? | Use to size the unit cell $W_{unit}$ before applying the multiplier $M$. |
| 3 | **$I_{peak}$ vs $R_{on}/g_m$** | $R_{on}/g_m$ (log) | Peak current $I_{peak}$ (log) | How much peak current does the RC settling phase deliver? | Higher $I_{peak}$ → faster large-signal charging. Sets initial slew rate into $C_{load}$. |
| 4 | **$g_{m,bias}$ vs $R_{on}/g_m$** | $R_{on}/g_m$ (log) | Bias-path $g_{m,bias}$ | What is the bias-path $g_m$? | Verifies small-signal settling bandwidth: $BW = g_{m,bias} / (2\pi C_{load})$. |
| 5 | **$g_m/I_D$ vs $R_{on}/g_m$** | $R_{on}/g_m$ (log) | Efficiency $g_m/I_D$ | What power efficiency am I operating at for this $R_{on}/g_m$ choice? | Bridges the $R_{on}/g_m$ and $g_m/I_D$ methodologies - shows the efficiency cost of a given $R_{on}$ target. |
| 6 | **$V_{swing}$ vs $R_{on}/g_m$** | $R_{on}/g_m$ (log) | Output voltage swing $V_{swing}$ | What output swing is achievable at this operating point? | Confirms the chosen bias allows the required signal swing without clipping. |

**Colour coding:** Each trace is coloured by process corner (TT/SS/FF). The spread between corner traces shows how robust the chosen operating point is across process variation - this is information the $g_m$/$I_d$ methodology cannot provide pre-simulation.

---


In [ ]:
# data_path : str
#   Root directory of simulation result CSVs.  Defaults to
#   '/content/data/large_steps'; overridden via DATA_PATH env var
#   (useful for local/non-Colab runs).
#
# data : pd.DataFrame
#   Master in-memory table of all loaded simulation data.
#   Starts empty; populated by load_data().  All plot_data() calls
#   read from this global — avoids re-loading between plot cells.
#
# file_cache : dict[str, dict]
#   Mtime-based incremental cache keyed by filename.
#   Each entry: { 'mtime': float, 'df': pd.DataFrame }
#   load_data() uses this to skip unchanged files across repeated calls.
#
import re
import os
import pandas as pd
from pandas import DataFrame
from typing import Tuple, Dict, Literal, Any

data_path = os.getenv("DATA_PATH", "/content/data/large_steps")

data: DataFrame = pd.DataFrame()
file_cache: Dict[str, Dict[str, Any]] = {}


In [ ]:
# Extracts channel length (metres) and process corner label from a CSV filename
# using two independent regular expressions.
#
# FILENAME EXAMPLE:  CAC_MULVARY_LV_NMOS_L_0_50_Ibias_ff.csv
#                                            └len┘        └cor┘
#
# REGEX 1 — LENGTH:  r'L_(\d+)_(\d+)'
#   L_0_50  → major=0, minor=50  → (0 + 50/100) × 1e-6 = 0.5 µm
#   L_1_00  → major=1, minor=0   → 1.0 µm
#   L_10_00 → major=10, minor=0  → 10.0 µm
#   Minor ÷ 100 because the encoding uses two digits (50 → 0.50).
#
# REGEX 2 — CORNER:  r'_([fsht]{2})\.csv$'
#   _ff.csv → 'FF', _ss.csv → 'SS', _tt.csv → 'TT'
#   Upper-cased for consistent filter comparisons downstream.
#
# RETURNS: (length_metres, corner_str)  — either may be None if no match.
#
def extract_length_corner(filename: str) -> Tuple[float | None, str | None]:
    """
    Extracts the length (converted to meters) and corner value from a filename.
    Example:
      CAC_MULVARY_LV_NMOS_L_0_50_Ibias_ff.csv
    Returns:
      length (float in meters), corner (str)
    """
    # Regex to match L_<digit>_<digit> and corner (e.g. ff, ss, tt, etc.)
    length_match = re.search(r"L_(\d+)_(\d+)", filename, re.IGNORECASE)
    corner_match = re.search(r"_([fsht]{2})\.csv$", filename, re.IGNORECASE)

    length = None
    corner = None

    if length_match:
        major = int(length_match.group(1))  # before underscore
        minor = int(length_match.group(2))  # after underscore
        length = (major + minor / 100) * 1e-6  # convert to meters

    if corner_match:
        corner = corner_match.group(1).upper()

    return length, corner

In [ ]:
# Reads a single simulation CSV and prepares it for concatenation.
#   1. pd.read_csv() — parses header + numeric rows.
#   2. columns.str.strip() — removes leading/trailing whitespace from
#      column names (ngspice sometimes writes ' v(VG)' with a leading space).
#   3. df['corner'] = corner — injects the process corner label ('TT','SS','FF')
#      since the CSV itself does not record which corner it came from.
# RETURNS: pd.DataFrame with all CSV columns + 'corner' column.
#
def read_data_files(file_path: str, corner: Literal["FF", "SS", "TT"]) -> pd.DataFrame:
    df = pd.read_csv(file_path)

    # clean column names
    df.columns = df.columns.str.strip()

    df["corner"] = corner
    return df

In [ ]:
# Incrementally loads all valid TT/SS/FF CSV files from data_path into the
# global `data` DataFrame.  Safe to call multiple times — only re-reads
# changed or new files and evicts deleted ones from the cache.
#
# ALGORITHM:
#   1. Guard: exits early if data_path does not exist.
#   2. For each file in data_path:
#        a. Skip non-files (directories, symlinks).
#        b. extract_length_corner() → corner label; skip if not TT/SS/FF.
#        c. Compare os.path.getmtime() vs cache mtime → skip if unchanged.
#        d. Otherwise: read file, store in file_cache, set updated=True.
#   3. Evict files from cache that no longer exist on disk.
#   4. Rebuild `data` = pd.concat(...) only if something changed.
#
# GLOBALS MUTATED: `data` (master DataFrame), `file_cache` (per-file cache).
#
def load_data():
    if not os.path.isdir(data_path):
        print(f"⚠️  data_path does not exist: {data_path}")
        print("   Run download_large_steps() first, then re-run this cell.")
        return
    files = os.listdir(data_path)
    global data
    global file_cache

    updated = False

    # helper to track current files to identify deletions
    current_files = set()

    for file in files:
        file_path = os.path.join(data_path, file)

        # Skip if not a file
        if not os.path.isfile(file_path):
            continue

        current_files.add(file)

        _, corner = extract_length_corner(file)

        # We only care about valid corner files
        if corner in ["FF", "SS", "TT"]:
            mtime = os.path.getmtime(file_path)

            # Check if file is in cache and hasn't changed
            if file in file_cache and file_cache[file]['mtime'] == mtime:
                continue

            print(f"Loading {file}...")
            file_df = read_data_files(file_path=file_path, corner=corner)
            file_cache[file] = {'mtime': mtime, 'df': file_df}
            updated = True
        else:
            # print(f"Skipping {file}...")
            pass

    # Remove files that no longer exist
    for cached_file in list(file_cache.keys()):
        if cached_file not in current_files:
            print(f"Removing {cached_file} from cache...")
            del file_cache[cached_file]
            updated = True

    if updated or data.empty:
        if file_cache:
            data = pd.concat([item['df'] for item in file_cache.values()], ignore_index=True)
        else:
            data = pd.DataFrame()
        print("Global dataframe updated.")
    else:
        print("No changes detected. Data remains same.")

In [ ]:
# Downloads the full simulation result CSV dataset from a shared Google
# Drive folder using gdown.download_folder().
#   - url: the Drive folder sharing URL (default: project folder)
#   - data_path: local destination directory
#   - quiet=False: shows per-file progress bars
#   - use_cookies=False: works with publicly shared folders without login
# Expected runtime: ~1–3 min depending on Drive bandwidth.
#
import gdown

def download_large_steps(
    url="https://drive.google.com/drive/folders/1QN5Frm7yg9VJRuCMFaIr-2wcme2KDDkc",
    data_path=data_path,
):
    """
    Download the Google Drive folder into ./data/large_steps
    """

    os.makedirs(data_path, exist_ok=True)

    print(f"Downloading dataset to {data_path}...")

    gdown.download_folder(
        url=url,
        output=data_path,
        quiet=False,
        use_cookies=False
    )

    print("Download complete.")

In [ ]:
# PLOTTING INFRASTRUCTURE
# filter_dataframe / generate_hovertext / plot_data
# filter_dataframe(df, filters):
#   Applies a dict of column→value constraints to df using AND logic.
#   NUMERIC columns: np.isclose() instead of '==' to tolerate SPICE
#   floating-point rounding (e.g. 1.6500000001 vs 1.65).
#   LIST values: OR logic within that column, then AND with others.
#   mask=True broadcasts automatically to a boolean Series on first &=.
#
# generate_hovertext(df, hover_cols, hover_all_cols):
#   Builds HTML tooltip strings 'col: val<br>col: val' per row.
#   hover_cols=list → only listed columns; hover_all_cols=True → all cols.
#   Returns None if neither is set (Plotly falls back to x+y tooltip).
#
# plot_data(x_col, y_col, filters, group_col, color_group, ...):
#   Full plotting pipeline reading from the global `data` DataFrame:
#     1. Guard checks (empty data, missing columns).
#     2. filter_dataframe() → relevant slice.
#     3. Downsample if len(df) > max_points (default 50 000) to avoid lag.
#     4. COLOR GROUP: unique (col-value tuples) → Plotly qualitative palette.
#     5. GROUPING: df.groupby(group_col) → one trace per group.
#     6. go.Scattergl (WebGL) traces for performance on large datasets.
#     7. Layout: optional log axes, auto-sized title, hovermode='closest'.
#   RETURNS: go.Figure (caller calls fig.show()).
#
import math
from typing import Dict, List, Optional, Union

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "colab"

# -------------------------------
# Filtering
# -------------------------------
def filter_dataframe(
    df: DataFrame,
    filters: Optional[Dict[str, Union[str, int, float, list]]] = None,
):
    if not filters:
        return df

    mask = True

    for col, val in filters.items():

        if col not in df.columns:
            continue

        if isinstance(val, list):

            mask_col = False

            for v in val:

                if pd.api.types.is_numeric_dtype(df[col]) and isinstance(v, (int, float)):
                    mask_col |= np.isclose(df[col], v)
                else:
                    mask_col |= df[col].astype(str) == str(v)

            mask &= mask_col

        else:

            if pd.api.types.is_numeric_dtype(df[col]) and isinstance(val, (int, float)):
                mask &= np.isclose(df[col], val)
            else:
                mask &= df[col].astype(str) == str(val)

    return df[mask]

# Hover text
def generate_hovertext(df, hover_cols=None, hover_all_cols=False):

    if hover_cols is not None:

        return df.apply(
            lambda row: "<br>".join(
                f"{col}: {row[col]}" for col in hover_cols if col in df.columns
            ),
            axis=1,
        )

    elif hover_all_cols:

        return df.apply(
            lambda row: "<br>".join(f"{col}: {row[col]}" for col in df.columns),
            axis=1,
        )

    return None


# Main plotting function
def plot_data(
    x_col: str,
    y_col: str,
    filters: Optional[Dict] = None,
    group_col: Optional[Union[str, List[str]]] = None,
    color_group: Optional[Union[str, List[str]]] = None,
    hover_cols: Optional[List[str]] = None,
    hover_all_cols: bool = False,
    hovermode="closest",
    x_log: bool = False,
    y_log: bool = False,
    title: Optional[str] = None,
    width: int = 900,
    height: int = 500,
    max_points: int = 50000,
):
    """
    Plot columns from the global dataframe `data`.
    """

    global data

    if data.empty:
        print("⚠️  No data loaded yet — run the download and load_data() cells first.")
        return
    if x_col not in data.columns or y_col not in data.columns:
        print(f"⚠️  Column not found. Available: {data.columns.tolist()}")
        return

    df = filter_dataframe(data, filters)

    if df.empty:
        print("No data after filtering.")
        return

    # Downsample large datasets
    if len(df) > max_points:
        step = math.ceil(len(df) / max_points)
        df = df.iloc[::step]

    fig = go.Figure()

    # Color grouping

    color_map = {}

    if color_group:

        if isinstance(color_group, str):
            color_group = [color_group]

        df = df.copy()

        color_keys = df[color_group].apply(lambda row: tuple(row), axis=1)

        palette = px.colors.qualitative.Plotly

        unique_keys = list(color_keys.unique())

        colors = [palette[i % len(palette)] for i in range(len(unique_keys))]

        color_map = dict(zip(unique_keys, colors))

        df["_color_key"] = color_keys

    # Grouping

    if group_col:

        group_cols = [group_col] if isinstance(group_col, str) else group_col

        grouped = df.groupby(group_cols)

    else:

        grouped = [(None, df)]

    # Add traces

    for group_vals, df_group in grouped:

        hovertext = generate_hovertext(df_group, hover_cols, hover_all_cols)

        label = ""

        if group_vals is not None:

            if isinstance(group_vals, tuple):

                label = " | ".join(
                    f"{c}={v}" for c, v in zip(group_cols, group_vals)
                )

            else:

                label = f"{group_cols[0]}={group_vals}"

        trace_color = None

        if color_group:
            trace_color = color_map[df_group["_color_key"].iloc[0]]

        fig.add_trace(
            go.Scattergl(
                x=df_group[x_col],
                y=df_group[y_col],
                mode="lines+markers",
                name=label if label else y_col,
                hovertext=hovertext,
                hoverinfo="text" if hovertext is not None else "x+y",
                line=dict(color=trace_color) if trace_color else None,
            )
        )

    # Layout

    fig.update_layout(
        width=width,
        height=height,
        title=title if title else f"{y_col} vs {x_col}",
        xaxis=dict(
            title=x_col,
            type="log" if x_log else "linear",
        ),
        yaxis=dict(
            title=y_col,
            type="log" if y_log else "linear",
        ),
        hovermode=hovermode,
    )

    return fig

In [ ]:
download_large_steps()

In [ ]:
# Load data
load_data()

In [ ]:
# Print loaded column names
print(data.columns.tolist())

In [ ]:
# Show unique VDS sweep values in the dataset
data["v(VD)"].unique()

### How to Read the Following 6 Design Plots

Each of the 6 cells below calls `plot_data()`, which:
1. Slices the global `data` DataFrame using `filter_dataframe()` selecting specific VGS, VDS, length, and corner values
2. Groups the result by the specified columns and assigns colours by corner (TT/SS/FF)
3. Generates an interactive Plotly figure with hover tooltips showing all relevant values

**How to use the plots in the design flow:**

- **Hover** over any trace to read exact ($R_{on}$/$g_m$, $V_{bias}$, width, $g_m$/$I_d$) values at any point
- **Legend** entries can be clicked to isolate individual corners or lengths
- All X-axes are $R_{on}/g_m$ (log scale), your design target goes on this axis first
- Work left-to-right through the plots: start at Plot 1 to pick $R_{on}/g_m$, then use Plots 2–6 to confirm width, current, and swing requirements


In [ ]:
# DESIGN PLOT 4.1a: Vbias vs log(Ron/Gm)
# DESIGN QUESTION: 'What gate bias voltage achieves my target Ron/Gm ratio?'
# Filters: VGS=VDD (fully on), VDS=0.05V (triode), length=1u, corner=TT.
# Designer reads V_DZN (NMOS deadzone voltage) directly from this plot.
# KEY ADVANTAGE: gm/ID cannot provide this Ron/Gm vs Vbias mapping pre-simulation.
# Use this to understand the plot

_fig = plot_data(
    x_col="Ron_Gm_unit",
    y_col="v(Vbias)",
    filters={
        "v(VG)": 1.65,
        "v(VD)": 0.05,
        "length":[1e-5],
        "corner": "TT",
    },
    group_col=["i-sweep","v(VG)","length", "corner"],
    color_group=["i-sweep"],
    hover_cols=["v(Vbias)", "Ron_Gm_unit"],
    hover_all_cols=True,
    x_log=True,
    y_log=False,
    title="4.1a Vbias vs log(Ron/gm)",
    width=900,
    height=500,
    max_points=50000,
)
if _fig is not None: _fig.show()

In [ ]:
# DESIGN PLOT 4.1b: Vbias vs log(Ron/Gm)
# DESIGN QUESTION: 'What gate bias voltage achieves my target Ron/Gm ratio?'
# Filters: VGS=VDD (fully on), VDS=0.05V (triode), length=10u.
# Corner spread (TT/SS/FF) shows process sensitivity of Vbias.
# Designer reads V_DZN (NMOS deadzone voltage) directly from this plot.
# KEY ADVANTAGE: gm/ID cannot provide this Ron/Gm vs Vbias mapping pre-simulation.
# Use this to understand the plot

_fig = plot_data(
    x_col="Ron_Gm_unit",
    y_col="v(Vbias)",
    filters={
        "v(VG)": 1.65,
        "v(VD)": 0.05,
        "length": [1e-5],
    },
    group_col=["i-sweep","v(VG)","v(VD)","length", "corner"],
    color_group=[ "corner"],
    hover_cols=["v(Vbias)", "Ron_Gm_unit"],
    hover_all_cols=True,
    x_log=True,
    y_log=False,
    title="4.1b Vbias vs log(Ron/gm)",
    width=900,
    height=500,
    max_points=50000,
)
if _fig is not None: _fig.show()

In [ ]:
# DESIGN PLOT 4.1c: Vbias vs log(Ron/Gm)
# DESIGN QUESTION: 'What gate bias voltage achieves my target Ron/Gm ratio?'
# Filters: VGS=VDD (fully on), VDS=0.05V (triode), all 5 lengths, corner=TT, I-sweep=10u.
# Corner spread (TT/SS/FF) shows process sensitivity of Vbias.
# Designer reads V_DZN (NMOS deadzone voltage) directly from this plot.
# KEY ADVANTAGE: gm/ID cannot provide this Ron/Gm vs Vbias mapping pre-simulation.
# Use this to understand the plot

_fig = plot_data(
    x_col="Ron_Gm_unit",
    y_col="v(Vbias)",
    filters={
        "v(VG)": 1.65,
        "v(VD)": 0.05,
        "length": [2e-7, 5e-7, 1e-6, 4e-6, 1e-5],
        "corner": "TT",
        "i-sweep":1e-5
    },
    group_col=["i-sweep","v(VG)","v(VD)", "length"],
    color_group=[ "length"],
    hover_cols=["v(Vbias)", "Ron_Gm_unit"],
    hover_all_cols=True,
    hovermode="x unified",
    x_log=True,
    y_log=False,
    title="4.1c Vbias vs log(Ron/gm)",
    width=900,
    height=500,
    max_points=50000,
)
if _fig is not None: _fig.show()

In [ ]:

# DESIGN PLOT 4.1d: Vbias vs log(Ron/Gm)
# DESIGN QUESTION: 'What gate bias voltage achieves my target Ron/Gm ratio?'
# Filters: VGS=VDD (fully on), VDS=0.05V (triode), all 5 lengths.
# Corner spread (TT/SS/FF) shows process sensitivity of Vbias.
# Designer reads V_DZN (NMOS deadzone voltage) directly from this plot.
# KEY ADVANTAGE: gm/ID cannot provide this Ron/Gm vs Vbias mapping pre-simulation.
# Use this to design

_fig = plot_data(
    x_col="Ron_Gm_unit",
    y_col="v(Vbias)",
    filters={
        "v(VG)": 1.65,
        "v(VD)": 0.05,
    },
    group_col=["i-sweep","v(VG)","v(VD)", "length", "corner"],
    color_group=[ "corner", "length"],
    hover_cols=["v(Vbias)", "Ron_Gm_unit"],
    hover_all_cols=True,
    x_log=True,
    y_log=False,
    title="4.1d Vbias vs log(Ron/gm)",
    width=900,
    height=500,
    max_points=50000,
)
if _fig is not None: _fig.show()

## 4.1 $V_{bias}$ vs log($\frac{R_{on}}{g_m}$)
---

The gate bias voltage $V_{bias}$ of the diode-connected replica transistor varies with device geometry and bias current across all process corners. It is set by the diode connection as $V_{bias} = V_{TH} + V_{ov}$.


The replica is diode-connected in saturation, so from $g_m = 2I_D/V_{ov}$:

$$V_{ov} = \frac{2I_D}{g_m}$$

$$\therefore \quad V_{bias} = V_{TH} + \frac{2I_D}{g_m}$$

Multiplying and dividing by $R_{on}$:

$$\boxed{V_{bias} = V_{TH} + \frac{2I_D}{V_{swing}} \cdot I_{peak} \cdot \frac{R_{on}}{g_m}}$$

where $R_{on} = V_{swing}/I_{peak}$ is the switch resistance during the non-linear settling phase. This shows that **$V_{bias}$ is linear in $R_{on}/g_m$**, which on a logarithmic x-axis produces the exponential rise visible in the plots. Two design dependencies are immediately readable:

- **Higher $I_D$** at fixed $R_{on}/g_m$: more current demands more overdrive → $V_{bias}$ rises
- **Larger $R_{on}/g_m$** at fixed $I_D$: directly scales $V_{bias}$ upward along each trace

---

### Trends

> **Reading the plot:**
> - **Top → Bottom** at fixed $R_{on}/g_m$: $I_{sweep}$ **decreases**
>   (lower current → lower overdrive required → lower $V_{bias}$)
> - **Left → Right** along a fixed-$I_{sweep}$ trace: $L$ **increases**
>   (larger $R_{on}/g_m$ directly raises $V_{bias}$ through the linear relationship above)

As expected, the **SS corner** sits highest at any given $L$ and $I_{sweep}$: a slow-corner device has a larger $V_{TH}$ and lower $\mu C_{ox}$, both of which demand a higher gate voltage to source the same current. The FF corner sits lowest.

---

### Key Insight

$V_{bias}$ is the deadzone voltage, the output-stage transistor does not enter the small-signal settling region until its gate voltage crosses $V_{bias}$. The expression above makes the binding constraint explicit: **the SS corner at the largest $L$ and highest $I_{sweep}$ sets the worst-case $V_{bias}$**, which must be met by the preceding stage's output swing across all corners.

Width and current are coupled through $g_m$: increasing $W$ lowers $V_{bias}$ (through $g_m \uparrow$, so $V_{ov} \downarrow$) but also lowers $R_{on}$, directly affecting the non-linear RC settling speed. Reading this plot together with the $g_{m,\text{bias}}$ vs $R_{on}/g_m$ plot gives the complete picture, the target $R_{on}/g_m$ maps to a specific ($W$, $I_{sweep}$) pair, and the corresponding $V_{bias}$ here is the voltage that must be guaranteed across all corners for the output stage to transition into the small-signal phase.

---

In [ ]:
# DESIGN PLOT 4.2a: Vbias vs Width
# DESIGN QUESTION: 'What device width W corresponds to my chosen Vbias?'
# Filters: VGS=VDD (fully on), VDS=0.05V (triode), length=1u, corner=TT.
# From Plot 1, read target Vbias; enter this plot to get W_unit (unit width
# before multiplier M is applied).
# Use this to understand the plot

_fig = plot_data(
    x_col="width",
    y_col="v(Vbias)",
    filters={
        "v(VG)": 1.65,
        "v(VD)": 0.05,
        "length": [1e-6],
        "corner": "TT",
    },
    group_col=["length", "corner", "v(VG)", "v(VD)", "i-sweep"],
    color_group=["i-sweep"],
    hover_cols=["width", "v(Vbias)"],
    hover_all_cols=True,
    x_log=False,
    y_log=False,
    title="4.2a Vbias vs width",
    width=1100,
    height=500,
    max_points=50000,
)
if _fig is not None: _fig.show()

In [ ]:
# DESIGN PLOT 4.2b: Vbias vs Width Corner Spread
# DESIGN QUESTION: 'What device width W corresponds to my chosen Vbias?'
# Filters: VGS=VDD (fully on), VDS=0.05V (triode), length=1u, i-sweep=10u.
# From Plot 1, read target Vbias; enter this plot to get W_unit (unit width
# Use this to understand the plot

_fig = plot_data(
    x_col="width",
    y_col="v(Vbias)",
    filters={
        "v(VG)": 1.65,
        "v(VD)": 0.05,
        "length": [1e-6],
        "i-sweep":[1e-5],
    },
    group_col=["length", "corner", "v(VG)", "v(VD)", "i-sweep"],
    color_group=["corner"],
    hover_cols=["width", "v(Vbias)"],
    hover_all_cols=True,
    x_log=False,
    y_log=False,
    title="4.2b Vbias vs width",
    width=1100,
    height=500,
    max_points=50000,
)
if _fig is not None: _fig.show()

In [ ]:
# DESIGN PLOT 4.2c: Vbias vs Width Corner Spread
# DESIGN QUESTION: 'What device width W corresponds to my chosen Vbias?'
# Filters: VGS=VDD (fully on), VDS=0.05V (triode), length=1u, i-sweep=10u.
# From Plot 1, read target Vbias; enter this plot to get W_unit (unit width
# Use this to design

_fig = plot_data(
    x_col="width",
    y_col="v(Vbias)",
    filters={
        "v(VG)": 1.65,
        "v(VD)": 0.05,
    },
    group_col=["length", "corner", "v(VG)", "v(VD)", "i-sweep"],
    color_group=["corner", "length"],
    hover_cols=["width", "v(Vbias)"],
    hover_all_cols=True,
    x_log=False,
    y_log=False,
    title="4.2c Vbias vs width",
    width=1100,
    height=500,
    max_points=50000,
)
if _fig is not None: _fig.show()

## 4.2 $V_{bias}$ vs $Width$
---

The gate bias voltage $V_{bias}$ of the diode-connected replica transistor varies with device width $W$, across all bias currents $I_{sweep}$, VDS values, and process corners for a given channel length.

The diode-connected replica sets $V_{bias} = V_{TH} + V_{ov}$, where:

$$V_{bias} = V_{TH} + \sqrt{\frac{2I_D}{\mu C_{ox}(W/L)}} \qquad \Rightarrow \qquad V_{bias} - V_{TH} \propto \frac{1}{\sqrt{W}}$$

This $1/\sqrt{W}$ decay is the dominant feature of the plot, the rapid drop at small $W$ (1–5 µm) followed by a flattening at larger widths. Each trace is one combination of ($I_{sweep}$, $V_{DS}$, corner), and all follow the same curve shape, shifted vertically by their bias current.

- **Higher $I_{sweep}$** → trace shifts upward: more current demands higher overdrive to sustain it in the replica device
- **Larger $W$** → $V_{bias}$ decreases: a wider device carries the same current at a lower $V_{GS}$, reducing the required bias voltage
- **Corner spread** → SS corner (higher $V_{TH}$) sits above TT and FF at the same width and current, directly showing which corner demands the highest $V_{bias}$ to enter the small-signal settling region
- **Multiple $V_{DS}$ values overlaid** → the spread between $V_{DS}$ traces is small, confirming that $V_{bias}$ is primarily set by the diode-connected replica (which operates at $V_{DS} = V_{GS}$) and is relatively insensitive to the drain voltage of the main-path device

---

### Trends

> **Reading the plot:**
> - **Top → Bottom** at fixed $Width$: $Length$ and $I-sweep$ **decreases**
>   (lower current → lower overdrive required → lower $V_{bias}$)

As expected, the **SS corner** sits highest at any given $L$ and $I_{sweep}$: a slow-corner device has a larger $V_{TH}$ and lower $\mu C_{ox}$, both of which demand a higher gate voltage to source the same current. The FF corner sits lowest.

---

### Key Insight

With the **given $W$ and bias current, what $V_{bias}$ must be applied to the gate of the final-stage transistor to ensure it operates in the small-signal settling region?**

Width and current are coupled through $g_m$ increasing $W$ lowers $V_{bias}$ but also lowers $R_{on}$, which affects the non-linear RC settling speed (Plot 3). Reading Plot 2 together with Plot 1 ($V_{bias}$ vs $R_{on}/g_m$) gives the designer the complete picture: the target $R_{on}/g_m$ from Plot 1 maps to a specific ($W$, $I_{sweep}$) combination here, and the corresponding $V_{bias}$ is the deadzone voltage that must be met across all process corners for the IBA final stage to enter small-signal mode.

In [ ]:
# DESIGN PLOT 4.3a: log(I_peak) vs log(Ron/Gm)
# DESIGN QUESTION: 'How much peak current does the RC settling phase deliver?'
# Filters: VGS=VDD (fully on), VDS=0.05V (triode), length=1u, corner=TT.
# Higher I_NL → faster large-signal capacitor charging (faster slewing).
# Both axes log-scale: reveals the power-law relationship between I_NL and Ron/Gm.
# Use this to understand the plot

_fig = plot_data(
    x_col="Ron_Gm_unit",
    y_col="Ipeak_unit",
    filters={
        "v(VG)": 1.65,
        "v(VD)": 0.05,
        "length": 1e-6,
        "corner": "TT",
    },
    group_col=["i-sweep", "v(VG)", "v(VD)", "length", "corner"],
    color_group=[ "i-sweep"],
    hover_cols=["Ron_Gm_unit", "Ipeak"],
    hover_all_cols=True,
    x_log=True,
    y_log=True,
    title="4.3a log(I_NL) vs log(Ron/gm)",
    #y_label="log(I_NL) (A)",
    width=1100,
    height=500,
    max_points=50000,
)
if _fig is not None: _fig.show()

In [ ]:
# DESIGN PLOT 4.3b: log(I_peak) vs log(Ron/Gm)
# DESIGN QUESTION: 'How much peak current does the RC settling phase deliver?'
# Filters: VGS=VDD (fully on), VDS=0.05V (triode), length=1u, i-sweep=10u.
# Higher Ipeak → faster large-signal capacitor charging (faster slewing).
# Both axes log-scale: reveals the power-law relationship between Ipeak and Ron/Gm.
# Use this to understand the plot

_fig = plot_data(
    x_col="Ron_Gm_unit",
    y_col="Ipeak_unit",
    filters={
        "v(VG)": 1.65,
        "v(VD)": 0.05,
        "length": 1e-6,
        "i-sweep": [1e-5]
    },
    group_col=["i-sweep", "v(VG)", "v(VD)", "length", "corner"],
    color_group=[ "corner"],
    hover_cols=["Ron_Gm_unit", "Ipeak"],
    hover_all_cols=True,
    x_log=True,
    y_log=True,
    title="4.3b log(I_peak) vs log(Ron/gm)",
    width=1100,
    height=500,
    max_points=50000,
)
if _fig is not None: _fig.show()

In [ ]:
# DESIGN PLOT 4.3c: log(I_peak) vs log(Ron/Gm)
# DESIGN QUESTION: 'How much peak current does the RC settling phase deliver?'
# Filters: VGS=VDD (fully on), VDS=0.05V (triode), i-sweep=10u.
# Higher Ipeak → faster large-signal capacitor charging (faster slewing).
# Both axes log-scale: reveals the power-law relationship between Ipeak and Ron/Gm.
# Use this to understand the plot

_fig = plot_data(
    x_col="Ron_Gm_unit",
    y_col="Ipeak_unit",
    filters={
        "v(VG)": 1.65,
        "v(VD)": 0.05,
        "i-sweep": [1e-5]
    },
    group_col=["i-sweep", "v(VG)", "v(VD)", "length", "corner"],
    color_group=[ "length"],
    hover_cols=["Ron_Gm_unit", "Ipeak"],
    hover_all_cols=True,
    x_log=True,
    y_log=True,
    title="4.3c log(I_peak) vs log(Ron/gm)",
    width=1100,
    height=500,
    max_points=50000,
)
if _fig is not None: _fig.show()

In [ ]:
# DESIGN PLOT 4.3d: log(Ipeak) vs log(Ron/Gm)
# DESIGN QUESTION: 'How much peak current does the RC settling phase deliver?'
# Filters: VGS=VDD (fully on), VDS=0.05V (triode), i-sweep=10u.
# Higher Ipeak → faster large-signal capacitor charging (faster slewing).
# Both axes log-scale: reveals the power-law relationship between Ipeak and Ron/Gm.
# Use this to design

_fig = plot_data(
    x_col="Ron_Gm_unit",
    y_col="Ipeak_unit",
    filters={
        "v(VG)": 1.65,
        "v(VD)": 0.05,
    },
    group_col=["i-sweep", "v(VG)", "v(VD)", "length", "corner"],
    color_group=[ "corner", "length"],
    hover_cols=["Ron_Gm_unit", "Ipeak"],
    hover_all_cols=True,
    x_log=True,
    y_log=True,
    title="4.3d log(I_peak) vs log(Ron/gm)",
    width=1100,
    height=500,
    max_points=50000,
)
if _fig is not None: _fig.show()

## 4.3 Log($I_{peak}$) vs Log($\frac{R_{on}}{g_m}$)
---

**$I_{peak}$** is the initial large-signal current that flows through the output-stage transistor at the onset of the settling transient. This current charges the load capacitor $C_L$ during the non-linear RC phase it determines how fast the output moves toward its final value before the small-signal exponential phase takes over.

In the linear (triode) region, the output-stage transistor on-resistance is:
$$R_{on} = \frac{L}{\mu C_{ox} W V_{ov}} \propto \frac{L}{W I_D}$$

Combining these:
$$\boxed{I_{peak} = \frac{V_{swing}}{R_{on}} = V_{swing} \cdot g_m \propto \frac{1}{R_{on}/g_m}}$$

On a log-log scale this gives:
$$\log(I_{peak}) = -\log\!\left(\frac{R_{on}}{g_m}\right) + \text{const}$$

which is approximately a **straight line with slope $-1$** seen in the plot. The spread between TT/SS/FF traces at each bias current reflects the process-corner sensitivity of $I_{peak}$ - the FF corner (green) consistently delivers higher $I_{peak}$ than SS (red) at the same $R_{on}/g_m$, since a faster process has lower $R_{on}$ and higher carrier mobility, both of which increase the initial charging current. This corner spread grows at larger $R_{on}/g_m$ values (longer $L$ or lower $I_D$), making worst-case SS-corner $I_{peak}$ the binding constraint when sizing for non-linear settling speed.

---
### Trends

> **Reading the plot:**
> - **Top → Bottom** at fixed $R_{on}/g_m$: $I_{sweep}$ **decreases**  
>   (lower current → lower $I_{peak}$)
>
> - **Left → Right** along a fixed-$I_{sweep}$ trace: $R_{on}/g_m$ **increases**  
>   ($L \uparrow$ or $W \downarrow$ → $R_{on} \uparrow$ → $I_{peak} = V_{swing}/R_{on} \downarrow$ → slower initial charging)

---

- **Moving Right ($R_{on}/g_m \uparrow$):**  
  $R_{on} \uparrow$ → $I_{peak} \downarrow$ → slower non-linear settling  

- **Moving Left ($R_{on}/g_m \downarrow$):**  
  $R_{on} \downarrow$ → $I_{peak} \uparrow$ → faster initial charging  

- **Each trace = fixed $I_{sweep}$:**  
  Higher $I_{sweep}$ shifts curves upward (higher $I_{peak}$)

- **Corners:**  
  **SS** sits lowest (higher $R_{on}$ → lower $I_{peak}$),  
  **FF** sits highest (lower $R_{on}$ → higher $I_{peak}$)
---

### Key Insight

The geometry dependence of $R_{on}/g_m$ is explicit:

$$\frac{R_{on}}{g_m} \propto \frac{L^2}{W I_D}$$

This means $L$ is the dominant geometry knob for the non-linear, but the replica bias path allows $I_D$ to be adjusted independently tuning $g_m$ and small-signal bandwidth without disturbing the main-path $R_{on}$. This is the pre-simulation design handle that the $g_m/I_D$ methodology does not provide.

---

In [ ]:
# DESIGN PLOT 4.4a: gmbias_unit vs log(Ron/Gm)
# DESIGN QUESTION: 'What is the bias-path gm at my target operating point?'
# Filters: VGS=VDD (fully on), VDS=0.05V (triode), corner=TT.
# Sets small-signal settling bandwidth: BW = gmbias / (2π × C_load).
# Verifying gmbias ensures small-signal phase completes within the time budget.
# Use this to understand the plot

_fig = plot_data(
    x_col="Ron_Gm_unit",
    y_col="gmbias_unit",
    filters={
        "v(VG)": 1.65,
        "v(VD)": 0.05,
        "corner": "TT",
    },
    group_col=["i-sweep", "v(VG)","v(VD)", "length" ],
    #color_group=["corner"],
    color_group=["i-sweep"],
    hover_cols=["Ron_Gm_unit", "gmbias_unit"],
    hover_all_cols=True,
    x_log=True,
    y_log=False,
    title="4.4a gmbias_unit vs log(Ron/gm)",
    width=1100,
    height=500,
    max_points=50000,
)
if _fig is not None: _fig.show()

In [ ]:
# DESIGN PLOT 4.4b: log(gmbias_unit) vs log(Ron/Gm)
# DESIGN QUESTION: 'What is the bias-path gm at my target operating point?'
# Filters: VGS=VDD (fully on), VDS=0.05V (triode), corner=TT.
# Sets small-signal settling bandwidth: BW = gmbias / (2π × C_load).
# Verifying gmbias ensures small-signal phase completes within the time budget.
# Use this to understand the plot

_fig = plot_data(
    x_col="Ron_Gm_unit",
    y_col="gmbias_unit",
    filters={
        "v(VG)": 1.65,
        "v(VD)": 0.05,
        "corner": "TT",
    },
    group_col=["i-sweep", "v(VG)","v(VD)", "length" ],
    #color_group=["corner"],
    color_group=["i-sweep"],
    hover_cols=["Ron_Gm_unit", "gmbias_unit"],
    hover_all_cols=True,
    x_log=True,
    y_log=True,
    title="4.4b log(gmbias_unit) vs log(Ron/gm)",
    width=1100,
    height=500,
    max_points=50000,
)
if _fig is not None: _fig.show()

In [ ]:
# DESIGN PLOT 4.4c: log(gmbias_unit) vs log(Ron/Gm)
# DESIGN QUESTION: 'What is the bias-path gm at my target operating point?'
# Filters: VGS=VDD (fully on), VDS=0.05V (triode).
# Sets small-signal settling bandwidth: BW = gmbias / (2π × C_load).
# Verifying gmbias ensures small-signal phase completes within the time budget.
# Use this to Design

_fig = plot_data(
    x_col="Ron_Gm_unit",
    y_col="gmbias_unit",
    filters={
        "v(VG)": 1.65,
        "v(VD)": 0.05,
    },
    group_col=["i-sweep", "v(VG)","v(VD)", "length", "corner" ],
    color_group=["corner", "length"],
    hover_cols=["Ron_Gm_unit", "gmbias_unit"],
    hover_all_cols=True,
    x_log=True,
    y_log=False,
    title="4.4c log(gmbias_unit) vs log(Ron/gm)",
    width=1100,
    height=500,
    max_points=50000,
)
if _fig is not None: _fig.show()

## 4.4 $g_{m,\text{bias}}$ vs Log($R_{on}/g_m$)
---

**$g_{m,\text{bias}}$** is the transconductance of the transistor in the replica bias branch, the same device as the unit output-stage transistor. It sets the small-signal settling bandwidth and appears as the $g_m$ denominator in the $R_{on}/g_m$ design parameter. Because the bias branch mirrors the output transistor, tuning $I_{sweep}$ directly controls $g_{m,\text{bias}}$ without disturbing the main signal path.

These two quantities are evaluated at **different operating points**: $R_{on}$ is extracted with the switch fully on in triode ($V_{GS} = V_{DD}$, $V_{DS} = 50\,\text{mV}$), while $g_{m,\text{bias}}$ is evaluated in saturation at the replica bias current — a different overdrive from $V_{GS,\text{sw}} - V_{TH}$:

$$R_{on} = \frac{L}{\mu C_{ox} W (V_{GS,\text{sw}} - V_{TH})}, \qquad g_{m,\text{bias}} = \sqrt{2\mu C_{ox}\frac{W}{L}I_D} = \frac{2I_D}{V_{ov,\text{bias}}}$$

Because $V_{ov,\text{bias}} \neq V_{GS,\text{sw}} - V_{TH}$, the product $R_{on} \cdot g_{m,\text{bias}}$ is **not constant**. Eliminating $L$ at fixed $W$ and $I_D$, using $R_{on} \propto L$, $g_{m,\text{bias}} \propto L^{-1/2}$, so $R_{on}/g_m \propto L^{3/2}$ gives the fundamental scaling:

$$g_{m,\text{bias}} \propto \left(\frac{R_{on}}{g_m}\right)^{-1/3}$$

This is a power law with exponent $-1/3$: on a log-log axis (Plot 4.4b) it appears as a **straight line of slope $-1/3$**, while on the semilog axis (Plot 4.4a) it produces a concave falling curve, the logarithmic x-axis compresses the right side, making the decay appear steeper than the underlying exponent.

- **Higher $I_{sweep}$** → curve shifts upward: more bias current raises $g_{m,\text{bias}} = 2I_D/V_{ov}$ at every $R_{on}/g_m$ value ($g_{m,\text{bias}} \propto \sqrt{I_D}$, so doubling $I_{sweep}$ lifts the curve by $\sqrt{2}$)
- **Left → Right** along a fixed-$I_{sweep}$ trace: $L$ increases ($R_{on}/g_m \propto L^{3/2}$ pushes the operating point rightward while $g_{m,\text{bias}} \propto L^{-1/2}$ pulls it downward)
- **Corner spread** → process corners shift $\mu C_{ox}$ and $V_{TH}$, vertically separating the constant-$I_D$ curves; the SS corner sits highest at any given $(W, L, I_D)$

---

### Geometry Universality at Fixed $I_D$

The same $-1/3$ exponent holds whether $W$ or $L$ is varied any $(W, L)$ pair mapping to the same $R_{on}/g_m$ value at the same $I_D$ produces the same $g_{m,\text{bias}}$. The traces for different lengths therefore do not form separate curves; each length accesses a **different segment of the same universal curve**, and all collapse onto a single locus in the $(R_{on}/g_m,\, g_{m,\text{bias}})$ plane.

> **Reading the plot:**
> - **Top → Bottom** at fixed $R_{on}/g_m$: $I_{sweep}$ **decreases**
>   (lower bias current → lower $g_{m,\text{bias}} = 2I_D/V_{ov,\text{bias}}$)
> - **Left → Right** along a fixed-$I_{sweep}$ trace: $L$ **increases**
>   ($R_{on}/g_m \propto L^{3/2}$ grows rapidly, extending the trace rightward and downward along the same universal curve)

---

### Key Insight

**Given a target small-signal time constant $\tau_{ss} = C_L / g_{m,\text{bias}}$, what bias current is required without committing to a specific $(W, L)$?**

Because $g_{m,\text{bias}}$ is uniquely determined by $R_{on}/g_m$ and $I_D$ alone, the required $I_{sweep}$ can be read directly from this plot once $\tau_{ss}$ is specified. Width is then chosen independently to meet the $R_{on}/g_m$ target (non-linear phase speed), completing the two-axis co-design that $g_m/I_D$ alone cannot decouple.

In [ ]:
# DESIGN PLOT 4.5a: Gm/Id vs log(Ron/Gm)
# DESIGN QUESTION: 'What power-efficiency operating point am I at?'
# Filters: VGS=VDD (fully on), VDS=0.05V (triode), length=1u, corner=TT.
# Bridges the Ron/Gm and gm/ID methodologies: enter from either axis.
# This is why Ron/Gm is described as 'subsuming' gm/ID for dynamic amplifiers.
# Use this to understand the plot

_fig = plot_data(
    x_col="Ron_Gm_unit",
    y_col="GmoverId",
    filters={
        "v(VG)": 1.65,
        "v(VD)": 0.05,
        "length": 1e-6,
        "corner":"TT"
    },
    group_col=["i-sweep", "v(VG)", "v(VD)", "length", "corner"],
    color_group=["i-sweep"],
    hover_cols=["Ron_Gm_unit", "Gmoverid"],
    hover_all_cols=True,
    x_log=True,
    y_log=False,
    title="4.5a Gm/Id vs log(Ron/gm)",
    width=1100,
    height=500,
    max_points=50000,
)
if _fig is not None: _fig.show()

In [ ]:
# DESIGN PLOT 4.5b: Gm/Id vs log(Ron/Gm)
# DESIGN QUESTION: 'What power-efficiency operating point am I at?'
# GmoverId = gm/ID (V⁻¹). High → subthreshold (efficient); low → strong inversion.
# Bridges the Ron/Gm and gm/ID methodologies: enter from either axis.
# This is why Ron/Gm is described as 'subsuming' gm/ID for dynamic amplifiers.
# Use this to understand the plot

_fig = plot_data(
    x_col="Ron_Gm_unit",
    y_col="GmoverId",
    filters={
        "v(VG)": 1.65,
        "v(VD)": 0.05,
        "length": 1e-6,
        #"corner":"TT",
        "i-sweep": [1e-5]
    },
    group_col=["i-sweep", "v(VG)", "v(VD)", "length", "corner"],
    color_group=["corner"],
    hover_cols=["Ron_Gm_unit", "Gmoverid"],
    hover_all_cols=True,
    x_log=True,
    y_log=False,
    title="4.5b Gm/Id vs log(Ron/gm)",
    width=1100,
    height=500,
    max_points=50000,
)
if _fig is not None: _fig.show()

In [ ]:
# DESIGN PLOT 4.5c: Gm/Id vs log(Ron/Gm)
# DESIGN QUESTION: 'What power-efficiency operating point am I at?'
# GmoverId = gm/ID (V⁻¹). High → subthreshold (efficient); low → strong inversion.
# Bridges the Ron/Gm and gm/ID methodologies: enter from either axis.
# This is why Ron/Gm is described as 'subsuming' gm/ID for dynamic amplifiers.
# Use this to understand the plot

_fig = plot_data(
    x_col="Ron_Gm_unit",
    y_col="GmoverId",
    filters={
        "v(VG)": 1.65,
        "v(VD)": 0.05,
        "corner":"TT",
        "i-sweep": [1e-5]
    },
    group_col=["i-sweep", "v(VG)", "v(VD)", "length", "corner"],
    color_group=["length"],
    hover_cols=["Ron_Gm_unit", "Gmoverid"],
    hover_all_cols=True,
    hovermode="x unified",
    x_log=True,
    y_log=False,
    title="4.5c Gm/Id vs log(Ron/gm)",
    width=1100,
    height=500,
    max_points=50000,
)
if _fig is not None: _fig.show()

In [ ]:
# DESIGN PLOT 4.5d: Gm/Id vs log(Ron/Gm)
# DESIGN QUESTION: 'What power-efficiency operating point am I at?'
# GmoverId = gm/ID (V⁻¹). High → subthreshold (efficient); low → strong inversion.
# Bridges the Ron/Gm and gm/ID methodologies: enter from either axis.
# This is why Ron/Gm is described as 'subsuming' gm/ID for dynamic amplifiers.
# Use this to Design

_fig = plot_data(
    x_col="Ron_Gm_unit",
    y_col="GmoverId",
    filters={
        "v(VG)": 1.65,
        "v(VD)": 0.05,
    },
    group_col=["i-sweep", "v(VG)", "v(VD)", "length", "corner"],
    color_group=["corner","length"],
    hover_cols=["Ron_Gm_unit", "Gmoverid"],
    hover_all_cols=True,
    x_log=True,
    y_log=False,
    title="4.5d Gm/Id vs log(Ron/gm)",
    width=1100,
    height=500,
    max_points=50000,
)
if _fig is not None: _fig.show()

## 4.5 $g_m/I_D$ vs Log($\frac{R_{on}}{g_m}$)
---

- Here we look at the $g_m/I_D$ of the transistor as a function of the $R_{on}/g_m$ design metric, sweeping bias current $I_{sweep}$ across process corners.

- We can see that $g_m/I_D$ decreases as $R_{on}/g_m$ increases

- $R_{on}/g_m \propto L^2/WI_D$ and $g_m/I_D = 2/V_{ov}$ as $R_{on}/g_m$ increases (wider device, lower $I_D$, or longer $L$), the overdrive $V_{ov}$ increases, pushing the device deeper into strong inversion and reducing transconductance efficiency:

$$\frac{g_m}{I_D} = \frac{2}{V_{ov}} \propto \frac{1}{\sqrt{R_{on}/g_m \cdot I_D}}$$

The negative slope seen in both plots directly confirms this, moving right on the x-axis corresponds to moving from moderate inversion toward strong inversion, where the device is less efficient per unit current.

The bias current is the primary design knob in this plot. Each trace corresponds to a fixed $I_{sweep}$:

- **Higher $I_{sweep}$** → trace shifts **downward** at the same $R_{on}/g_m$,  more current for the same geometry means the device operates deeper into strong inversion, reducing $g_m/I_D$. The $R_{on}/g_m$ operating range of the trace remains the same; only the efficiency drops.
- **Lower $I_{sweep}$** → trace shifts **upward** at the same $R_{on}/g_m$, less current keeps the device closer to moderate inversion, improving $g_m/I_D$ while the RC settling speed (set by geometry) is unchanged.

For a given geometry ($W$, $L$), the designer chooses $I_{sweep}$ to trade between transconductance efficiency and settling speed, this trade-off is directly readable from the plot.

This plot **connects the $R_{on}/g_m$ and $g_m/I_D$ methodologies**, and the relationship is strictly one-directional:

> **From $R_{on}/g_m$ → you can read $g_m/I_D$ directly off this plot.**
> **From $g_m/I_D$ → you cannot recover $R_{on}/g_m$ without additional simulation.**

The reason is fundamental that $g_m/I_D$ characterises only the small-signal operating point. It contains no information about $R_{on}$, which depends separately on $V_{ov}$, $W$, $L$, and the large-signal drain voltage. A designer entering from $g_m/I_D$ can size the device for bandwidth, but has no pre-simulation handle on where that size sits on the $R_{on}/g_m$ axis and therefore no visibility into the non-linear RC settling phase.

Entering from $R_{on}/g_m$ subsumes the $g_m/I_D$ design entirely: once $R_{on}/g_m$ and $I_{sweep}$ are chosen, this plot immediately tells the designer what $g_m/I_D$ and therefore what inversion region and power efficiency corresponds to that operating point. **The $R_{on}/g_m$ methodology is therefore a strict superset of the $g_m/I_D$ methodology for dynamic amplifier design.**

---

### Key Insight

A designer entering from $g_m/I_D$ can size the device for bandwidth, but has no pre-simulation visibility into Non-Linear settling information ($R_{on}$), the deadzone bias voltages, or worst-case corner behaviour all of which require additional SPICE runs to discover. Entering from $R_{on}/g_m$ subsumes this entirely: once $R_{on}/g_m$ and $I_{sweep}$ are fixed, $g_m/I_D$ is directly readable from this plot and both settling phases are co-designed in a single step. The $g_m/I_D$ axis is a projection of the $R_{on}/g_m$ design space onto a single dimension, every $g_m/I_D$ point maps to a unique $R_{on}/g_m$ point, but the reverse does not exist without simulation.

In [ ]:
# DESIGN PLOT 4.6a: Vswing vs log(Ron/Gm)
# DESIGN QUESTION: 'What output swing is achievable at this operating point?'
# Filters: VGS=VDD (fully on), VDS=0.05V (triode), length=1u, corner=TT.
# vswing must be ≥ the required full-scale signal swing.
# If vswing is too small, increase W or L at the target Ron/Gm.
# SS corner (usually) gives worst-case (smallest) swing.
# Use this to understand the plot

_fig = plot_data(
    x_col="Ron_Gm_unit",
    y_col="vswing",
    filters={
        "v(VG)": 1.65,
        "v(VD)": 0.05,
        "length": [1e-6],
        "corner":"TT"
    },
    group_col=["i-sweep", "v(VG)", "v(VD)", "length", "corner"],
    color_group=["i-sweep"],
    hover_cols=["Ron_Gm_unit", "vswing"],
    hover_all_cols=True,
    x_log=True,
    y_log=False,
    title="4.6a vswing vs log(Ron/gm)",
    width=1100,
    height=500,
    max_points=50000,
)
if _fig is not None: _fig.show()

In [ ]:
# DESIGN PLOT 4.6b: Vswing vs log(Ron/Gm)
# DESIGN QUESTION: 'What output swing is achievable at this operating point?'
# Filters: VGS=VDD (fully on), VDS=0.05V (triode), length=10u.
# vswing must be ≥ the required full-scale signal swing.
# If vswing is too small, increase W or L at the target Ron/Gm.
# SS corner (usually) gives worst-case (smallest) swing.
# Use this to understand the plot

_fig = plot_data(
    x_col="Ron_Gm_unit",
    y_col="vswing",
    filters={
        "v(VG)": 1.65,
        "v(VD)": 0.05,
        "length": [1e-6],
    },
    group_col=["i-sweep", "v(VG)", "v(VD)", "length", "corner"],
    color_group=["corner"],
    hover_cols=["Ron_Gm_unit", "vswing"],
    hover_all_cols=True,
    x_log=True,
    y_log=False,
    title="4.6b vswing vs log(Ron/gm)",
    width=1100,
    height=500,
    max_points=50000,
)
if _fig is not None: _fig.show()

In [ ]:
# DESIGN PLOT 4.6c: Vswing vs log(Ron/Gm)
# DESIGN QUESTION: 'What output swing is achievable at this operating point?'
# Filters: VGS=VDD (fully on), VDS=0.05V (triode), all 5 lengths, corner=TT, I-sweep=10u.
# vswing must be ≥ the required full-scale signal swing.
# If vswing is too small, increase W or L at the target Ron/Gm.
# SS corner (usually) gives worst-case (smallest) swing.
# Use this to understand the plot

_fig = plot_data(
    x_col="Ron_Gm_unit",
    y_col="vswing",
    filters={
        "v(VG)": 1.65,
        "v(VD)": 0.05,
        "length": [2e-7, 5e-7, 1e-6, 4e-6, 1e-5],
        "corner": "TT",
        "i-sweep":1e-5
    },
    group_col=["i-sweep", "v(VG)", "v(VD)", "length", "corner"],
    color_group=["length"],
    hover_cols=["Ron_Gm_unit", "vswing"],
    hover_all_cols=True,
    x_log=True,
    y_log=False,
    title="4.6c vswing vs log(Ron/gm)",
    width=1100,
    height=500,
    max_points=50000,
)
if _fig is not None: _fig.show()

In [ ]:
# DESIGN PLOT 4.6d: Vswing vs log(Ron/Gm)
# DESIGN QUESTION: 'What output swing is achievable at this operating point?'
# Filters: VGS=VDD (fully on), VDS=0.05V (triode), all 5 lengths.
# vswing must be ≥ the required full-scale signal swing.
# If vswing is too small, increase W or L at the target Ron/Gm.
# SS corner (usually) gives worst-case (smallest) swing.
# Use this to design

_fig = plot_data(
    x_col="Ron_Gm_unit",
    y_col="vswing",
    filters={
        "v(VG)": 1.65,
        "v(VD)": 0.05,
    },
    group_col=["i-sweep", "v(VG)", "v(VD)", "length", "corner"],
    color_group=["corner","length"],
    hover_cols=["Ron_Gm_unit", "vswing"],
    hover_all_cols=True,
    x_log=True,
    y_log=False,
    title="4.6d vswing vs log(Ron/gm)",
    width=1100,
    height=500,
    max_points=50000,
)
if _fig is not None: _fig.show()

## 4.6 $V_{swing}$ vs Log($\frac{R_{on}}{g_m}$)
---

**$V_{swing}$** is defined as:

$$V_{swing} = |v(VG) - v(Vbias)| = v(VG) - v(Vbias) \quad \text{(when } v(VG) > v(Vbias)\text{)}$$

It represents the gate drive headroom beyond the quiescent bias point the voltage the gate must traverse before the output-stage transistor enters the small-signal settling region. During non-linear settling, the dynamic amplifier sees an initial large-signal gate swing close to $V_{DD}$; as the output approaches its final value and the input voltage to the final-stage devices $V_{GP} = v(Vbias)_P$ and $V_{GN} = v(Vbias)_N$, $V_{swing}$ equals zero, at which point $I_{out} = 0$ and the circuit transitions to small-signal mode.

In the triode region, $R_{on}$ depends on overdrive voltage, which itself depends on how far the gate is driven above the bias point:

$$R_{on} = \frac{L}{\mu C_{ox} W (v(VG) - V_{th})} \qquad \Rightarrow \qquad R_{on} \downarrow \text{ as } v(VG) \uparrow$$

A larger $V_{swing} = v(VG) - v(Vbias)$ means the transistor is driven harder into the triode region resulting in lower $R_{on}$, faster RC settling. This is why the plot shows a **negative slope**: larger $V_{swing}$ corresponds to smaller $R_{on}/g_m$.

$$\boxed{V_{swing} = v(VG) - v(Vbias) \propto \frac{1}{\sqrt{R_{on}/g_m}}}$$

At $v(VG) = V_{DD} = 1.65\,\text{V}$ (the fully-on condition used in earlier design plots), this simplifies to:

$$V_{swing} = V_{DD} - v(Vbias)$$

confirming that the available swing equals the gate rail minus the quiescent bias a quantity entirely predictable from the LUT without transient simulation.

A common assumption in dynamic amplifier design is that the output-stage transistor always sees a full-rail swing ($V_{swing} \approx V_{DD}$) during the non-linear phase. **This is not always true.**

Consider a designer who targets a small $R_{on}/g_m$ expecting fast non-linear settling. From Plot 4.3, small $R_{on}/g_m$ implies large $I_{peak}$ and fast RC charging. But Plot 4.6 reveals the other side: **a small $R_{on}/g_m$ is only achievable if the input swing is large enough to drive the gate far above $v(Vbias)$**. If the actual input signal never delivers that swing for example, in a low-voltage or small-signal application the transistor never reaches the operating point the designer assumed, and the expected settling speed is not achieved.

Conversely, at large $R_{on}/g_m$ (right side of the plot), $V_{swing}$ is small, the gate barely rises above $v(Vbias)$, the transistor stays near the edge of the linear region, and the non-linear RC phase contributes little to nothing in the overall settling. The small-signal phase dominates.

This plot therefore answers: **for a given $R_{on}/g_m$ target, what input swing must the circuit actually see to operate at that point?**

---

### Trends

> **Reading the plot:**
> - **Top → Bottom** at fixed $R_{on}/g_m$: $I_{sweep}$ **increases**  
>   (more bias current → higher $v(Vbias)$ → less headroom → smaller $V_{swing}$)
>
> - **Left → Right** along a fixed-$I_{sweep}$ trace: $R_{on}/g_m$ **increases**  
>   ($L \uparrow$ or $W \downarrow$ → $R_{on} \uparrow$ → device driven less hard → $V_{swing} \downarrow$)

---

- **Moving Left ($R_{on}/g_m \downarrow$):**  
  $V_{swing} \uparrow$ - gate is driven hard above $v(Vbias)$, transistor deep in triode, $R_{on}$ low, fast non-linear settling. Only achievable if the input signal delivers sufficient swing.

- **Moving Right ($R_{on}/g_m \uparrow$):**  
  $V_{swing} \downarrow$ - gate barely clears $v(Vbias)$, non-linear phase contributes little, small-signal settling dominates.

- **Higher $I_{sweep}$** raises $v(Vbias)$ (more current needs more gate drive), reducing $V_{swing}$ at the same geometry, traces shift downward.

- **Longer $L$** pushes traces rightward (higher $R_{on}/g_m \propto L^2$) with lower $V_{swing}$ at the same $I_{sweep}$.

- **Corners:** SS corner gives lowest $V_{swing}$ at the same $R_{on}/g_m$ and higher $V_{th}$ raises $v(Vbias)$, eating into the available headroom. This is the worst-case corner for non-linear settling speed, consistent with Plot 4.1.

---

### Key Insight

$V_{swing}$ is not a free parameter, it is set by the application's input signal amplitude and the bias point simultaneously. This plot makes visible a constraint that the $g_m/I_D$ methodology cannot surface: **there is a minimum required input swing to operate at any given $R_{on}/g_m$ target**. A designer who picks a small $R_{on}/g_m$ for fast settling must verify that the circuit actually sees the corresponding $V_{swing}$ otherwise the assumed operating point is never reached and the settling speed budget is violated. Reading Plot 4.6 alongside Plot 4.1 ($V_{bias}$ vs $R_{on}/g_m$) and Plot 4.3 ($I_{peak}$ vs $R_{on}/g_m$) gives the complete pre-simulation picture of the non-linear settling phase.


---

An enhanced and more interactive version of this plotting tool has been developed by the authors and is available online. **Reviewers are strongly encouraged to explore this tool**, as it provides a more intuitive interface for analysis. Users can create custom expressions, modify plots, and interact with the visualizations in a highly flexible manner, offering insights beyond the figures presented in this notebook.  

<div align="center">

**Website:** <a href="https://wrongm.azurewebsites.net" target="_blank"><b>wrongm.azurewebsites.net</b></a>

</div>



---
# 5. Design of Inverter-Based Amplifier

---

Let's compare the performance of an inverter-based amplifier (IBA), which is inherently a dynamic amplifier, designed using traditional $g_m/I_D$ and $R_{on}/g_m$ based methodologies and try to understand that which methodology provides more useful design insights into Dynamic amplifiers and also reduce the number of design/simulation iteration user has to do to reach the final specification.

<div align="center">

<img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/ckt_images/fig_7_IBA_schematic.png" width="900">

**Figure 9:** Schematic of Inverter-Based Amplifier(IBA) in Capacitive feedback.

</div>

---

> **Purpose of this section:** This section designs the same IBA using the *conventional $g_m/I_D$ approach* as a **baseline for comparison**. The core limitation of this approach that $g_m/I_D$ gives no information about $R_{on}$ or the non-linear RC settling phase or the bias deadzones for dynamic amplifiers will become clear once we compare the results against the $R_{on}/g_m$ design in the next section.

---

### Specifications / Design Goals

| Parameter | Symbol | Value | Notes |
|---|---|---|---|
| Supply voltage | $V_{DD}$ | 1.65 V | IHP SG13G2 nominal |
| Settling time | $T_{settle}$ | 250 ns | 5$\tau$, 99.3% settled |
| Settling time constant | $\tau_{cl}$ | 50 ns | $T_{settle}/5$ |
| Closed-loop bandwidth | $\omega_{cl}$ | 20 Mrad/s | $1/\tau_{cl}$ |
| OTA unity-gain bandwidth | UGB | 9.55 MHz (60 Mrad/s) | $\omega_{cl}/\beta$ |
| Target transconductance | $G_m$ | 40 µA/V | $\text{UGB} \times C_{L,eff}$ |
| OTA DC gain | $A_0$ | 50 | $g_m/g_{ds}$ limit, lv\_nmos |
| Sampling capacitor | $C_s$ | 500 fF | |
| Feedback capacitor | $C_f$ | 250 fF | |
| Load capacitor | $C_l$ | 500 fF | |
| Effective load | $C_{L,eff}$ | 667 fF | $C_l + C_f C_s/(C_s+C_f)$ |
| Feedback factor | $\beta$ | 1/3 | $C_f/(C_s+C_f)$ |
| Closed-loop gain (ideal) | $G_{cl}$ | $-2$ V/V | $-C_s/C_f$ |
| Closed-loop gain (actual) | $G_{cl}$ | $-1.887$ V/V | Finite $A_0$ effect |
| Input bias point | $V_{in,dc}$ | 825 mV | $V_{DD}/2$, inverter switching threshold |
| Input swing | $\Delta V_{in}$ | $\pm 200$ mV | $V_{DD}/2 \pm 200$ mV |

**Large-signal condition:** Input kick at virtual ground $\Delta V_x = \pm 133$ mV $> V_{ov,lv\_nmos} \approx 100$ mV, guaranteeing the amplifier enters the slewing (large-signal) regime on every cycle before transitioning to exponential small-signal settling. This is the regime of interest for the $R_{on}/G_m$ methodology.

---


### Design of IBA using Traditional ($g_m/I_D$) Methodology

---

- For a settling time $T_{settle} = 250\,\text{ns}$ (5$\tau$), the required closed-loop bandwidth is:
  $$
  \omega_{cl} = \frac{1}{\tau_{cl}} = \frac{5}{T_{settle}} = 20\,\text{Mrad/s}, \qquad \text{UGB} = \frac{\omega_{cl}}{\beta} = \frac{20\,\text{Mrad/s}}{1/3} = 60\,\text{Mrad/s}
  $$

- The total required transconductance is:
  $$
  G_m = \text{UGB} \times C_{L,eff} = 60 \times 10^6 \times 667 \times 10^{-12} \approx 40\,\mu\text{S}
  $$

- Assume NMOS and PMOS contribute equally:
  $$
  g_{mn} = g_{mp} = \frac{G_m}{2} = 20\,\mu\text{S}
  $$

- With multiplier $m = 4$, the unit cell (replica path) must provide:
  $$
  g_{m,unit} = \frac{20\,\mu\text{S}}{4} = 5\,\mu\text{S}
  $$

- Replica path quiescent current $I_q = 0.5\,\mu\text{A}$, signal path current $= m \times I_q = 2\,\mu\text{A}$. This gives:
  $$
  \frac{g_{m,unit}}{I_q} = \frac{5\,\mu\text{S}}{0.5\,\mu\text{A}} = 10\,\frac{\text{S}}{\text{A}}
  $$

- Target $g_m/I_D = 10\,\text{V}^{-1}$ for both LV_NMOS and LV_PMOS — corresponding to **moderate inversion**, where large-signal and small-signal settling contributions are both significant.

- Assume $g_m/g_{ds} = 50$ (intrinsic gain limit of lv\_nmos in IHP SG13G2)

---

- Using our `gmid_IHP130` model to extract device sizes:
  - Target $g_m/I_D = 10\,\text{V}^{-1}$
  - Target $g_m/g_{ds} = 50$
  - Run once for `LV_NMOS`, once for `LV_PMOS`

---

> **➜ Design Step - Use the Gm/Id Design Helper immediately below**
>
> Enter the targets derived above into the **Gm/Id Design Helper**:
>
> | Slider | Value | Why |
> |:--|:--|:--|
> | **Target gm/id** | 10 | $g_{m,unit}/I_q = 5\,\mu\text{S} / 0.5\,\mu\text{A}$ |
> | **Target gm/gds** | 50 | intrinsic gain limit, lv\_nmos IHP SG13G2 |
> | **Device** | `LV_NMOS` then `LV_PMOS` | run twice, once per device |
>
> The helper returns the best-match **L**, **$I_D/W$**, and **$f_T$**.  
> Unit cell width follows: $W_{unit} = I_q / (I_D/W)\big|_{\text{LUT}}$, then scale signal path by $m = 4$.

## **$G_m/I_d$ Section (Cells Below)**
---

The next group of cells implements the gm/Id design methodology in full.
Here is the sequence and what each part contributes:

| Cells | What they do | Connection to the design |
|:--|:--|:--|
| **Install deps** | `pip install gdown plotly pandas numpy kaleido` | One-time setup; idempotent |
| **Download LUT ZIP** | Downloads `gmid_Data.zip` from Drive → extracts to `/content/gmid_Data/{LV_NMOS,LV_PMOS,HV_NMOS,HV_PMOS}/*.txt` | These are the raw Spectre simulation files that the gm/Id LUT plots are built from |
| **`get_data()` + plots** | Parses the `.txt` files and generates the characteristic curves (gm/Id vs Vgs, Id/W vs gm/Id, etc.) | The engineer reads these plots to choose target gm/id=16, gm/gds=100 the inputs to the helper |
| **Imports** | `matplotlib`, `ipywidgets`, `numpy`, `pandas` | Powers the interactive dashboard |
| **`build_df()`** | Generates synthetic LUT data for the helper widget | See note in the cell below about why this is synthetic |
| **`solve_optimization()`** | σ-normalised nearest-neighbour search over the LUT | The mathematical core of the gm/Id helper |
| **`render_dashboard()`** | Scatter + loss surface plot + Top-5 table | Visual output the engineer uses to confirm the selected operating point |
| **Widget + display** | `ipywidgets` sliders + callbacks | The interactive interface; every slider change reruns `render_dashboard()` live |

---


In [ ]:
# DEPENDENCY INSTALL for gm/ID analysis section
# gdown  → Google Drive downloader; plotly → interactive charts;
# pandas/numpy → data manipulation; kaleido → static plot export for paper.
# Idempotent: already-installed packages are skipped by pip.
#
!pip install gdown plotly pandas numpy kaleido


In [ ]:
# DOWNLOAD gm/ID MODEL DATA — ZIP from Google Drive
# Downloads a ZIP archive of pre-extracted gm/ID characterisation data
# (HV_NMOS, HV_PMOS, LV_NMOS, LV_PMOS) and unpacks to /content/gmid_Data/.
# IDEMPOTENCY: if /content/gmid_Data/ exists and is non-empty, skip download.
# gdown.download(fuzzy=True) handles Drive confirmation pages transparently.
# ZIP is deleted after extraction to free disk space.
# OUTPUT: /content/gmid_Data/{LV_NMOS,LV_PMOS,HV_NMOS,HV_PMOS}/*.txt
#
# Script Purpose:
# This script downloads a ZIP file (containing the gmid_IHP130 model data)
# directly from Google Drive, extracts its contents to a local folder,
# and then deletes the ZIP file to clean up.

# Importing Required Libraries
import gdown        # Used to download files from Google Drive using a file ID or shareable link
import zipfile      # Used for extracting contents from ZIP archive files
import os           # Used for directory and file operations (like making folders, deleting files, etc.)

# Step 1: Define the Google Drive File ID and Output Paths

# This is the unique Google Drive File ID of the ZIP file to download.
# It corresponds to the file’s unique identifier in the Google Drive sharing URL.
gmid_file_id = '1Gd_mxW6A0DBHGzFmbQJ1NT2dvuKrFsCy'

# The name you want to give to the downloaded ZIP file once it’s saved locally.
output_zip = '/content/mydata.zip'  # absolute path — safe regardless of CWD

# Step 2: Download the File from Google Drive
# gdown.download() downloads files from Google Drive using a direct link format.
# The `quiet=False` parameter displays a progress bar during the download.
# The f-string constructs a proper direct download URL from the Google Drive file ID.
# Idempotency: skip download if already extracted
extract_path = '/content/gmid_Data'
if os.path.isdir(extract_path) and os.listdir(extract_path):
    print(f'✅ {extract_path} already exists — skipping download.')
else:
    result = gdown.download(
        f'https://drive.google.com/uc?id={gmid_file_id}',
        output_zip,
        quiet=False,
        fuzzy=True,       # handles redirects and confirmation pages
    )
    if result is None:
        raise RuntimeError(
            'gdown.download() returned None — download failed.\n'
            'Check the file ID or share permissions on Google Drive.'
        )

# Step 3: Prepare the Extraction Folder
# extract_path already defined above

# Create the extraction folder if it doesn’t already exist.
# 'exist_ok=True' prevents errors if the folder already exists.
    os.makedirs(extract_path, exist_ok=True)

# Step 4: Extract the ZIP File
# Open the downloaded ZIP file in 'read' mode.
# The 'with' context ensures the file is automatically closed after extraction.
    with zipfile.ZipFile(output_zip, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

# Step 5: Clean Up Temporary Files

# After extracting, the ZIP file is no longer needed, so it’s deleted to save disk space.
    os.remove(output_zip)

# Step 6: Confirmation Message
# Print a success message showing where the data has been extracted.
    print(f"✅ Data unzipped to: {extract_path}")


---
## 5.1 $G_m/I_D$ Characterisation: IHP SG13G2 130 nm BiCMOS (LV NMOS)
---
Look-up tables (LUTs) are extracted from Spectre simulations across 13 channel lengths
($L = 0.13\,\mu\text{m}$ → $3\,\mu\text{m}$) sweeping $V_{GS}$ at a fixed $W = 2\,\mu\text{m}$.
Each plot below captures a fundamental design trade-off used to bias the IBA core.

- **$g_m/I_D$ vs $V_{GS}$, $V_{ov}$** - identifies the inversion region and sets the DC bias point
- **$g_m/g_{ds}$ vs $g_m/I_D$** - intrinsic gain $A_{v0}$ vs transconductance efficiency trade-off
- **$I_D/W$ vs $g_m/I_D$** - current density curve used to size transistor width
- **$f_T$ vs $g_m/I_D$** - speed–power trade-off ($f_T = g_m / 2\pi C_{gg}$)
- **$C_{gd}/C_{gg}$, $C_{gs}/C_{gg}$ vs $g_m/I_D$** - parasitic capacitance partitioning and Miller coupling

> Each trace corresponds to one channel length. Longer $L$ → higher $A_{v0}$ but lower $f_T$.
> The Ron/Gm methodology uses these LUTs to select the $g_m/I_D$ operating point that
> simultaneously satisfies RC settling (via $R_{on}$) and small-signal bandwidth (via $g_m$).

In [ ]:
# gm/ID DATA LOADING, DERIVED METRICS, AND PLOTTING FUNCTION
#
# get_data(data, key):
#   Parses raw gm/ID .txt files (one per channel length) for one device type.
#   Each line: space-separated floats [Vgs, gm, ID, Vth, gds, Cgg, Cgs, Cgd, ...]
#   Processing: insert device name at index 0; filter to keep first 3 +
#   every even-indexed value (removes duplicate export columns);
#   insert length label at index 1.
#   OUTPUT COLUMNS: Device, Length, Vgs, gm, id_val, Vth, gds, Cgg, Cgs, Cgd
#
# File discovery: dict {device_name: sorted([filenames])} from os.listdir().
# Device selector: sel=2 → LV_NMOS (the IBA design device).
# Script Purpose:
# This module loads raw text-based transistor model data (from GM/ID analysis)
# stored in subfolders under `./gmid_Data/`, converts them into
# structured Pandas DataFrames, and organizes them for later plotting or analysis.

# Import Required Libraries

import plotly.graph_objects as go  # Used later for plotting transistor curves (gm/ID plots, etc.)
import numpy as np                 # For efficient numeric operations
import pandas as pd                # For tabular data organization and processing
import os                          # For directory traversal and file handling

# Step 1: Define Base Directory Containing All Model Data

# Folder where the ZIP file was extracted previously.
# Inside this directory, there are likely multiple subfolders — one for each device type.
# Example folder structure:
#   ./gmid_Data/
#       ├── NMOS/
#       │     ├── L_0p13u.txt
#       │     ├── L_0p25u.txt
#       │     └── ...
#       ├── PMOS/
#       │     ├── L_0p13u.txt
#       │     ├── ...
#       └── etc.

gmid_data = "/content/gmid_Data"  # absolute path — safe regardless of CWD

# Step 2: Function to Parse and Structure Raw Data Files

def get_data(data: list, key: str) -> pd.DataFrame:
    """
    Reads and structures GM/ID transistor model data files for a given device type.

    Parameters
    ----------
    data : list
        List of filenames (e.g., ['L_0p13u.txt', 'L_0p25u.txt', ...]) corresponding to one device type.
    key : str
        The name of the device folder (e.g., 'NMOS', 'PMOS') — used for labeling each row.

    Returns
    -------
    pd.DataFrame
        A combined DataFrame with standardized columns:
        ['Device', 'Length', 'Vgs', 'gm', 'id_val', 'Vth', 'gds', 'Cgg', 'Cgs', 'Cgd']
    """

    # Define the standard set of column names for the structured dataset.
    # Each row corresponds to one bias point for a given transistor geometry.
    columns = ['Device', 'Length', 'Vgs', 'gm', 'id_val', 'Vth', 'gds', 'Cgg', 'Cgs', 'Cgd']

    # Predefined channel length labels corresponding to the input data order.
    # These are human-readable labels (like '0.13u', '0.25u', ...) used in the output DataFrame.
    labels = ['0.13u', '0.25u', '0.5u', '0.75u', '1u', '1.25u', '1.5u', '1.75u',
              '2u', '2.25u', '2.5u', '2.75u', '3u']

    # Initialize an empty list to accumulate parsed and filtered data entries.
    new_data = []

    # Loop Through Each File (Each Represents a Device Length)
    for i, file_path in enumerate(data):
        # Open the data file for reading.
        # Example path: ./gmid_Data/NMOS/L_0p13u.txt
        with open(f'{gmid_data}/{key}/{file_path}', 'r') as file:

            # Iterate through each line in the file.
            # Each line typically contains space-separated numeric values for different electrical parameters.
            for line in file:
                # Split the line into separate strings, convert each to float → numeric list.
                entry = list(map(float, line.strip().split()))

                # Insert the device name (key, e.g., 'NMOS') at the start of each entry.
                entry.insert(0, key)

                # Filter the data:
                # Keep the first 3 values (Device, Vgs, gm/id?) and then every alternate value after that.
                # This pattern removes unnecessary duplicate or redundant entries in some raw files.
                filtered_entry = [val for i, val in enumerate(entry) if i < 3 or i % 2 == 0]

                # Insert the corresponding device length label (e.g., '0.13u', '0.25u') after the 'Device' field.
                filtered_entry.insert(1, labels[i])

                # Append the cleaned row to the accumulated list.
                new_data.append(filtered_entry)

    # Step 3: Convert All Parsed Entries to a Pandas DataFrame

    # The resulting DataFrame will have consistent columns for all devices and lengths.
    return pd.DataFrame(new_data, columns=columns)

# Step 4: Automatically Discover All Data Files in Subdirectories
# The 'files' dictionary maps each device folder to its list of data files.
# Example:
# {
#   'NMOS': ['L_0p13u.txt', 'L_0p25u.txt', ...],
#   'PMOS': ['L_0p13u.txt', 'L_0p25u.txt', ...]
# }

if not os.path.isdir(gmid_data):
    raise FileNotFoundError(
        f"gmid_Data directory not found: {gmid_data}\n"
        "Run Cell 44 (download ZIP) before this cell."
    )
files = {
    directory: sorted(os.listdir(os.path.join(gmid_data, directory)))
    for directory in sorted(os.listdir(gmid_data))
    if os.path.isdir(os.path.join(gmid_data, directory))  # skip stray files
}

# At this stage:
#   - `files` contains the mapping of available datasets.
#   - `get_data()` can be called like this:
#       df_nmos = get_data(files['NMOS'], 'NMOS')
#       df_pmos = get_data(files['PMOS'], 'PMOS')


# List of supported device folders inside ./gmid_Data/
# Each device type corresponds to a different process flavor:
#   HV_NMOS / HV_PMOS → High-Voltage devices (thicker oxide)
#   LV_NMOS / LV_PMOS → Low-Voltage devices (thin oxide, faster)
devices = ['HV_NMOS', 'HV_PMOS', 'LV_NMOS', 'LV_PMOS']

# Select which device to analyze.
# 0 → HV_NMOS
# 1 → HV_PMOS
# 2 → LV_NMOS
# 3 → LV_PMOS
sel = 2  # Currently selected: LV_NMOS

# Step 5: Load the Selected Device Data into a Pandas DataFrame

# Fetch and parse the raw text files for the selected device type.
# The `files` dictionary (from previous snippet) provides the filenames.
# The `get_data()` function reads them, structures them, and returns a clean DataFrame.

df = get_data(files[devices[sel]], devices[sel])

# At this point, df contains columns:
# ['Device', 'Length', 'Vgs', 'gm', 'id_val', 'Vth', 'gds', 'Cgg', 'Cgs', 'Cgd']

# Step 6: Compute Derived Electrical Quantities

# 1️. Overdrive voltage (Vov = Vgs - Vth)
# Represents how strongly the transistor is turned on beyond threshold.
df['Vov'] = df['Vgs'] - df['Vth']

# 2️. Transconductance efficiency (gm/ID)
# Key analog design figure of merit — measures how much gm is achieved per unit bias current.
# High gm/ID → higher gain and better power efficiency.
df['gm/id'] = df['gm'] / df['id_val']

# 3️. Intrinsic voltage gain metric (gm/gds)
# Indicates transistor’s inherent amplification capability (higher is better).
# gm/gds ≈ intrinsic gain (A_v0) in linear region.
df['gm/gds'] = df['gm'] / df['gds']

# 4️. Device width (W)
# Set to a constant value (2 µm) for normalization of current density and capacitance.
# Units: meters
df['W'] = 2e-6

# 5️. Current density normalization (ID/W)
# Used to compare devices independent of width.
# Expresses drain current per unit transistor width.
df['id/W'] = df['id_val'] / df['W']

# 6️. Transit frequency (fT)
# fT = gm / (2πCgg)
# This is the frequency where current gain drops to unity — indicator of transistor speed.
df['ft'] = df['gm'] / (2 * np.pi * df['Cgg'])

# 7️. Capacitance ratio: Cgd/Cgg
# Fraction of total gate capacitance that is gate-drain overlap (affects Miller effect).
df['Cgd/Cgg'] = df['Cgd'] / df['Cgg']

# 8️. Capacitance ratio: Cgs/Cgg
# Fraction of total gate capacitance that is gate-source (controls input response).
df['Cgs/Cgg'] = df['Cgs'] / df['Cgg']

# After these computations:
# The DataFrame `df` now contains both raw and derived metrics, e.g.:
# ['Device', 'Length', 'Vgs', 'gm', 'id_val', 'Vth', 'gds',
#  'Cgg', 'Cgs', 'Cgd', 'Vov', 'gm/id', 'gm/gds', 'W', 'id/W',
#  'ft', 'Cgd/Cgg', 'Cgs/Cgg']
#
# You can now group, plot, or fit relationships like:
#   - gm/ID vs Vov
#   - ft vs gm/ID
#   - gm/gds vs Vov
# etc.

# Convert 'Length' column to string to ensure consistent comparison/filtering.
# This prevents issues when mixing numeric and string types (e.g., '0.5u' vs 0.5).
df['Length'] = df['Length'].astype(str)

# Step 7: Identify All Available Channel Length Labels

# Extract all unique device lengths from the DataFrame (e.g., ['0.13u', '0.25u', '0.5u', ...]).
length_labels = df['Length'].unique()

# For high-voltage devices (sel = 0 or 1 → HV_NMOS / HV_PMOS),
# exclude the smallest lengths that may not physically exist in HV technology.
# In other words:
# - HV devices typically have longer minimum L (≥ 0.5 µm)
# - LV devices (sel ≥ 2) can include shorter ones like 0.13 µm, 0.25 µm.
if sel < 2:
    length_labels = np.setdiff1d(df['Length'].unique(), ['0.13u', '0.25u'])

# Step 8: Define a Reusable Plotly Plotting Function
def plot_interactive(df, x_col, y_col, x_label, y_label, title, log_y=False):
    """
    Create an interactive Plotly line plot showing a transistor
    characteristic (e.g., gm/ID vs Vov) for all device lengths.

    Parameters
    ----------
    df : pd.DataFrame
        The main data table containing transistor measurements and derived metrics.
    x_col : str
        Column name for the X-axis variable (e.g., 'Vov', 'gm/id').
    y_col : str
        Column name for the Y-axis variable (e.g., 'gm/id', 'ft').
    x_label : str
        Label displayed on the X-axis.
    y_label : str
        Label displayed on the Y-axis.
    title : str
        Title of the plot.
    log_y : bool, optional
        Whether to use a logarithmic Y-axis scale (default: False).
        Useful when plotting values like ft (which span multiple decades).
    """

    # Initialize an empty Plotly Figure
    fig = go.Figure()

    # Predefined list of colors to visually distinguish curves for each channel length.
    # If there are more lengths than colors, the list cycles automatically using modulus.
    colors = [
        'blue', 'red', 'green', 'purple', 'orange', 'brown', 'pink',
        'gray', 'olive', 'cyan', 'magenta', 'gold', 'lime'
    ]

    # Loop Over Each Unique Device Length and Add Its Data Trace
    for i, length in enumerate(length_labels):

        # Filter the DataFrame for the current channel length.
        length_df = df[df['Length'] == length]

        # Handle missing or empty data cases gracefully.
        if length_df.empty:
            print(f" Warning: No data found for Length = {length}")
            continue

        # Add one line trace per length.
        # Each trace represents how the chosen Y-parameter varies with X
        # for a fixed transistor length.
        fig.add_trace(go.Scatter(
            x=length_df[x_col],             # Data for X-axis (e.g., Vov)
            y=length_df[y_col],             # Data for Y-axis (e.g., gm/id)
            mode='lines',                   # Plot as smooth continuous lines
            name=f"L={length}",             # Legend label for clarity
            line=dict(
                color=colors[i % len(colors)],  # Cycle through color list
                width=2                         # Line thickness for visibility
            )
        ))

    # Customize Graph Layout and Axis Settings

    # Configure the figure layout for clarity and scientific look
    fig.update_layout(
        title=title,                     # Dynamic plot title (passed when function is called)
        xaxis_title=x_label,             # Label text for X-axis
        yaxis_title=y_label,             # Label text for Y-axis
        legend_title="Transistor Length",# Legend title describing what each color/trace represents
        hovermode="x unified",           # Align hover tooltip values vertically for easy comparison
        margin=dict(l=50, r=50, t=50, b=50),  # Set margins (left, right, top, bottom) to prevent label cutoffs
        width=950,                       # Define fixed figure width for consistent display
        height=650                       # Define fixed figure height for clear, readable plots
    )

    #  Apply logarithmic Y-axis scale if requested
    # Some transistor metrics (e.g., ft or current density) vary over
    # several orders of magnitude. Log scaling improves visualization
    # of such wide dynamic ranges.
    if log_y:
        fig.update_yaxes(type="log")

    #  Use scientific (SI) number formatting for better readability
    # tickformat=".2s" → Compactly displays large/small numbers using
    # SI prefixes like K (kilo), M (mega), G (giga).
    # Example: 1250000 → 1.25M
    fig.update_yaxes(tickformat=".2s")  # ".2s" formats values like 1.2M, 3.4G

    #  Automatically scale axes and enable grid lines
    # - autorange=True : Automatically fits axis limits to data
    # - showgrid=True  : Displays reference gridlines to improve readability
    # - zeroline=False : Hides the dark zero axis line for a cleaner look
    fig.update_xaxes(autorange=True, showgrid=True, zeroline=False)
    fig.update_yaxes(autorange=True, showgrid=True, zeroline=False)

    #  Display the final interactive Plotly figure
    # This opens the interactive chart directly in the notebook or
    # browser window, allowing zooming, panning, and legend toggling.
    fig.show()


# Generate standard gm/ID characterization plots
# Each call below produces an important transistor design curve.
# These relationships are critical for analog/RF design benchmarking.

# 1️. gm/ID vs Vgs — Efficiency of transconductance as a function of gate bias.
plot_interactive(df, 'Vgs', 'gm/id', 'Vgs', 'gm/id',
                 f'{df["Device"].unique()[0]} gm/id versus Vgs')

# 2️. gm/ID vs Vov — gm/ID expressed against overdrive voltage for bias region comparison.
plot_interactive(df, 'Vov', 'gm/id', 'Vov', 'gm/id',
                 f'{df["Device"].unique()[0]} gm/id versus Vov')

# 3️. gm/gds vs gm/ID — Intrinsic gain plotted versus transconductance efficiency.
#     Helps analyze trade-off between gain and speed.
plot_interactive(df, 'gm/id', 'gm/gds', 'gm/id', 'gm/gds',
                 f'{df["Device"].unique()[0]} gm/gds versus gm/id')

# 4️. id/W vs gm/ID — Current density per unit width versus gm/ID.
#     Useful for evaluating power efficiency.
plot_interactive(df, 'gm/id', 'id/W', 'gm/id', 'id/W',
                 f'{df["Device"].unique()[0]} id/W versus gm/id')  # Log scale for better visibility

# 5️. ft vs gm/ID — Transit frequency versus gm/ID, a speed–efficiency trade-off metric.
plot_interactive(df, 'gm/id', 'ft', 'gm/id', 'ft (Hz)',
                 f'{df["Device"].unique()[0]} ft versus gm/id')  # Log scale for frequency

# 6️. Cgd/Cgg vs gm/ID — Shows the ratio of gate-drain capacitance to total gate capacitance.
#     Indicates feedback coupling and frequency performance trends.
plot_interactive(df, 'gm/id', 'Cgd/Cgg', 'gm/id', 'Cgd / Cgg',
                 f'{df["Device"].unique()[0]} Cgd / Cgg versus gm/id')

# 7️. Cgs/Cgg vs gm/ID — Fraction of gate-source capacitance, useful for analyzing input coupling.
plot_interactive(df, 'gm/id', 'Cgs/Cgg', 'gm/id', 'Cgs / Cgg',
                 f'{df["Device"].unique()[0]} Cgs / Cgg versus gm/id')


---
## 5.2 $G_m/I_D$ Design Helper
---
Given a target transconductance efficiency $g_m/I_D$ and intrinsic gain $g_m/g_{ds}$,
the helper performs a weighted, $\sigma$-normalised $L_2^2$ nearest-neighbour search
over the LUT:

$$\mathcal{L} = w_1\!\left(\frac{g_m/I_D - \hat{x}_A}{\sigma_A}\right)^{\!2}
              + w_2\!\left(\frac{g_m/g_{ds} - \hat{x}_B}{\sigma_B}\right)^{\!2}$$

where $\sigma_A$, $\sigma_B$ are the dataset standard deviations, making $w_1$, $w_2$
dimensionless and the two axes commensurable. A minimum gain constraint
$\hat{x}_B = \max(g_{m}/g_{ds}\big|_{\text{target}},\, B_{\min})$ is enforced before search.
The best match returns channel length $L$ and current density $I_D/W$,
from which transistor width follows directly:

$$W = \frac{I_D}{(I_D/W)\big|_{\text{LUT}}}$$

In [ ]:
#   IMPORTS for interactive dashboard widgets
#   matplotlib.pyplot → dark-theme dashboard plots
#   ipywidgets        → sliders, dropdowns, multi-select, output widgets
#   interact          → convenience decorator for auto-wired callbacks
#   IPython.display   → display() + clear_output() for live dashboard refresh
#
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact
from typing import Dict, Any
from IPython.display import display, clear_output

### **About the Gm/Id Design Helper Dataset**

The helper runs on **real IHP SG13G2 SPICE characterisation data** loaded by `get_data()` above.
Every point in the scatter plot and every row in the Top-5 table comes directly from the
pre-simulated `.txt` LUT files, the same files used for the gm/id vs Vov characteristic curves.

The two device types available are `LV_NMOS`, `LV_PMOS`.
Each device is characterised across 13 channel lengths (0.13 µm – 3 µm for LV; 0.5 µm – 3 µm for HV),
with a Vgs sweep at fixed W = 2 µm.

In [ ]:
# BUILD DF_ALL FROM REAL SPICE DATA
# Replaces the old synthetic build_df() entirely.
# Uses get_data() (Cell 81) for all device folders found in `files`,
# computes gm/id, gm/gds, id/W (A/m), ft (GHz), then filters out
# non-physical rows caused by near-zero gds or id_val at extreme bias.
#
# DF_ALL columns: Device, Length, gm/id, gm/gds, id/W, ft

def _build_df_all(files_dict: dict) -> 'pd.DataFrame':
    """
    Build the design-helper LUT from real SPICE characterisation data.

    Parameters
    ----------
    files_dict : dict
        Mapping {device_folder_name: [sorted list of .txt filenames]}
        as produced by the os.listdir() block in Cell 81.

    Returns
    -------
    pd.DataFrame
        Columns: Device, Length, gm/id, gm/gds, id/W, ft
    """
    W_CHAR = 2e-6          # characterisation width used in SPICE sweep (m)
    frames = []

    for dev_name, file_list in files_dict.items():
        df_dev = get_data(file_list, dev_name)          # raw columns from Cell 81

        # Derived metrics — identical to Cell 81 post-processing
        df_dev['gm/id']  = df_dev['gm']      / df_dev['id_val']
        df_dev['gm/gds'] = df_dev['gm']      / df_dev['gds']
        df_dev['id/W']   = df_dev['id_val']  / W_CHAR          # A/m  (= µA/µm × 1e-6)
        df_dev['ft']     = df_dev['gm'] / (2 * np.pi * df_dev['Cgg']) / 1e9  # GHz

        # Drop non-physical rows: near-zero id or gds causes exploding ratios
        mask = (
            df_dev['gm/id'].between(0.5, 40)    &
            df_dev['gm/gds'].between(0.5, 600)  &
            df_dev['id_val'].gt(1e-12)           &
            df_dev['gds'].gt(1e-12)
        )
        frames.append(
            df_dev.loc[mask, ['Device', 'Length', 'gm/id', 'gm/gds', 'id/W', 'ft']]
        )

    return pd.concat(frames, ignore_index=True)


DF_ALL = _build_df_all(files)

# Diagnostic summary
print(f"DF_ALL: {len(DF_ALL)} rows")
for dev in DF_ALL['Device'].unique():
    sub = DF_ALL[DF_ALL['Device'] == dev]
    print(f"  {dev:10s}  gm/id=[{sub['gm/id'].min():.1f}, {sub['gm/id'].max():.1f}]"
          f"  gm/gds=[{sub['gm/gds'].min():.1f}, {sub['gm/gds'].max():.1f}]"
          f"  lengths={sub['Length'].nunique()}")


### `solve_optimization()` (The Optimizer)

The cell below defines `solve_optimization()` the function that searches the LUT for the operating point closest to your targets:

$$\mathcal{L} = w_1 \left(\frac{g_m/I_D - \hat{x}_A}{\sigma_A}\right)^2 + w_2 \left(\frac{g_m/g_{ds} - \hat{x}_B}{\sigma_B}\right)^2$$

**Why σ-normalise?** $g_m/I_D$ spans roughly $2$–$28\,\text{V}^{-1}$ while $g_m/g_{ds}$ spans $10$–$400$. Without normalisation, $g_m/g_{ds}$ dominates the distance simply because its numbers are larger not because it matters more. Dividing by $\sigma$ puts both terms on the same dimensionless scale, so $w_1$ and $w_2$ are true importance weights.

**The bound constraint:** $\hat{x}_B = \max(B_{\min},\, g_m/g_{ds}\big|_{\text{target}})$ clamps the target upward to the minimum gain bound so the optimiser never recommends a device with gain below your floor.

**Returns:** the top-5 LUT rows sorted by $\mathcal{L}$, the best row, a simulated 6-step convergence trace for display, and $\sigma_A$, $\sigma_B$ (passed to `render_dashboard()` for the loss surface plot).

In [ ]:
# OPTIMISER — solve_optimization(df, tA, tB, w1, w2, bB)
#
# Step 1 — scipy.optimize.minimize finds the optimal (gm/id, gm/gds) in
#   continuous space subject to a hard lower bound on gm/gds (bB).
#   The objective is a σ-normalised weighted L2² loss so w1, w2 are
#   true dimensionless importance weights independent of parameter scale.
#
# Step 2 — nearest-neighbour search over the LUT using the same
#   normalised distance to find the best real (L, bias) operating point.
#
# NOTE: for a simple quadratic objective the scipy call returns the
#   closed-form solution (tA, max(bB, tB)) — it is kept here for
#   explicitness and because it correctly handles the hard gm/gds bound.
#
# RETURNS: dict with optA, optB, top5, best_row, logs, gmid_std, gmgds_std.

from scipy.optimize import minimize

def solve_optimization(df, tA, tB, w1, w2, bB):
    """
    Parameters
    ----------
    tA : float  target gm/id
    tB : float  target gm/gds
    w1 : float  importance weight for gm/id  (dimensionless after σ-norm)
    w2 : float  importance weight for gm/gds (dimensionless after σ-norm)
    bB : float  hard lower bound on gm/gds — optimizer never goes below this
    """
    # σ-normalisation constants — computed from the filtered LUT subset
    # so the range of the actual data (not synthetic values) sets the scale.
    gmid_std  = df['gm/id'].std()  or 1.0
    gmgds_std = df['gm/gds'].std() or 1.0

    # ── Step 1: continuous optimisation ─────────────────────────────────────
    def objective(x):
        a, b = x
        return (w1 * ((a - tA) / gmid_std) **2
              + w2 * ((b - tB) / gmgds_std)**2)

    x0 = np.array([df['gm/id'].mean(), df['gm/gds'].mean()])

    # Hard bound: gm/gds >= max(bB, tB)  — never recommend below the floor
    gm_gds_floor = max(bB, tB)
    bounds = [(None, None), (gm_gds_floor, None)]

    result   = minimize(objective, x0, method='L-BFGS-B', bounds=bounds)
    optA, optB = result.x

    # ── Step 2: nearest-neighbour search in LUT ──────────────────────────────
    df_calc = df.copy()
    df_calc['distance'] = (
        w1 * ((df_calc['gm/id']  - optA) / gmid_std) **2
      + w2 * ((df_calc['gm/gds'] - optB) / gmgds_std)**2
    )
    top5 = df_calc.sort_values('distance').head(5).copy()

    # ── Convergence trace (illustrative — 6 gradient steps from mean) ────────
    logs = []
    a_log, b_log = df['gm/id'].mean(), max(bB, df['gm/gds'].mean())
    for i in range(6):
        a_log += (optA - a_log) * 0.55
        b_log += (optB - b_log) * 0.55
        lv = (w1 * ((a_log - tA) / gmid_std) **2
            + w2 * ((b_log - tB) / gmgds_std)**2)
        logs.append(f"i{i+1}: {lv:.2e}")

    return {
        'optA': optA, 'optB': optB,
        'top5': top5, 'best_row': top5.iloc[0],
        'logs': logs,
        'gmid_std': gmid_std, 'gmgds_std': gmgds_std,
    }


### `render_dashboard()` (Visualisation Engine)

The cell below defines `render_dashboard()`, the function that runs every time a slider changes.
It does three things:

1. **Filters the LUT** - calls `solve_optimization()` with the current slider values; receives the top-5 matches and the σ-normalised distances.

2. **Draws two panels:**
   - *Left - Scatter plot*: every LUT row as a dot (plasma colourmap by L); the amber ▲ is your target, the green ★ is the best match, the red dashed line is your `gm/gds` lower bound.
   - *Right - Loss surface*: plots the objective function $\mathcal{L}$ vs gm/id (gm/gds held at the optimum). The narrow minimum (green dot) is where the solver landed. A steep, deep minimum means the design point is well-determined; a flat curve means any value in that range is acceptable.

3. **Prints results** - the convergence trace (6 iterative loss values), the optimised metrics (best L, $f_T$, $I_D/W$, device), and the Top-5 colour-coded table (green = best match, red = worst of the five).


In [ ]:
#  VISUALISATION — render_dashboard()  (Gm/Id Design Helper)
#
# Produces a 2-panel matplotlib figure + Top-5 styled table on widget change.
#
# PANEL 1 — SCATTER (gm/gds vs gm/id):
#   Plasma colourmap per length; markers: ▲ amber (target), × green (optimum),
#   ★ green star (best row), red dashed line (gm/gds bound).
#
# PANEL 2 — LOSS SURFACE:
#   std-normalised L2² loss vs gm/id (gm/gds fixed at optB).
#   Log Y-axis; narrow minimum = well-conditioned design problem.
#
# PRINTED OUTPUTS:
#   - Convergence trace (6 loss values).
#   - Optimized metrics (targets vs optima, best device/length/ft/id_W).
#   - Top-5 Styled DataFrame: heat-map distance column
#     (green=closest, amber=mid, red=furthest).
#
# VISUALIZATION (Desktop/Colab Dashboard)
def render_dashboard(devices, lengths, tA, tB, w1, w2, bB):
    # 1. Filter
    df = DF_ALL[(DF_ALL['Device'].isin(devices)) & (DF_ALL['Length'].isin(lengths))].copy()
    if df.empty:
        print("Please selection at least one Device and Length.")
        return

    # 2. Solve
    res = solve_optimization(df, tA, tB, w1, w2, bB)
    best = res['best_row']


    # Warn if target gm/gds is above the max achievable in this dataset
    max_achievable_gmgds = df['gm/gds'].max()
    if tB > max_achievable_gmgds:
        print(f"\033[93m⚠  Target gm/gds={tB:.0f} exceeds the maximum in the "
              f"selected data ({max_achievable_gmgds:.1f}). "
              f"Showing best achievable point instead.\033[0m")

    # 3. Setup Layout
    plt.style.use('dark_background')
    plt.rcParams.update({'xtick.labelsize': 11, 'ytick.labelsize': 11})
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    fig.patch.set_facecolor('#080b12')

    accent_g = '#00ffb2' # Neon Green
    accent_b = '#3b82f6' # Royal Blue
    accent_y = '#f59e0b' # Amber/Orange
    accent_r = '#ef4444' # Red

    # PLOT 1: Scatter gm/id vs gm/gds
    ax_sc = axes[0]
    # plasma colormap per length — exactly matches Cell 58 (Ron/Gm dashboard)
    lengths_sorted = sorted(df['Length'].unique())
    cmap = plt.get_cmap('plasma')
    for idx_l, L in enumerate(lengths_sorted):
        mask = df['Length'] == L
        if mask.any():
            ax_sc.scatter(df.loc[mask, 'gm/id'], df.loc[mask, 'gm/gds'],
                          color=cmap(idx_l / max(1, len(lengths_sorted) - 1)), alpha=0.30, s=25,
                          label=f"L={L}", zorder=2)

    # Target (Triangle as in JS)
    ax_sc.scatter(tA, tB, color=accent_y, marker='^', s=100, label='Target', zorder=10)
    # Optimum (Cross as in JS)
    ax_sc.scatter(res['optA'], res['optB'], color=accent_g, marker='x', s=120, label='Optimum (Clamped)', zorder=11, linewidths=2)
    # Best Match (Star as in JS)
    ax_sc.scatter(best['gm/id'], best['gm/gds'], color=accent_g, marker='*', s=180, label='Best Row', edgecolors='white', zorder=12)
    # Bound Line (Dashed red as in JS)
    ax_sc.axhline(bB, color=accent_r, linestyle='--', alpha=0.6, label='Bound')

    ax_sc.set_title("Scatter · gm/id vs gm/gds", color='#94a3b8', pad=15, fontsize=15, fontweight="bold")
    ax_sc.set_xlabel("gm/id", alpha=0.7, fontsize=15)
    ax_sc.set_ylabel("gm/gds", alpha=0.7, fontsize=15)
    ax_sc.set_facecolor('#0e1320')
    ax_sc.grid(True, color='#1e2535', alpha=0.4)
    ax_sc.legend(fontsize=11, ncols=2, frameon=True, facecolor='#0e1320', edgecolor='#1e2535', loc=1)

    # PLOT 2: Loss Surface
    ax_ls = axes[1]
    sweep_x = np.linspace(max(0.5, tA - 15), tA + 15, 200)

    _gs  = res['gmid_std'];  _ggs = res['gmgds_std']     # ← FIX 2: unpack stds
    losses = (                                             # ← FIX 1 + FIX 2
        w1 * ((sweep_x     - tA) / _gs )**2              # gm/id  sweep — std-normalized
      + w2 * ((res['optB'] - tB) / _ggs)**2              # gm/gds const — std-normalized
    )
    min_loss = (                                           # ← FIX 1 + FIX 2: dot at minimum
        w1 * ((res['optA'] - tA) / _gs )**2
      + w2 * ((res['optB'] - tB) / _ggs)**2
    )

    ax_ls.plot(sweep_x, losses, color=accent_y, linewidth=2, label='Loss Surface')
    ax_ls.fill_between(sweep_x, losses, alpha=0.1, color=accent_y)
    ax_ls.axvline(tA, color=accent_r, linestyle='--', alpha=0.4)
    ax_ls.scatter(res['optA'], min_loss, color=accent_g, s=100, zorder=10)

    ax_ls.set_title(f"Loss Surface (sweep gm/id @ gm/gds={res['optB']:.1f})", color='#94a3b8', pad=15, fontsize=15, fontweight="bold")
    ax_ls.set_xlabel("gm/id", alpha=0.7, fontsize=15)
    ax_ls.set_ylabel("Loss", alpha=0.7, fontsize=15)
    ax_ls.set_facecolor('#0e1320')
    ax_ls.set_yscale('log')
    ax_ls.grid(True, color='#1e2535', alpha=0.4)

    plt.tight_layout()
    plt.show()

    # OUTPUT METRICS
    print("\n" + "── CONVERGENCE TRACE ───────────────────────────────────────────────")
    print(" | ".join(res['logs']))

    B = '\033[1m'; E = '\033[0m'; G = '\033[92m'; Y = '\033[93m'
    print("\n" + Y + "══ OPTIMIZED METRICS " + "═"*47 + E)
    print(f" Target gm/id:   {tA:7.2f}  |  Optimized gm/id:  {res['optA']:7.2f}")
    print(f" Target gm/gds:  {tB:7.2f}  |  Optimized gm/gds: {res['optB']:7.2f}")
    print(G + B + f" ★ Best Match L:   {best['Length']:>7}  |  Best Match ft:  {best['ft']:7.2f} GHz" + E)
    print(G + B + f" ★ id/W (µA/µm):  {best['id/W']:7.4f}  |  Device:         {best['Device']:>7}" + E)

    print(Y + "═"*68 + E)

    print("\n" + "── TOP 5 MATCHES ───────────────────────────────────────────────────")
    tbl = res['top5'][['Device', 'Length', 'gm/id', 'gm/gds', 'id/W', 'ft', 'distance']].copy()
    # Scale id/W from A/m → µA/µm for display
    tbl['id/W (µA/µm)'] = tbl['id/W']
    tbl = tbl.drop(columns=['id/W'])

    # Normalise dist to [0,1] for colour mapping — matches Cell 58 (Ron/Gm dashboard)
    d_min, d_max = tbl['distance'].min(), tbl['distance'].max()
    d_range = d_max - d_min if d_max != d_min else 1

    def dist_color(val):
        t = (val - d_min) / d_range
        if   t < 0.33: bg = '#1a7a3a'   # green  — best match
        elif t < 0.66: bg = '#a07800'   # amber  — mid
        else:          bg = '#8b1a1a'   # red    — worst
        return f'background-color: {bg}; color: white; font-weight: bold; text-align: center'

    styled = (
        tbl.style
           .format({'distance': '{:.2e}', 'id/W (µA/µm)': '{:.4f}',
                    'gm/id': '{:.2f}', 'gm/gds': '{:.2f}', 'ft': '{:.2f}'})
           .map(dist_color, subset=['distance'])
    )
    display(styled)

### How to Use the Gm/Id Design Helper

This interactive dashboard helps you find the optimal transistor size based on **($g_m/I_D$)** based design methodology.

**Step-by-step:**
1. **Select Devices** - choose `LV_NMOS`, `LV_PMOS` using the multi-select box.
2. **Select Lengths** - pick the channel lengths or all the lengths you want to explore (selct the Checkbox to select multiple lengths).
3. **Set Target gm/id** - drag the slider to your desired $g_m/I_D$ value (typical range: 5–25 V⁻¹). Higher $g_m/I_D$ means more efficien (subthreshold region)and lower $g_m/I_D$ means faster but less efficient (strong inversion).
4. **Set Target gm/gds** - this controls intrinsic gain. Higher values give more gain but trade off with speed.
5. **Set Bound** - minimum acceptable $g_m/g_{ds}$ constraint. The optimizer won't accept solutions below this.

**Understanding the Plots:**

- **Scatter (gm/gds vs gm/id):** Each dot is one simulation point, coloured by channel length (plasma colourmap: purple = short L, yellow = long L). The **▲ amber triangle** is your target. The **★ green star** is the best match found. The **red dashed line** is the Bound constraint.
- **Loss Surface:** Shows how the optimizer's cost function varies as gm/id sweeps across its range (gm/gds held at optimum). The minimum of this curve (green dot) is the optimizer's solution. Log-scale Y-axis a steep, narrow minimum means a well-conditioned problem.

**Colour Coding in Top-5 Table:**
- 🟢 **Green** - best match (lowest distance to your query)
- 🟡 **Amber** - moderate match
- 🔴 **Red** - worst of the top 5

**Key Output - OPTIMIZED METRICS:**  
The highlighted green lines show the most important results: the **best matching channel length**, **ft (transition frequency)**, **id/W (current density in µA/µm)**, and **device type**. Use these values to size your transistors.


In [ ]:
# WIDGET LAYOUT & CALLBACKS — Gm/Id Design Helper
# Slider ranges are derived from the real DF_ALL data so they always
# reflect what IHP SG13G2 devices can actually achieve.
# Header text corrected to 'Real SPICE LUT data'.

import ipywidgets as widgets
from IPython.display import clear_output
import matplotlib.pyplot as plt

# Derive slider bounds from real data — no hardcoded magic numbers
_gmgds_max = float(DF_ALL['gm/gds'].max())
_gmid_max  = float(DF_ALL['gm/id'].max())
_lengths_all = sorted(DF_ALL['Length'].unique().tolist(),
                       key=lambda x: float(x.replace('u','')))
_devices_all  = sorted(DF_ALL['Device'].unique().tolist())

header = widgets.HTML(
    "<h2 style='color:#0f172a; font-family:monospace; margin:0'> Gm/Id Design Helper </h2>"
    "<p style='color:#2563eb; margin:4px 0 10px 0; font-size:13px'>"
    "Real SPICE LUT data (IHP SG13G2) · dark-theme plots · live sliders</p>"
)

# WIDGETS
w_devices = widgets.SelectMultiple(
    options=['LV_NMOS', 'LV_PMOS'],
    value=['LV_NMOS'],
    description='Devices',
    layout=widgets.Layout(height='130px')
)
w_lengths = widgets.SelectMultiple(
    options=_lengths_all,
    value=_lengths_all,
    description='Lengths',
    layout=widgets.Layout(height='130px')
)
w_all_lengths = widgets.Checkbox(
    value=True,
    description='All',
    indent=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='60px', height='130px')
)
w_tA = widgets.FloatSlider(
    value=10.0, min=1.0, max=round(_gmid_max, 0), step=0.1,
    description='Target gm/id',
    layout=widgets.Layout(width='500px')
)
w_tB = widgets.FloatSlider(
    value=50.0, min=2.0, max=round(_gmgds_max, 0), step=1.0,
    description='Target gm/gds',
    layout=widgets.Layout(width='500px'),
    style={'description_width': '90px'}
)
w_bB = widgets.FloatSlider(
    value=55.0, min=2.0, max=round(_gmgds_max, 0), step=1.0,
    description='Bound (min gm/gds)',
    layout=widgets.Layout(width='500px'),
    style={'description_width': '120px'}
)
# w1/w2 kept for power users but not exposed in UI
w_w1 = widgets.FloatSlider(value=10.0, min=0.1, max=50.0,  step=0.1, description='w1')
w_w2 = widgets.FloatSlider(value=20.0, min=1.0, max=200.0, step=1.0, description='w2')

out_gmid = widgets.Output()

def _toggle_all_lengths(change):
    if change['new']:
        w_lengths.value = tuple(w_lengths.options)
    else:
        w_lengths.value = ()

w_all_lengths.observe(_toggle_all_lengths, names='value')

def _trigger_gmid(*args):
    with out_gmid:
        clear_output(wait=True)
        render_dashboard(
            devices=w_devices.value,
            lengths=w_lengths.value,
            tA=w_tA.value, tB=w_tB.value,
            w1=w_w1.value, w2=w_w2.value,
            bB=w_bB.value
        )

for _w in (w_devices, w_lengths, w_tA, w_tB, w_w1, w_w2, w_bB):
    _w.observe(_trigger_gmid, names='value')

controls_top_gmid = widgets.HBox(
    [w_devices, w_lengths, w_all_lengths],
    layout=widgets.Layout(gap='16px', margin='4px 0', align_items='flex-start')
)
controls_bot_gmid = widgets.VBox(
    [w_tA, w_tB, w_bB],
    layout=widgets.Layout(margin='4px 0')
)

display(header, controls_top_gmid, controls_bot_gmid, out_gmid)
_trigger_gmid()


### Gm/Id Helper Output → Transistor Sizing

The helper returned the following best-match operating points:

| Device | L | $I_D/W$ (µA/µm) | $I_D$ (µA) | $W_{unit}$ (µm) | $f_T$ (GHz) |
|:--|:--|:--|:--|:--|:--|
| **LV_NMOS** | 3.0 µm | 1.8725 µA/µm | 0.5 µA | 0.3 µm | ~0.14 |
| **LV_PMOS** | 0.25 µm | 5.000 µA/µm | 0.5 µA | 0.15 µm | — |

**Width calculation** with unit quiescent current $I_q = 0.5\,\mu\text{A}$:

$$W_{\text{NMOS,unit}} = \frac{0.5\,\mu\text{A}}{1.8725\,\mu\text{A}/\mu\text{m}} \approx 0.3\,\mu\text{m}$$

$$W_{\text{PMOS,unit}} = \frac{0.5\,\mu\text{A}}{5.000\,\mu\text{A}/\mu\text{m}} = 0.1\,\mu\text{m} \approx 0.15\,\mu\text{m}$$

Multiplier $m = 4$ is applied to the signal path to scale $G_m$ to the required 20 µA/V per device type (40 µA/V total):

| Path | Device | $m$ | $W$ (µm) | $L$ (µm) | $I_D$ (µA) |
|:--|:--|:--|:--|:--|:--|
| **Signal** | LV_NMOS | 4 | 0.3 | 3.0 | **2.0** |
| **Signal** | LV_PMOS | 4 | 0.15 | 0.25 | **2.0** |
| **Replica** | LV_NMOS | 1 | 0.3 | 3.0 | **0.5** |
| **Replica** | LV_PMOS | 1 | 0.15 | 0.25 | **0.5** |

The replica path burns $0.5\,\mu\text{A}$ (single unit cell) while the main signal path burns $2\,\mu\text{A}$ (4× unit cell), giving a total quiescent current of $2.5\,\mu\text{A}$ for whole circuit.

---

> ⚠️ **Critical limitation of the gm/Id approach at this stage:**
> The sizing satisfies the **small-signal bandwidth** target ($G_m \approx 40\,\mu\text{S}$), but gives **no information** about:
> - $R_{on}$ of the output transistors and any information about the Non-Linear settling information(sets non-linear RC settling speed)
> - Deadzone bias voltages $V_{DZP}$ / $V_{DZN}$ (the small-signal settling window)
> - Whether the RC phase even completes within the clock half-cycle
> - Worst-case SS-corner behaviour for any of the above
>
> These unknowns require additional SPICE iterations which is the gap the $R_{on}/g_m$ methodology closes.

---
# Inverter Based Amplifier designed using gm/Id based methodology

Schematic of the testbench for IBA in capacitive feedback in Xschem:
<div align="center">
    <img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/ckt_images/fig_8_IBA_with_cap_FB_using_gmid.jpg" width="1100">
    <br>
    <strong>Figure 10:</strong> Inverter-based amplifier in capacitive feedback configuration designed using the $g_{m}/I_d$ methodology..
</div>

<br>

---


## 5.3 Results from the IBA designed using **$g_m/I_D$** design methodology

The four figures below are generated from `INV_OTA_gmid.txt`, a transient simulation of the IBA (Inverter-Based Amplifier) sized using the $g_m/I_D$ methodology.

(LV_NMOS: $W = 0.3\,\mu\text{m}$, $L = 3.0\,\mu\text{m}$, $m = 4$; LV_PMOS: $W = 0.15\,\mu\text{m}$, $L = 0.25\,\mu\text{m}$, $m = 4$; $I_{q,replica} = 0.5\,\mu\text{A}$, $I_{q,signal} = 2.0\,\mu\text{A}$)

**What each figure shows and what it exposes about the gm/Id limitation:**

| Figure | Signal plotted | What the code computes | What gm/Id cannot predict pre-simulation |
|:--|:--|:--|:--|
| **Fig 11** : $V_{out}$ | `sv_g` = Vout(t) from file | Direct read from transient file | Whether RC phase completes in time |
| **Fig 12** : Log error | `err_g = (Vout − Vfinal)/Vfinal` | Relative error on log scale; `curve_fit` extracts τ | τ of the RC phase (driven by $R_{on}$) - unknown to gm/Id |
| **Fig 13** : $I_{out}$ | `sm_g` = branch current × 1e6 | Raw current waveform in µA | $I_{peak}$ magnitude — set by $R_{on}$, invisible to gm/Id |
| **Fig 14** : $dV_{out}/dt$ | `dvdt_g = np.gradient(sv_g, st_us_g)` | Numerical derivative = slew rate | Slew rate peak = $I_{peak}/C_L$ — depends on $R_{on}$ |

> The gm/Id design settles correctly, but the designer had no pre-simulation visibility into
> $R_{on}$, the deadzone bias ranges, or the SS-corner worst case
> all of which are characterised pre-simulation in the $R_{on}/g_m$ section that follows.

In [ ]:
# SPARSE GIT CLONE — Download settling analysis data files
#
# Fetches ONLY Plots_Images/INV_OTA_ERROR_PLOTS/ from gmid_IHP130 repo
# using sparse checkout (--filter=blob:none --no-checkout + sparse-checkout set).
# Avoids downloading 500 MB of model files, images, and CSVs.
# IDEMPOTENCY: if REPO_DIR exists → git pull; else full sparse clone.
# _git(cmd,cwd): subprocess wrapper returning True/False on returncode.
# OUTPUT: /content/gmid_IHP130/Plots_Images/INV_OTA_ERROR_PLOTS/
#   INV_OTA_gmid.txt  (transient data for gm/ID-designed IBA)
#   INV_OTA_rogm.txt  (transient data for Ron/Gm-designed IBA)
#
import os, subprocess

REPO_DIR = "/content/gmid_IHP130"

def _git(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    if r.returncode != 0:
        print(r.stderr[-400:])
    return r.returncode == 0

if not os.path.exists(REPO_DIR):
    _git("git clone --filter=blob:none --no-checkout https://github.com/chennakeshavadasa/gmid_IHP130.git", cwd="/content")
    _git("git sparse-checkout init --cone", cwd=REPO_DIR)
    _git("git sparse-checkout set Plots_Images/INV_OTA_ERROR_PLOTS", cwd=REPO_DIR)
    _git("git checkout", cwd=REPO_DIR)
else:
    _git("git pull", cwd=REPO_DIR)

print("\u2705 Repo ready at:", REPO_DIR)

In [ ]:
# IMPORTS for settling analysis plots
#   numpy            → array maths: error, gradient, exp fit
#   plotly.graph_objects → Scatter traces for Vout, error, Iout, dVdt plots
#   plotly.io        → pio.renderers.default='colab' for inline display
#   plotly.subplots  → make_subplots() for multi-panel combined figures
#   scipy.curve_fit  → nonlinear least-squares fit for settling τ
#
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy.optimize import curve_fit
import os

# Use a clean renderer for Colab
pio.renderers.default = 'colab'

In [ ]:
# SETTLING ANALYSIS HELPERS — theme + load/extract/fit functions
#
# _SA_FS: font size dict — change here to resize ALL settling plots at once.
#
# _SA_COLORS: signal colour palette.
#   'vout'='#E63946'(red), 'vfinal'='#1D3557'(navy), 'error'='#457B9D'(blue),
#   'expfit'='#2A9D8F'(teal), 'iout'='#2DC653'(green), 'dvdt'='#7B2D8B'(violet)
#
# _SA_LAYOUT: base Plotly layout dict (width, height, hovermode, margin,
#   legend, hoverlabel) unpacked with **_SA_LAYOUT in every figure.
#
# _sa_styled_axis(title_text): returns axis config dict with title, tick
#   font, grid, zeroline=False, autorange=True.
#
# _sa_load_data(file_path):
#   Parses whitespace-delimited SPICE transient file, skipping * and # lines.
#   Extracts col0=time, col1=Vout, col11=Vmeas from rows with ≥12 columns.
#   Returns (times, vouts, vmeas) numpy arrays.
#
# _sa_extract_settling(times, vouts, vmeas, Vout_final, start, end):
#   Slices [start_time, end_time] window; computes relative settling error
#   error = (Vout_final - Vout)/Vout and dVout/dt via np.gradient.
#
# _sa_try_exp_fit(fig, valid_t_us, valid_err):
#   Fits err(t) = A·exp(-t/τ) to the decaying portion of the error curve.
#   Strategy: find decreasing indices → start at 25th percentile → remove
#   outliers >10×median → scipy.curve_fit() → overlay dashed teal trace.
#   Returns τ in µs, or None if fit fails.
#
#  FONT SIZE CONTROLS  — change these to resize everything at once
_SA_FS = dict(
    axis_label   = 20,   # xlabel / ylabel on standalone plots
    axis_tick    = 15,   # tick numbers on standalone plots
    legend       = 17,   # legend text (standalone + combined panels)
    annotation   = 16,   # Vfinal annotation on the plot
    panel_label  = 18,   # ylabel on combined 4-panel figures
    panel_tick   = 14,   # tick numbers on combined 4-panel figures
    title        = 20,   # individual plot titles
    combo_title  = 22,   # combined-panel main title
)

#  Colour palette
_SA_COLORS = {
    'vout'   : '#E63946',   # vivid red
    'vfinal' : '#1D3557',   # dark navy  (dashed reference line)
    'error'  : '#457B9D',   # steel blue
    'expfit' : '#2A9D8F',   # teal       (exponential fit)
    'iout'   : '#2DC653',   # emerald green
    'dvdt'   : '#7B2D8B',   # deep violet
}

#  Base layout — matches gmid notebook style
_SA_LAYOUT = dict(
    width        = 1200,
    height       = 650,
    hovermode    = "x unified",
    margin       = dict(l=50, r=50, t=50, b=50),
    legend       = dict(
        font       = dict(size=_SA_FS['legend']),
        title_text = "Signal",
    ),
    hoverlabel   = dict(
        font_size  = 16,        # uniform popup font size across all plots
        namelength = -1,        # never truncate the trace name
    ),
)

def _sa_styled_axis(title_text, font_size=None):
    """Return an axis dict matching the gmid notebook style."""
    if font_size is None:
        font_size = _SA_FS['axis_label']
    return dict(
        title     = dict(text=title_text, font=dict(size=font_size)),
        tickfont  = dict(size=_SA_FS['axis_tick']),
        showgrid  = True,
        zeroline  = False,
        autorange = True,
    )

#  Data loader
def _sa_load_data(file_path):
    """Parse a SPICE-style whitespace-delimited data file."""
    times, vouts, vmeas = [], [], []
    with open(file_path, 'r') as fid:
        for line in fid:
            line = line.strip()
            if not line or line.startswith('*') or line.startswith('#'):
                continue
            vals = line.split()
            if len(vals) >= 12:
                times.append(float(vals[0]))
                vouts.append(float(vals[1]))
                vmeas.append(float(vals[11]))
    return np.array(times), np.array(vouts), np.array(vmeas)

#  Settling region extractor
def _sa_extract_settling(times, vouts, vmeas, Vout_final, start_time, end_time):
    mask        = (times >= start_time) & (times <= end_time)
    st          = times[mask]
    sv          = vouts[mask]
    sm          = vmeas[mask]
    st_us       = st * 1e6
    error       = (Vout_final - sv) / sv
    dvout_dt_us = np.gradient(sv, st) * 1e-6
    return st_us, sv, sm, error, dvout_dt_us

#  Exponential fit helper
def _sa_try_exp_fit(fig, valid_t_us, valid_err):
    """Attempt an exponential fit and add it to the figure. Returns tau or None."""
    def exp_func(x, a, tau): return a * np.exp(-x / tau)
    try:
        dec_idx = np.where(np.diff(valid_err) < 0)[0]
        if len(dec_idx) <= 10:
            return None
        fit_start = dec_idx[int(len(dec_idx) / 4)]
        ft, fe    = valid_t_us[fit_start:], valid_err[fit_start:]
        good      = fe < np.median(fe) * 10
        ft, fe    = ft[good], fe[good]
        if len(ft) <= 10:
            return None
        ft_adj    = ft - ft[0]
        popt, _   = curve_fit(exp_func, ft_adj, fe, p0=[fe[0], 0.1], maxfev=10000)
        a_fit, tau_fit = popt
        t_plot    = np.linspace(0, ft_adj.max(), 1000)
        y_plot    = exp_func(t_plot, a_fit, tau_fit)
        y_min     = valid_err.min()
        mask      = y_plot >= y_min
        fig.add_trace(go.Scatter(
            x=t_plot[mask] + ft[0], y=y_plot[mask],
            mode='lines',
            line=dict(color=_SA_COLORS['expfit'], width=2.5, dash='dash'),
            name=f'Exp fit  \u03c4 = {tau_fit:.3f} \u03bcs'
        ))
        return tau_fit
    except Exception as e:
        print(f'Exp fit skipped: {e}')
        return None

print('\u2705 Theme and helpers loaded.')


In [ ]:
# LOAD SETTLING DATA — gm/ID designed IBA
#
# FILE: INV_OTA_gmid.txt
#   Transient simulation output for IBA designed with gm/ID methodology:
#     LV_NMOS:  W=0.3μm ,  L=3.0μm ,  m=4 ; LV_PMOS:  W=0.15μm ,  L=0.25μm ,  m=4 ;  Iq,replica=0.5μA ,  Iq,signal=2.0μA )
# VOUT_FINAL_GMID = 0.55731 V (reference for settling error calculation).
# START_TIME/END_TIME = [0.05µs, 3µs] — excludes initial transient + tail noise.
# _sa_load_data() → (t_g, v_g, m_g)
# _sa_extract_settling() → (st_us_g, sv_g, sm_g, err_g, dvdt_g)
#
BASE_DIR        = "/content/gmid_IHP130/Plots_Images/INV_OTA_ERROR_PLOTS"
FILE_GMID       = os.path.join(BASE_DIR, "INV_OTA_gmid.txt")

VOUT_FINAL_GMID = 0.6756   # V
START_TIME      = 0.05e-6   # s
END_TIME        = 3e-6      # s

print(f"File    : {FILE_GMID}")
print(f"Exists  : {os.path.exists(FILE_GMID)}")

if not os.path.exists(FILE_GMID):
    raise FileNotFoundError(
        f"Data file not found: {FILE_GMID}\n"
        "Re-run Cell 54 to clone/pull the repo and ensure sparse-checkout fetched INV_OTA_ERROR_PLOTS/"
    )
t_g, v_g, m_g = _sa_load_data(FILE_GMID)
print(f"Parsed  : {len(t_g)} data points")

st_us_g, sv_g, sm_g, err_g, dvdt_g = _sa_extract_settling(
    t_g, v_g, m_g, VOUT_FINAL_GMID, START_TIME, END_TIME
)
print(f"Settling: {len(st_us_g)} points in [{START_TIME*1e6:.2f}, {END_TIME*1e6:.1f}] \u03bcs")

In [ ]:
# FIGURE 11: Vout waveform — gm/ID designed IBA
# Plots Vout vs time.  Dotted navy line marks Vout_final for visual reference.
# Ghost trace (x=[None],y=[None]) adds V_final to legend without drawing points.
#
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=st_us_g, y=sv_g, mode='lines',
    line=dict(color=_SA_COLORS['vout'], width=3),
    name='V<sub>out</sub>',
    hovertemplate='t = %{x:.3f} \u03bcs<br>V<sub>out</sub> = %{y:.5f} V<extra></extra>'
))
fig.add_hline(
    y=VOUT_FINAL_GMID,
    line=dict(color=_SA_COLORS['vfinal'], width=1.8, dash='dot'),
    annotation_text=f'V<sub>final</sub> = {VOUT_FINAL_GMID:.5f} V',
    annotation_font=dict(size=_SA_FS['annotation'], color=_SA_COLORS['vfinal']),
    annotation_position='top left'
)
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode='lines',
    line=dict(color=_SA_COLORS['vfinal'], width=1.8, dash='dot'),
    name=f'V<sub>final</sub> = {VOUT_FINAL_GMID:.5f} V'
))
fig.update_layout(
    **_SA_LAYOUT,
    title=dict(text='<b>Figure 11:</b> Output voltage (Vout) waveform of the system for IBA using traditional g<sub>m</sub>/I<sub>D</sub> based design.', font=dict(size=_SA_FS['title']), x=0.02),
    xaxis=_sa_styled_axis('Time (\u03bcs)'),
    yaxis=_sa_styled_axis('V<sub>out</sub> (V)'),
)
fig.show()


In [ ]:
# FIGURE 12: Log settling error — gm/ID designed IBA
# Filters NaN/Inf/zero error samples; constructs log decade tick marks.
# _sa_try_exp_fit() overlays fitted e^(-t/τ) dashed curve and prints τ.
# τ(gm/ID) is the small-signal settling time constant of the gm/ID design.
# Nonlinear settling phase is highlighted for both positive and negative edges.
#
valid_g      = ~np.isnan(err_g) & ~np.isinf(err_g) & (np.abs(err_g) > 0)
valid_t_g    = st_us_g[valid_g]
valid_err_g  = np.abs(err_g[valid_g])
if len(valid_err_g) == 0:
    print("⚠️  No valid (non-zero, finite) error samples found — skipping error plot.")
    valid_err_g = np.array([1e-6])
    valid_t_g   = np.array([0.0])
_emin_g = valid_err_g.min()
_emax_g = valid_err_g.max()
log_min_g = int(np.floor(np.log10(_emin_g))) if _emin_g > 0 else -6
log_max_g = int(np.ceil(np.log10(_emax_g)))  if _emax_g > 0 else  0

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=valid_t_g, y=valid_err_g, mode='lines',
    line=dict(color=_SA_COLORS['error'], width=3),
    name='Settling Error',
    hovertemplate='t = %{x:.3f} \u03bcs<br>Error = %{y:.2e}<extra></extra>'
))
tau_g = _sa_try_exp_fit(fig, valid_t_g, valid_err_g)
if tau_g: print(f'\u03c4 (gm/ID) = {tau_g:.3f} \u03bcs')

# ── highlight nonlinear settling phase — both edges, all cycles ─────────────
_period      = 1.0    # µs — full clock period
_nl_duration = 0.05   # µs — 50 ns nonlinear phase per edge
_t_max       = valid_t_g.max()
_n_cycles    = int(np.ceil(_t_max / _period))

# explicitly distinct colors — orange for +edge, steel blue for -edge
_edges = [
    (0.5, 'rgba(210, 105,  30, 0.25)', 'rgba(210, 105,  30, 1.0)', '+edge nonlinear Phase'),
    (1.0, 'rgba( 30, 144, 255, 0.25)', 'rgba( 30, 144, 255, 1.0)', '−edge nonlinear Phase'),
]

for _offset, _fill, _line_color, _label in _edges:
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(size=10, color=_fill, symbol='square'),
        name=_label, showlegend=True,
    ))
    for i in range(_n_cycles):
        _x0 = i * _period + _offset
        _x1 = _x0 + _nl_duration
        if _x0 >= _t_max:
            break
        _x1 = min(_x1, _t_max)
        fig.add_vrect(
            x0=_x0, x1=_x1,
            fillcolor=_fill,
            layer='below', line_width=0,
        )
        fig.add_vline(x=_x0, line=dict(color=_line_color, width=1.5, dash='dash'))
        fig.add_vline(x=_x1, line=dict(color=_line_color, width=1.5, dash='dash'))
# ────────────────────────────────────────────────────────────────────────────

fig.update_layout(
    **_SA_LAYOUT,
    title=dict(
        text='<b>Figure 12:</b> Logarithmic error of the output with respect to reference for IBA using traditional g<sub>m</sub>/I<sub>D</sub> based design.',
        font=dict(size=_SA_FS['title']), x=0.02
    ),
    xaxis=_sa_styled_axis('Time (\u03bcs)'),
    yaxis=dict(
        **_sa_styled_axis('|V<sub>out,final</sub> − V<sub>out</sub>| / V<sub>out</sub>'),
        type='log',
        range=[log_min_g-1, log_max_g+1],
        tickvals=[10**i for i in range(log_min_g, log_max_g+1)],
        ticktext=[f'10<sup>{i}</sup>' for i in range(log_min_g, log_max_g+1)],
    ),
)
fig.show()

It is observed that the error envelope exhibits **two distinct settling regimes**: an initial rapid decay with a **steep negative slope (500–550 ns)** attributable to **large-signal nonlinear operation**, followed by a gradual exponential decay with a **comparatively smaller slope**, characteristic of **small-signal linear settling**. It is worth noting that the boundary between
these two regimes, and consequently the proportion of the settling window occupied by the nonlinear phase, **cannot be determined beforehand from conventional design methodologies including both the $g_m/I_D$ method and the approach presented by Conrad et al [4].** This information becomes available only upon post-simulation analysis. This constitutes a
**fundamental limitation of conventional design**: the large-signal and
small-signal settling contributions **cannot be independently budgeted or optimized** at the
transistor sizing stage.

In [ ]:
# FIGURE 13: Output current Iout — gm/ID designed IBA
# sm_g × 1e6 converts branch current (A) to µA for display.
# Large initial spike = large-signal phase (high slew current).
# Exponential decay = small-signal phase (NMOS/PMOS currents balancing).
#
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=st_us_g, y=sm_g * 1e6, mode='lines',
    line=dict(color=_SA_COLORS['iout'], width=3),
    name='I<sub>out</sub>',
    hovertemplate='t = %{x:.3f} \u03bcs<br>I<sub>out</sub> = %{y:.3f} \u03bcA<extra></extra>'
))

# ── highlight nonlinear vs linear phase — both edges, all cycles ─────────────
_period      = 1.0    # µs — full clock period
_nl_duration = 0.05   # µs — 50 ns nonlinear phase per edge
_t_max       = st_us_g.max()
_n_cycles    = int(np.ceil(_t_max / _period))

_edges = [
    (0.5, 'rgba(210, 105,  30, 0.25)', 'rgba(210, 105,  30, 1.0)', '+edge nonlinear Phase'),
    (1.0, 'rgba( 30, 144, 255, 0.25)', 'rgba( 30, 144, 255, 1.0)', '−edge nonlinear Phase'),
]

for _offset, _fill, _line_color, _label in _edges:
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(size=10, color=_fill, symbol='square'),
        name=_label, showlegend=True,
    ))
    for i in range(_n_cycles):
        _x0 = i * _period + _offset
        _x1 = _x0 + _nl_duration
        if _x0 >= _t_max:
            break
        _x1 = min(_x1, _t_max)
        fig.add_vrect(
            x0=_x0, x1=_x1,
            fillcolor=_fill,
            layer='below', line_width=0,
        )
        fig.add_vline(x=_x0, line=dict(color=_line_color, width=1.5, dash='dash'))
        fig.add_vline(x=_x1, line=dict(color=_line_color, width=1.5, dash='dash'))
# ────────────────────────────────────────────────────────────────────────────

fig.update_layout(
    **_SA_LAYOUT,
    title=dict(
        text='<b>Figure 13:</b> — Output Current I<sub>out</sub> across the circuit branch for IBA using traditional g<sub>m</sub>/I<sub>D</sub> based design.',
        font=dict(size=_SA_FS['title']), x=0.02
    ),
    xaxis=_sa_styled_axis('Time (\u03bcs)'),
    yaxis=_sa_styled_axis('I<sub>out</sub> (\u03bcA)'),
)
fig.show()

It is observed that the output charging current $I_{out}$ exhibits **two distinct regimes**: an initial **large nonlinear peak current** — measured at **+12 µA** on the positive edge and **−10 µA** on the negative edge that rapidly charges/discharges the output load capacitance toward the final value, followed by a progressively decreasing small-signal current with $I_{out} \rightarrow 0$ as the output converges. This behavior is characteristic of dynamic amplifiers: **the bulk of the charge transfer occurs in the initial nonlinear phase**, with the small-signal tail contributing only the final fine settling. Critically, **neither the magnitude of the peak nonlinear current nor the asymmetry between positive and negative edges can be determined beforehand from the $g_m/I_D$ methodology or the approach of Conrad et al. [4]**. Furthermore, this **peak current varies across process corners**, introducing additional uncertainty in the settling budget that remains entirely uncharacterized until post-simulation analysis. This represents a compounded limitation of conventional methodologies: not only is the nonlinear settling duration unknown at the sizing stage, but so is the **magnitude, polarity asymmetry, and corner dependence** of the dominant charging current.


In [ ]:
# FIGURE 14: dVout/dt — gm/ID designed IBA
# dvdt_g (V/µs) = dVout/dt via np.gradient (central differences).
# dVout/dt = Iout/C_load; peak reflects peak-current large-signal phase.
# Two-phase settling (fast slew → slow exponential) visible as slope change.
#
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=st_us_g, y=dvdt_g, mode='lines',
    line=dict(color=_SA_COLORS['dvdt'], width=3),
    name='dV<sub>out</sub>/dt',
    hovertemplate='t = %{x:.3f} \u03bcs<br>dV/dt = %{y:.4f} V/\u03bcs<extra></extra>'
))
fig.update_layout(
    **_SA_LAYOUT,
    title=dict(text='<b>Figure 14:</b> Derivative of output voltage dV<sub>out</sub>/dt for IBA using traditional g<sub>m</sub>/I<sub>D</sub> based design.', font=dict(size=_SA_FS['title']), x=0.02),
    xaxis=_sa_styled_axis('Time (\u03bcs)'),
    yaxis=_sa_styled_axis('dV<sub>out</sub>/dt (V/\u03bcs)'),
)
fig.show()

---

## 5.4 Summary of the gm/Id Design and Why We need to Go Further

---

Figures 10–13 confirmed that the gm/Id-sized IBA settles correctly at 20 MHz.
But three critical quantities remained unknown until post-simulation:

| Unknown | What it governs | How discovered in gm/Id flow |
|:--|:--|:--|
| $R_{on}$ of NMOS/PMOS | RC settling time constant $\tau_{ls} = R_{on}C_L$ (the Fig 12 current spike) | Only after transient SPICE |
| Deadzone voltages $V_{DZP}$, $V_{DZN}$ | Define small-signal settling window; too wide = instability | Only after iterative corner sweeps |
| SS-corner $R_{on}$ degradation | Worst-case non-linear settling | Only after running SS corner sim |

The **$R_{on}/g_m$ methodology** below resolves all three simultaneously,
pre-simulation, using the LUT plots generated in the characterisation section.

---


# 6. Design of Inverter-Based Amplifier using the $R_{on}/g_m$ Methodology
---

#### Objective
To design an inverter-based amplifier targeting a **20 MHz bandwidth**, while ensuring effective settling behavior and low power consumption through the $R_{on}/g_m$ methodology.

---

#### 1. Transconductance Requirement
- For a settling time $T_{settle} = 250\,\text{ns}$ (5$\tau$), the required closed-loop bandwidth is:
  $$
  \omega_{cl} = \frac{1}{\tau_{cl}} = \frac{5}{T_{settle}} = 20\,\text{Mrad/s}, \qquad \text{UGB} = \frac{\omega_{cl}}{\beta} = \frac{20\,\text{Mrad/s}}{1/3} = 60\,\text{Mrad/s}
  $$

- The total required transconductance is:
  $$
  G_m = \text{UGB} \times C_{L,eff} = 60 \times 10^6 \times 667 \times 10^{-12} \approx 40\,\mu\text{S}
  $$

- Assume NMOS and PMOS contribute equally:
  $$
  g_{mn} = g_{mp} = \frac{G_m}{2} = 20\,\mu\text{S}
  $$


---
#### 2. Target On-Resistance

- To optimize **settling speed and power**, we design assuming **the dominant settling is achieved by the non-linear RC phase**, while the $g_m$-based small-signal phase only removes a small residual.

- With $C_{load} = 667\,\text{fF}$ and $T_{settle} = 250\,\text{ns}$, we target:

  $$R_{on} = 1\,\text{k}\Omega$$

  giving a non-linear settling time of:

  $$5 \times 1\,\text{k}\Omega \times 667\,\text{fF} \approx 3.3\,\text{ns}$$

  This completes the majority of the output swing well within the 250 ns budget.

- The **worst-case $R_{on}$ occurs at the SS corner**, which drives our design:

  $$\frac{R_{\text{on}}}{G_m} = \frac{1\,\text{k}\Omega}{40\,\mu\text{S}} = 25 \times 10^6, \qquad
  \frac{R_{\text{on}}}{g_{mn}} = \frac{R_{\text{on}}}{g_{mp}} = \frac{1\,\text{k}\Omega}{20\,\mu\text{S}} = 50 \times 10^6$$

- Thus, we design at:

  $$R_{on}/g_m = 50 \times 10^6 \quad (\text{SS corner})$$

---

### Pre-Simulation Predictions from $R_{on}/g_m$ Plots for NMOS

| Quantity | Source | Pre-Sim Value |
|---|---|---|
| $V_{bias,TT}$ | $V_{bias}$ vs $R_{on}/g_m$ | 0.39 V |
| $V_{bias,SS}$ | $V_{bias}$ vs $R_{on}/g_m$ | 0.42–0.43 V |
| $V_{bias,FF}$ | $V_{bias}$ vs $R_{on}/g_m$ | 0.35–0.36 V |
| $g_m/I_D$ | $g_m/I_D$ vs $R_{on}/g_m$ | $\approx 10$ |
| $I_{peak}$ | $I_{peak}$ vs $R_{on}/g_m$ | $\approx 14\,\mu\text{A}$ |

---

- The $g_m/I_D$ curve confirms **moderate inversion** at this operating point.

- The predicted **peak current**:
  - Pre-simulation: $I_{peak} \approx 14\,\mu\text{A}$
  - Post-simulation: $\approx 12\,\mu\text{A}$  
  → within **~15% accuracy**

- This reveals a **fundamental one-way asymmetry** between the two methodologies:
  - $R_{on}/g_m$: DC bias range, corner spread, $g_m/I_D$ operating point, and $I_{peak}$ are all read **directly from plots before simulation**.
  - $g_m/I_D$ alone: all of the above are **only available after simulation**. The plots provide no path to any large-signal quantity.

  The $R_{on}/g_m$ methodology strictly subsumes $g_m/I_D$; the reverse is impossible.

---
#### 3. Design Units and Scaling
- **Base Units Defined:**
  - $R_{\text{unit}} = 8\,k\Omega$
  - $I_{\text{unit}} = 1\,\mu A$
  - $g_{m,\text{unit}} = 2*20\,\mu \text{S}$
- The **main path** structure can be scaled using **multipliers** depending on application requirements (e.g., settling time, drive strength).
- The **replica path** can tune $g_m$ independently by adjusting only $I_{\text{unit}}$, leaving the main structure unchanged.

---
#### 4. Insights and Advantages
- This methodology offers deeper understanding of **non-linear settling** by explicitly incorporating $R_{\text{on}}$.
- Provides **predictable Deadzone regions**, eliminating the need for iterative simulations.
- Enables systematic tuning via transistor multipliers:
  - As the **multiplication factor increases**, $R_{\text{on}}$ decreases, further improving performance.

  ---


## 6.1 $R_{on}/G_m$ Design Helper
---

> **Design Step:** The calculation for the unit cell above fixed $R_{on}/g_m = 50\times10^6$ and $I_{bias} = 0.5\,\mu\text{A}$, confirmed by reading Plot 1 and Plot 5.
> Use this helper to search the pre-characterised SPICE LUT for the exact $(W, L, V_{bias})$ that meets these targets at your chosen corner.
> **Set `Ron/Gm` to your target → set `i-sweep` to bias current → select SS corner → read the ★ green best-match row.**

An inverter-based dynamic amplifier (IBA) operates in two sequential phases.
During the large-signal phase, the output node is driven through the MOSFET switch,
whose on-resistance $R_{on}$ sets the RC settling time constant $\tau_{\text{ls}} = R_{on}C_L$.
During the subsequent small-signal phase, the transconductance $G_m$ sets the
exponential settling time constant $\tau_{\text{ss}} = C_L/G_m$.
Expressed together:

$$\tau_{\text{ls}} = R_{on}C_L, \qquad \tau_{\text{ss}} = \frac{C_L}{G_m}$$

A single $R_{on}/G_m$ target constrains both time constants simultaneously
$R_{on}$ bounds the large-signal settling budget and $G_m$ bounds the small-signal
settling bandwidth without requiring separate iterations.

The LUT is pre-characterised across different $V_{GS}$, $V_{DS}$ (So that user can design the $R_{on}/G_m$ based on the input swing eg,. Lets say we want to design the dynamic amplifier for an input swing of VDD/2 or Vdd/4 and so on), channel length $L$,
and process corner. For a specified bias condition $(V_{GS}, V_{DS}, \text{corner})$,
the helper filters the LUT and executes a $\sigma$-normalised $L_2^2$
nearest-neighbour search over the three-dimensional design space
$(I_D,\; (R_{on}/G_m)_{\text{unit}},\; L)$:

$$\mathcal{L} = \left(\frac{I_D - \hat{I}_D}{\sigma_{I}}\right)^{\!2}
              + \left(\frac{(R_{on}/G_m)_{\text{unit}} - \widehat{(R_{on}/G_m)}_{\text{unit}}}{\sigma_{RG}}\right)^{\!2}
              + \left(\frac{L - \hat{L}}{\sigma_{L}}\right)^{\!2}$$

$\sigma$-normalisation is essential: $I_D$, $(R_{on}/G_m)_{\text{unit}}$, and $L$ span
incompatible numerical scales and cannot be combined without it.
The best match returns $V_{\text{bias}}$, device width $W$, and channel length $L$ fully specifying the transistor operating point pre-simulation.

In [ ]:
# STEP 7 — DOWNLOAD Ron/Gm LUT CSV DATA (NMOS & PMOS)
#
# Downloads 10 pre-simulated LUT CSVs (5 lengths × NMOS+PMOS, TT corner)
# for the Ron/Gm Design Helper.  Uses individual file IDs via gdown.download()
# to bypass Drive's virus-scan confirmation wall.
# Caching: if dest file already exists → skip. Failed files listed for re-run.
# OUTPUT: /content/backtrack/*.csv
#
#  DOWNLOAD RON/GM LUT DATA
# Downloads NMOS + PMOS CSV files into /content/backtrack/
#
# FIX: gdown.download_folder() fails on individual file downloads when Drive
# triggers a virus-scan / rate-limit confirmation page.  Solution: download
# each file directly by its file-ID, which bypasses the confirmation wall.

import os, gdown
from pathlib import Path

BACKTRACK_DIR = Path("/content/backtrack")
BACKTRACK_DIR.mkdir(parents=True, exist_ok=True)

# File IDs extracted from the Google Drive folder
# (https://drive.google.com/drive/folders/1aCO-KZy-Yxk0TTw5Z1WVA9U1asWvCi06)
FILE_IDS = {
    "CAC_ONRES_PAPER_LV_NMOS_L_0_20_Ibias_tt.csv":  "1KyrzHRpZ7oktrMkrKQ1Dc7NpmIidzkHO",
    "CAC_ONRES_PAPER_LV_NMOS_L_0_50_Ibias_tt.csv":  "1Zu0koFa945A5ni2EuGLCbJs2oEYBPYlN",
    "CAC_ONRES_PAPER_LV_NMOS_L_1_00_Ibias_tt.csv":  "104a7h2Nobc4HhsY-SLj79JvdukIXBddF",
    "CAC_ONRES_PAPER_LV_NMOS_L_4_00_Ibias_tt.csv":  "1d96udsdlHrzoCGJqelbzgX3kHSOeNAaz",
    "CAC_ONRES_PAPER_LV_NMOS_L_10_00_Ibias_tt.csv": "1mwcVYThl_XBk8HbgFh-PboBgW-4zVdxP",
    "CAC_ONRES_PAPER_LV_PMOS_L_0_20_Ibias_tt.csv":  "1RPAVndmpJLxn9szVmTx1iea55Wnc_76V",
    "CAC_ONRES_PAPER_LV_PMOS_L_0_50_Ibias_tt.csv":  "1N_Dhj6KnTS2w6bqEnV-V0TXjTr7n4sab",
    "CAC_ONRES_PAPER_LV_PMOS_L_1_00_Ibias_tt.csv":  "19MXA2jOZ1XR-58charXaGHGs2WkbQ06U",
    "CAC_ONRES_PAPER_LV_PMOS_L_4_00_Ibias_tt.csv":  "1l8uk5kwaFXxz8ww-ByC2b6Ht0pZkPrnP",
    "CAC_ONRES_PAPER_LV_PMOS_L_10_00_Ibias_tt.csv": "1ud0oeIoe8T3HmEdSBuXj-dVR5owQPUWe",
}

print(f"Downloading {len(FILE_IDS)} Ron/Gm LUT CSV files into {BACKTRACK_DIR}\n")
failed = []
for fname, fid in FILE_IDS.items():
    dest = BACKTRACK_DIR / fname
    if dest.exists():
        print(f"  {fname}  [cached]")
        continue
    try:
        gdown.download(id=fid, output=str(dest), quiet=True, fuzzy=False)
        ok = dest.exists() and dest.stat().st_size > 0
        print(f"  {'✓' if ok else '✗'} {fname}")
        if not ok:
            failed.append(fname)
    except Exception as e:
        print(f"  ✗ {fname}  ({e})")
        failed.append(fname)

# Final inventory
csv_files  = list(BACKTRACK_DIR.glob("*.csv"))
nmos_count = sum(1 for f in csv_files if "NMOS" in f.name.upper())
pmos_count = sum(1 for f in csv_files if "PMOS" in f.name.upper())
icon = "✓" if not failed else "⚠️"
print(f"\n{icon} {BACKTRACK_DIR}: {len(csv_files)} CSV(s) total "
      f"| NMOS: {nmos_count} | PMOS: {pmos_count}")
if failed:
    print(f"   Failed: {failed}")
    print("   Tip: re-run this cell — transient Drive rate-limits usually clear in 30s.")


In [ ]:
# IMPORTS AND COLOUR CONSTANTS — Ron/Gm Design Helper
# plt.style.use('dark_background') → dark matplotlib theme.
# ACCENT_G='#00ffb2'(neon green=best), ACCENT_B='#3b82f6'(blue=loss curve),
# ACCENT_Y='#f59e0b'(amber=query), ACCENT_R='#ef4444'(red=bound/vline),
# BG_MAIN='#080b12', BG_PANEL='#0e1320', GRID_COL='#1e2535', LABEL_COL='#94a3b8'
#
# IMPORTS
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import os

# Dark theme (matches V1 dashboard)
plt.style.use('dark_background')

# Accent palette
ACCENT_G  = '#00ffb2'   # Neon green  — best / optimum
ACCENT_B  = '#3b82f6'   # Royal blue  — nearby
ACCENT_Y  = '#f59e0b'   # Amber       — input target
ACCENT_R  = '#ef4444'   # Red         — bound / vline
BG_MAIN   = '#080b12'
BG_PANEL  = '#0e1320'
GRID_COL  = '#1e2535'
LABEL_COL = '#94a3b8'

print("Imports OK")

In [ ]:
# LOAD RON/GM LUT DATA + COLUMN NORMALISATION
#
# DATA DISCOVERY: searches CANDIDATE_DIRS for NMOS/PMOS CSV files.
#
# _normalise(df, is_pmos): handles 3 CSV column-name conventions:
#   1. New PMOS: columns 'VSG','VSD' → rename to v(VG),v(VD).
#   2. Old PMOS: v(VG) stores VGD (negative) → auto-correct: VSG = VSD - VGD.
#   3. NMOS: v(VG)=VGS, v(VD)=VDS (positive, no change).
#   After normalisation: v(VG)=gate-source, v(VD)=drain-source, both ≥0.
#   Rounds to 4 dp to kill SPICE float noise; drops PMOS rows with VSG>VDD.
#
# _load_files(): reads+normalises each CSV, tags '_source_file', concatenates.
# check_cols(): validates required columns, prints row count + corner/VG/VD ranges.
#
# LOAD DATA

import os, numpy as np
from pathlib import Path

VDD = 1.65   # IHP SG13G2 LV supply

CANDIDATE_DIRS = [
    "/content/backtrack",
    "/content/1aCO-KZy-Yxk0TTw5Z1WVA9U1asWvCi06",
]
for d in Path("/content").iterdir():
    if d.is_dir() and d.name not in ("sample_data", "spice_generated",
                                      "simulation_results", "logs",
                                      "models", "psp_va"):
        CANDIDATE_DIRS.append(str(d))

def find_csvs_in(directory, keyword):
    d = Path(directory)
    if not d.exists():
        return []
    return [f for f in d.glob("*.csv") if keyword.upper() in f.name.upper()]

BASE = None
nmos_files, pmos_files = [], []
for cand in CANDIDATE_DIRS:
    n = find_csvs_in(cand, "NMOS")
    p = find_csvs_in(cand, "PMOS")
    if n or p:
        BASE, nmos_files, pmos_files = cand, n, p
        break

if BASE is None:
    raise FileNotFoundError(
        "Could not find backtrack CSV data.\n"
        "Please run the download cell (Cell 65) first."
    )

print(f" {BASE}")
print(f"   NMOS ({len(nmos_files)}): {[f.name for f in nmos_files]}")
print(f"   PMOS ({len(pmos_files)}): {[f.name for f in pmos_files]}")

# Normalisation
# Three CSV conventions exist:
#
#  1. New PMOS  — columns VSG, VSD (correct, just rename)
#  2. Old PMOS  — columns v(VG), v(VD) but VG stored as VGD (negative) → fix
#  3. NMOS      — columns v(VG), v(VD), both positive, no change needed
#
# After normalisation every DataFrame uses v(VG) and v(VD) where:
#   NMOS: v(VG)=VGS, v(VD)=VDS  (both positive, higher=more on)
#   PMOS: v(VG)=VSG, v(VD)=VSD  (both positive, higher=more on)
#
# PMOS validity clamp: VSG and VSD must both be <= VDD.
# The testbench sweep can produce VSG > VDD artefacts when vdsx and V2
# are simultaneously at extreme values — these rows are physically impossible
# and are dropped here.

def _normalise(df, is_pmos=False):
    cols = set(df.columns)

    if "VSG" in cols and "VSD" in cols:
        df = df.rename(columns={"VSG": "v(VG)", "VSD": "v(VD)"})

    elif "v(VG)" in cols and "v(VD)" in cols:
        if df["v(VG)"].min() < -0.01:   # old bad PMOS VGD format
            df["v(VG)"] = df["v(VD)"] - df["v(VG)"]

    # Round to 4 dp to kill SPICE float noise
    for col in ("v(VG)", "v(VD)"):
        if col in df.columns:
            df[col] = np.round(df[col].astype(float), 4)

    # Drop physically impossible rows (VSG or VSD > VDD)
    if is_pmos and "v(VG)" in df.columns and "v(VD)" in df.columns:
        before = len(df)
        df = df[(df["v(VG)"] <= VDD + 0.01) & (df["v(VD)"] <= VDD + 0.01)].copy()
        dropped = before - len(df)

    return df


def _load_files(file_list, device_label, is_pmos=False):
    frames = []
    for fp in file_list:
        try:
            df_tmp = pd.read_csv(fp)
            df_tmp.columns = df_tmp.columns.str.strip()
            df_tmp = _normalise(df_tmp, is_pmos=is_pmos)
            df_tmp["_source_file"] = fp.name
            frames.append(df_tmp)
        except Exception as e:
            print(f"  ⚠️  Could not read {fp.name}: {e}")
    if not frames:
        print(f"  ⚠️  No valid {device_label} files — returning empty DataFrame")
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


df_nmos_raw = _load_files(nmos_files, "NMOS", is_pmos=False)
df_pmos_raw = _load_files(pmos_files, "PMOS", is_pmos=True)

for df in (df_nmos_raw, df_pmos_raw):
    if not df.empty and "corner" in df.columns:
        df["corner"] = df["corner"].astype(str).str.strip().str.upper()

REQUIRED_COLS = {"v(VG)", "v(VD)", "i-sweep", "Ron_Gm_unit",
                 "length", "width", "v(Vbias)", "corner"}

def check_cols(df, label):
    if df.empty:
        print(f"  ⚠️  {label}: no data loaded"); return
    missing = REQUIRED_COLS - set(df.columns)
    if missing:
        print(f"  ⚠️  {label}: missing columns {missing}")
    else:
        vg_u = sorted(df['v(VG)'].unique())
        vd_u = sorted(df['v(VD)'].unique())
        print(f"  {label}: {len(df):,} rows | corners: {sorted(df['corner'].unique())}")
        print(f"     v(VG) [{len(vg_u)}]: {[round(v,4) for v in vg_u]}")
        print(f"     v(VD) [{len(vd_u)}]: {[round(v,4) for v in vd_u]}")

check_cols(df_nmos_raw, "NMOS")
check_cols(df_pmos_raw, "PMOS")
print(f"\nNMOS rows: {len(df_nmos_raw):,} | PMOS rows: {len(df_pmos_raw):,}")


In [ ]:
# FILTER + NEAREST-NEIGHBOUR SEARCH — Ron/Gm Design Helper
#
# apply_filter(df, vg, vd, corner):
#   Selects rows matching (VGS/VSG, VDS/VSD, corner) using np.isclose(atol=1e-3)
#   to tolerate SPICE floating-point rounding.  Sorted by i-sweep.
#
# compute(df, i_sweep, ron, length):
#   std-normalised L2² distance across (i-sweep, Ron_Gm_unit, length):
#     dist = ((i_sweep_col-i_sweep)/std)² + ((Ron-ron)/std)² + ((L-length)/std)²
#   Normalisation critical: three quantities have completely different scales.
#   'or 1' guards against std=0 on degenerate single-value columns.
#   Returns: best (min-dist row), top5 (5 closest), dist (full Series).
#
# FILTER HELPER
# Both NMOS and PMOS DataFrames now use v(VG) and v(VD) internally.
# NMOS: v(VG)=VGS, v(VD)=VDS  (positive when on)
# PMOS: v(VG)=VSG, v(VD)=VSD  (positive when on — normalised on load)

def apply_filter(df, vg, vd, corner):
    if df.empty:
        return pd.DataFrame()
    mask = (
        np.isclose(df["v(VG)"], vg, atol=1e-3) &
        np.isclose(df["v(VD)"], vd, atol=1e-3) &
        (df["corner"] == corner)
    )
    return df[mask].sort_values(by="i-sweep").reset_index(drop=True)


# COMPUTE BEST MATCH
def compute(df, i_sweep, ron, length):
    is_std  = df["i-sweep"].std()      or 1
    ron_std = df["Ron_Gm_unit"].std()  or 1
    len_std = df["length"].std()       or 1
    dist = (
        ((df["i-sweep"]     - i_sweep) / is_std )**2 +
        ((df["Ron_Gm_unit"] - ron)     / ron_std)**2 +
        ((df["length"]      - length)  / len_std)**2
    )
    best = df.loc[dist.idxmin()]
    top5 = df.loc[dist.nsmallest(5).index]
    return best, top5, dist


In [ ]:
# VISUALISATION — render()  (Ron/Gm Design Helper)
#
# Renders 2-panel matplotlib figure + Best Match printout + Top-5 table.
#
# DATA FLOW: reads vg/vd/corner from dropdowns → apply_filter() → df_use
# → compute() → best, top5, dist; score = 100×(1 - min_dist/max_dist).
#
# PANEL 1 — SCATTER (Ron_Gm_unit vs i-sweep):
#   Plasma colourmap per length; markers:
#     ▲ amber (query), ★ neon-green (best match),
#     white dashed connector (query→best), red dashed hline (Ron bound).
#
# PANEL 2 — LOSS SURFACE:
#   L2² loss vs i-sweep sweep (ron/length fixed at best values). Log Y-axis.
#   Narrow minimum = well-constrained design; green dot marks best match.
#
# PRINTED OUTPUTS:
#   Best Match (width, length, Vbias, quality%), Convergence Trace (6 dist vals),
#   Top-5 styled table with heat-map distance (green/amber/red).
#
# RENDER (dark-themed V1-style, 2-panel)
def render(device_val, i_sweep, ron, length):
    with out:
        clear_output(wait=True)

        vg     = vg_select.value
        vd     = vd_select.value
        corner = corner_select.value

        df_n = apply_filter(df_nmos_raw, vg, vd, corner)
        df_p = apply_filter(df_pmos_raw, vg, vd, corner)

        if   device_val == "NMOS": df_use = df_n
        elif device_val == "PMOS": df_use = df_p
        else:                      df_use = pd.concat([df_n, df_p])

        if len(df_use) == 0:
            print("⚠️  No valid data for selected filters")
            return

        best, top5, dist = compute(df_use, i_sweep, ron, length)
        score = 100 * (1 - dist.min() / (dist.max() or 1))

        # Layout: 2 panels
        fig, (ax_sc, ax_ls) = plt.subplots(1, 2, figsize=(16, 6))
        fig.patch.set_facecolor(BG_MAIN)

        for ax in (ax_sc, ax_ls):
            ax.set_facecolor(BG_PANEL)
            ax.grid(True, color=GRID_COL, alpha=0.5)
            ax.tick_params(colors=LABEL_COL, labelsize=11)
            for spine in ax.spines.values():
                spine.set_edgecolor(GRID_COL)

        # PANEL 1: Scatter — per-length coloured dots
        lengths_sorted = sorted(df_use["length"].unique())
        cmap = plt.get_cmap('plasma', len(lengths_sorted))
        for idx_l, L in enumerate(lengths_sorted):
            mask = df_use["length"] == L
            ax_sc.scatter(
                df_use.loc[mask, "i-sweep"],
                df_use.loc[mask, "Ron_Gm_unit"],
                color=cmap(idx_l), alpha=0.30, s=22,
                label=f"L={L:.2g}", zorder=2
            )

        # Ron bound dashed line at slider value
        ax_sc.axhline(ron, color=ACCENT_R, linestyle='--', alpha=0.5, label='Ron bound', zorder=3)

        # Dashed connector  input → best
        ax_sc.plot(
            [i_sweep, best["i-sweep"]], [ron, best["Ron_Gm_unit"]],
            '--', color='white', linewidth=1, alpha=0.4, zorder=4
        )
        # Input target  (▲ amber / Query)
        ax_sc.scatter(
            i_sweep, ron,
            color=ACCENT_Y, marker='^', s=160,
            edgecolors='black', linewidths=1, label='Query', zorder=6
        )
        # Best match  (★ neon green)
        ax_sc.scatter(
            best["i-sweep"], best["Ron_Gm_unit"],
            color=ACCENT_G, marker='*', s=260,
            edgecolors='white', linewidths=0.8, label='Best match', zorder=7
        )

        ax_sc.set_title(
            f"Scatter · Ron₁₂ vs i-sweep",
            color=LABEL_COL, pad=12, fontsize=15, fontweight="bold"
        )
        ax_sc.set_xlabel("i-sweep (A)",      color=LABEL_COL, fontsize=13)
        ax_sc.set_ylabel("Ron_Gm_unit (Ω)",  color=LABEL_COL, fontsize=13)
        ax_sc.legend(fontsize=11, ncol=2, frameon=True,
                     facecolor=BG_PANEL, edgecolor=GRID_COL,
                     labelcolor=LABEL_COL)

        # PANEL 2: Loss Surface
        sweep = np.linspace(df_use["i-sweep"].min(), df_use["i-sweep"].max(), 400)
        # Normalised L2² distance sweeping only i-sweep dimension
        is_std  = df_use["i-sweep"].std()     or 1
        ron_std = df_use["Ron_Gm_unit"].std() or 1
        len_std = df_use["length"].std()      or 1
        loss = (
            ((sweep - i_sweep) / is_std)**2 +
            ((ron   - best["Ron_Gm_unit"]) / ron_std)**2 +
            ((length- best["length"]) / len_std)**2
        )

        ax_ls.plot(sweep, loss, color=ACCENT_B, linewidth=2,
                   label='Loss (sweep i-sweep)')
        ax_ls.fill_between(sweep, loss, alpha=0.10, color=ACCENT_B)

        ax_ls.axvline(i_sweep,         color=ACCENT_R, linestyle='--', alpha=0.55)
        best_loss_val = loss[np.argmin(np.abs(sweep - best["i-sweep"]))]
        ax_ls.scatter(best["i-sweep"], best_loss_val,
                      color=ACCENT_G, s=120, zorder=10, label='Best match')

        ax_ls.set_yscale('log')
        ax_ls.set_title(
            f"Loss Surface  (Ron/Gm={ron:.4g}, L={length:.3g} m)",
            color=LABEL_COL, pad=12, fontsize=15, fontweight="bold"
        )
        ax_ls.set_xlabel("i-sweep (A)",             color=LABEL_COL, fontsize=13)
        ax_ls.set_ylabel("Loss (normalised L2²)",   color=LABEL_COL, fontsize=13)
        ax_ls.legend(fontsize=11, frameon=True,
                     facecolor=BG_PANEL, edgecolor=GRID_COL,
                     labelcolor=LABEL_COL)

        plt.tight_layout()
        plt.show()

        # BEST MATCH (bold CLI)
        B = '\033[1m';  E = '\033[0m'   # ANSI bold on/off
        print(f"\n{B}── BEST MATCH {'─'*50}{E}")
        print(f"  {B}width   {E}: {B}{best['width']:.3e}{E} m")
        print(f"  {B}length  {E}: {B}{best['length']:.3e}{E} m")
        print(f"  {B}v(Vbias){E}: {B}{best['v(Vbias)']:.6f}{E} V")
        print(f"  {B}Match Quality: {score:.2f}%{E}")

        # Convergence trace (V1 style)
        dist_sorted = dist.sort_values()
        print(f"\n{B}── CONVERGENCE TRACE {'─'*47}{E}")
        for step, (_, val) in enumerate(dist_sorted.head(6).items(), 1):
            print(f"  i{step}: {val:.3e}", end="  ")
        print()

        # Top-5 table with heat-map dist column
        print(f"\n{B}── TOP 5 MATCHES {'─'*51}{E}")
        tbl = top5[["i-sweep", "Ron_Gm_unit", "length", "width", "v(Vbias)"]].copy()
        tbl["distance"] = dist[top5.index]

        # Normalise dist to [0,1] for colour mapping (0=best=green, 1=worst=red)
        d_min, d_max = tbl["distance"].min(), tbl["distance"].max()
        d_range = d_max - d_min if d_max != d_min else 1

        def dist_color(val):
            t = (val - d_min) / d_range          # 0 = best, 1 = worst
            if   t < 0.33: bg = '#1a7a3a'        # green
            elif t < 0.66: bg = '#a07800'        # amber/yellow
            else:          bg = '#8b1a1a'        # red
            return f'background-color: {bg}; color: white; font-weight: bold; text-align: center'

        styled = (
            tbl.style
               .format({'distance': '{:.2e}'})
               .map(dist_color, subset=['distance'])
        )
        display(styled)

In [ ]:
# WIDGET LAYOUT AND CALLBACKS — Ron/Gm Design Helper Dashboard
#
# _vals(df, col): safe unique-value extractor; returns [] if df empty or col absent.
# vg/vd/corner_vals: union of NMOS+PMOS unique values with sane fallbacks.
#
# WIDGETS:
#   device (ToggleButtons: NMOS/PMOS/BOTH)
#   vg_select, vd_select, corner_select (Dropdowns)
#   w_isweep, w_ron, w_len (SelectionSliders: snapped to real data values)
#   t_isweep, t_ron, t_len (FloatText: two-way sync with sliders)
#
# CALLBACKS:
#   update_data(): rebuilds slider option lists on filter change.
#   _on_slider_change(): syncs slider→text, calls render().
#   _on_text_change(): finds nearest snap point, sets slider.
#
# LAYOUT: controls_top (HBox dropdowns), controls_bot (VBox slider rows).
# display() renders header+device+controls+status+output; update_data() fires render.
#
# UNIQUE VALUES

def _vals(df, col):
    if df.empty or col not in df.columns:
        return []
    return sorted(df[col].unique())

vg_nmos     = _vals(df_nmos_raw, "v(VG)")
vg_pmos     = _vals(df_pmos_raw, "v(VG)")
vg_vals     = sorted(set(vg_nmos) | set(vg_pmos)) or [1.65]
vd_vals     = sorted(set(_vals(df_nmos_raw, "v(VD)")) |
                     set(_vals(df_pmos_raw, "v(VD)"))) or [0.05]
corner_vals = sorted(set(_vals(df_nmos_raw, "corner")) |
                     set(_vals(df_pmos_raw, "corner"))) or ["TT"]

print(f"VGS  (NMOS) range : {vg_nmos[0]:.4f} → {vg_nmos[-1]:.4f}")
print(f"VSG  (PMOS) range : {vg_pmos[0]:.4f} → {vg_pmos[-1]:.4f}")
print(f"VD/S range        : {vd_vals}")
print(f"Corners           : {corner_vals}")

# WIDGETS
device = widgets.ToggleButtons(
    options=["NMOS", "PMOS", "BOTH"],
    description="Device",
    style={"button_width": "80px"},
    layout=widgets.Layout(margin="0 0 8px 0")
)

vg_select = widgets.Dropdown(
    options=vg_nmos or vg_vals,
    description="VGS / VSG",
    value=(vg_nmos or vg_vals)[-1],
    style={"description_width": "80px"},
    layout=widgets.Layout(width="260px")
)
vd_select = widgets.Dropdown(
    options=vd_vals,
    description="VDS / VSD",
    value=vd_vals[0],
    style={"description_width": "80px"},
    layout=widgets.Layout(width="260px")
)
corner_select = widgets.Dropdown(
    options=corner_vals,
    description="Corner",
    value=corner_vals[0],
    layout=widgets.Layout(width="180px")
)

# Sliders (snapped to real data values)
w_isweep = widgets.SelectionSlider(
    description="i-sweep", options=[0],
    layout=widgets.Layout(width='420px')
)
w_ron = widgets.SelectionSlider(
    description="Ron/Gm", options=[0],
    layout=widgets.Layout(width='420px'),
    #value=49878050.1
)
w_len = widgets.SelectionSlider(
    description="length", options=[0],
    layout=widgets.Layout(width='420px')
)

# Linked text inputs (type a value, slider snaps to nearest)
t_isweep = widgets.FloatText(
    description="", value=0,
    layout=widgets.Layout(width='130px'),
    step=1e-6
)
t_ron = widgets.FloatText(
    description="", value=0,
    layout=widgets.Layout(width='130px'),
    step=1e4
)
t_len = widgets.FloatText(
    description="", value=0,
    layout=widgets.Layout(width='130px'),
    step=1e-7
)

out = widgets.Output()
status_label = widgets.HTML(value="<i style='color:#94a3b8'>Initialising…</i>")

# Sync helpers
# These flags prevent infinite observer loops when one widget updates the other
_syncing = False

def _snap_to_nearest(value, options):
    """Return the option closest to value."""
    arr = np.array(options)
    return arr[np.argmin(np.abs(arr - value))]

def _on_slider_isweep(change):
    global _syncing
    if _syncing: return
    _syncing = True
    t_isweep.value = change['new']
    _syncing = False

def _on_text_isweep(change):
    global _syncing
    if _syncing or not w_isweep.options or w_isweep.options == [0]: return
    _syncing = True
    w_isweep.value = _snap_to_nearest(change['new'], w_isweep.options)
    _syncing = False

def _on_slider_ron(change):
    global _syncing
    if _syncing: return
    _syncing = True
    t_ron.value = change['new']
    _syncing = False

def _on_text_ron(change):
    global _syncing
    if _syncing or not w_ron.options or w_ron.options == [0]: return
    _syncing = True
    w_ron.value = _snap_to_nearest(change['new'], w_ron.options)
    _syncing = False

def _on_slider_len(change):
    global _syncing
    if _syncing: return
    _syncing = True
    t_len.value = change['new']
    _syncing = False

def _on_text_len(change):
    global _syncing
    if _syncing or not w_len.options or w_len.options == [0]: return
    _syncing = True
    w_len.value = _snap_to_nearest(change['new'], w_len.options)
    _syncing = False

w_isweep.observe(_on_slider_isweep, names='value')
t_isweep.observe(_on_text_isweep,  names='value')
w_ron   .observe(_on_slider_ron,   names='value')
t_ron   .observe(_on_text_ron,     names='value')
w_len   .observe(_on_slider_len,   names='value')
t_len   .observe(_on_text_len,     names='value')

print("\n Widgets + linked text inputs created")


In [ ]:
# UPDATE DATA CALLBACK

def _get_df_for_device(device_val):
    vg, vd, corner = vg_select.value, vd_select.value, corner_select.value
    df_n = apply_filter(df_nmos_raw, vg, vd, corner)
    df_p = apply_filter(df_pmos_raw, vg, vd, corner)
    if   device_val == "NMOS": return df_n, df_p, df_n
    elif device_val == "PMOS": return df_n, df_p, df_p
    else:                      return df_n, df_p, pd.concat([df_n, df_p], ignore_index=True)


def _update_vg_vd_options():
    dev, corner = device.value, corner_select.value
    if dev == "NMOS":   src_df = df_nmos_raw
    elif dev == "PMOS": src_df = df_pmos_raw
    else:               src_df = pd.concat([df_nmos_raw, df_pmos_raw], ignore_index=True)
    if src_df.empty: return
    sub    = src_df[src_df["corner"] == corner] if corner in src_df["corner"].values else src_df
    new_vg = sorted(sub["v(VG)"].unique()) or vg_vals
    new_vd = sorted(sub["v(VD)"].unique()) or vd_vals
    cur_vg = vg_select.value
    vg_select.options = new_vg
    vg_select.value   = cur_vg if cur_vg in new_vg else new_vg[-1]
    cur_vd = vd_select.value
    vd_select.options = new_vd
    vd_select.value   = cur_vd if cur_vd in new_vd else new_vd[0]


def update_data(*args):
    global _syncing
    _update_vg_vd_options()
    _, _, df_use = _get_df_for_device(device.value)

    if df_use.empty:
        src = df_pmos_raw if device.value == "PMOS" else df_nmos_raw
        valid_vg = sorted(src["v(VG)"].unique()) if not src.empty else []
        vg_label = "VSG" if device.value == "PMOS" else "VGS"
        status_label.value = (
            f"<span style='color:#ef4444'>⚠️ No data — {device.value} "
            f"{vg_label}={vg_select.value:.4g} V, "
            f"VD={vd_select.value:.4g} V, {corner_select.value} &nbsp;|"
            f"&nbsp;Valid {vg_label}: {[round(v,4) for v in valid_vg]}</span>"
        )
        with out:
            clear_output()
            print(f"⚠️  No data. Valid {vg_label}: {[round(v,4) for v in valid_vg]}")
        return

    is_vals      = sorted(df_use["i-sweep"].unique())
    ron_vals_cur = sorted(df_use["Ron_Gm_unit"].unique())
    len_vals     = sorted(df_use["length"].unique())

    if not is_vals or not ron_vals_cur or not len_vals:
        status_label.value = "<span style='color:#ef4444'>⚠️ Columns empty after filter</span>"
        return

    # Update sliders — seed text boxes from the new midpoint values
    _syncing = True
    w_isweep.options = is_vals;       w_isweep.value = is_vals[len(is_vals) // 2]
    w_ron.options    = ron_vals_cur;  w_ron.value    = ron_vals_cur[len(ron_vals_cur) // 2]
    w_len.options    = len_vals;      w_len.value    = len_vals[len(len_vals) // 2]
    t_isweep.value = w_isweep.value
    t_ron.value    = w_ron.value
    t_len.value    = w_len.value
    _syncing = False

    vg_label = "VSG" if device.value == "PMOS" else "VGS"
    vd_label = "VSD" if device.value == "PMOS" else "VDS"
    status_label.value = (
        f"<span style='color:#00ffb2'>✔ {len(df_use):,} rows &nbsp;|"
        f"&nbsp;{device.value} @ {vg_label}={vg_select.value:.4g} V, "
        f"{vd_label}={vd_select.value:.4g} V, {corner_select.value} &nbsp;|"
        f"&nbsp;i-sweep: {len(is_vals)} pts &nbsp;|"
        f"&nbsp;Ron_Gm: {len(ron_vals_cur)} pts &nbsp;|"
        f"&nbsp;lengths: {[round(l*1e6,2) for l in len_vals]} µm</span>"
    )
    trigger_update()


def trigger_update(change=None):
    if len(w_isweep.options) == 0 or w_isweep.options == [0]:
        return
    render(device.value, w_isweep.value, w_ron.value, w_len.value)


w_isweep.observe(trigger_update, names='value')
w_ron   .observe(trigger_update, names='value')
w_len   .observe(trigger_update, names='value')
device       .observe(update_data, names='value')
vg_select    .observe(update_data, names='value')
vd_select    .observe(update_data, names='value')
corner_select.observe(update_data, names='value')

print(" Callbacks wired")



## **How to Use the Ron/gm Design helper**

---


This tool helps user to size the dynamic amplifier based on $R_{on}/G_m$ based design methodology.

**Step-by-step:**
1. **Select Device** → choose `NMOS`, `PMOS`, or `BOTH`. The VGS/VSG and VDS/VSD dropdowns update automatically to valid sweep values for that device.
2. **Select Corner** → `TT` (typical), `SS` (slow and worst Ron), or `FF` (fast and best Ron). For worst case design, use SS or accordingly.
3. **Select VGS / VSG** → the gate to source voltage. For NMOS this is VGS; for PMOS it is VSG (both positive when the device is on).
4. **Select VDS / VSD** → the drain to source voltage. Use small values (e.g., 0.05 V) to characterise the linear-region $R_{on}$.
5. **Set i-sweep** → the bias current value. Use the slider or type directly in the text box. This corresponds to $I_{bias}$ in the replica path.
6. **Set Ron/Gm** → your target $R_{on}/g_m$ value in Ω·(A/V)⁻¹. Read your required value from the design methodology plots above.
7. **Set length** → target channel length. Longer channels give higher $R_{on}$ and lower $g_{ds}$; shorter channels are faster.

**Understanding the Plots:**

- **Scatter (Ron_Gm vs i-sweep):** Each dot is one simulation point, coloured by channel length (plasma colourmap: **purple = short L** → **yellow = long L**). The **▲ amber triangle** is your query point. The **★ green star** is the nearest match in the LUT. The **red dashed line** marks your Ron/Gm target.
- **Right: Loss Surface:** The cost function sweeping along i-sweep, with Ron/Gm and length held at best-match values. The minimum (green dot) is the optimal bias current. A narrow, deep minimum means the solution is well-determined; a flat curve means the result is insensitive to i-sweep and any value in that region is acceptable.

**Colour Coding in Top-5 Table:**
- 🟢 **Green** - best match (lowest distance to your query)
- 🟡 **Amber** - moderate match
- 🔴 **Red** - worst of the top 5

**Key Output BEST MATCH:**  
After the user input, the printed output shows `width`, `length`, and `v(Vbias)`. The transistor sizing and bias voltage to use in your design. The **Match Quality %** indicates how closely the LUT point matches your target (100% = exact match).



In [ ]:
# DISPLAY DASHBOARD
header = widgets.HTML(
    "<h2 style='color:#0f172a; font-family:monospace; margin:0'> Ron/Gm Design Helper </h2>"
    "<p style='color:#2563eb; margin:2px 0 8px 0; font-size:13px'>"
    "Real SPICE CSV data · dark-theme plots · live sliders · type a value or drag</p>"
)

controls_top = widgets.HBox(
    [vg_select, vd_select, corner_select],
    layout=widgets.Layout(gap='12px', margin='4px 0')
)

# Each row: label | slider | text input
def _row(label_str, slider, text):
    lbl = widgets.HTML(
        f"<span style='color:#94a3b8; font-family:monospace; "
        f"font-size:12px; min-width:70px; display:inline-block'>{label_str}</span>"
    )
    return widgets.HBox([lbl, slider, text],
                        layout=widgets.Layout(align_items='center', gap='8px'))

controls_bot = widgets.VBox([
    _row("i-sweep",  w_isweep, t_isweep),
    _row("Ron/Gm",   w_ron,    t_ron),
    _row("length",   w_len,    t_len),
], layout=widgets.Layout(margin='4px 0'))

display(
    header,
    device,
    controls_top,
    controls_bot,
    status_label,
    out
)

update_data()


## Ron/Gm Helper Output → Transistor Sizing
---

We design it for **SS corner** (worst-case $R_{on}$):

- Target $R_{on}/g_m = 50\times10^6$, $I_{bias} = 0.5\,\mu\text{A}$  

- Since there is **no LUT data for $I_{sweep} = 0.5\,\mu$A**, we scale from the minimum available data:
  - Available: $I_{sweep} = 5\,\mu$A  
  - Required: $I_{sweep} = 0.5\,\mu$A  
  - Scaling factor = **0.1× → Width scales by 0.1**

---

### Reference from Ron/gm Helper ($I_{sweep} = 5\,\mu$A)

- **NMOS:** $W = 5\,\mu\text{m},\; L = 4\,\mu\text{m}$  
- **PMOS:** $W = 1\,\mu\text{m},\; L = 0.2\,\mu\text{m}$  

---

### Scaled Design ($I_{sweep} = 0.5\,\mu$A)

Applying **width scaling (×0.1):**

- **NMOS:**  
  $W = 0.5\,\mu\text{m},\; L = 3\text{–}4\,\mu\text{m}$  
  $V_{DZN} \approx 0.34\,\text{V}$ (TT)

- **PMOS:**  
  $W = 0.1\,\mu\text{m} \approx 0.15\,\mu\text{m},\; L = 0.2\,\mu\text{m}$  
  $V_{DZP} \approx 1.11\,\text{V}$ (TT)

---

These values are **consistent with gm/Id estimates**, while additionally providing:
- Direct control over $R_{on}/g_m$
- Deadzone bias voltages at **SS corner**

---

### Peak Current (Worst-Case)

- I_peak(maximum Non-linear current) ≈ 14 µA (pre-simulation prediction)
Post-simulation: ≈ 12 µA (≈15% deviation)

---

### Key Insight

> Unlike gm/Id, this method allows **direct scaling across current levels** while preserving $R_{on}/g_m$, enabling reliable sizing even when LUT data is unavailable at the target bias.

---


# 6.2 Inverter Based Amplifier designed using Ron/Gm based

#### Results for the above sizes that were obtained based on Ron/Gm based methodology

## Schematic
Schematic of the inverter-based amplifier with capacitive feedback, designed using the $R_{on}/g_m$ methodology:

<p align="center">
  <img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/ckt_images/fig_13_IBA_with_cap_FB_using_ron_gm.jpg" alt="INV OTA Design" width="1100"><br>
  <em><strong>Fig. 15:</strong> Inverter-based amplifier in capacitive feedback configuration designed using the $R_{on}/g_m$ methodology.</em>
</p>


## 6.3 Results from the IBA designed using **$g_m/I_D$** design methodology

The four figures below are generated from `INV_OTA_gmid.txt`, a transient simulation of the IBA (Inverter-Based Amplifier) sized using the $R_{on}/g_m$ methodology.

(LV_NMOS: $W = 0.3\,\mu\text{m}$, $L = 3.0\,\mu\text{m}$, $m = 4$; LV_PMOS: $W = 0.15\,\mu\text{m}$, $L = 0.2\,\mu\text{m}$, $m = 4$; $I_{q,replica} = 0.5\,\mu\text{A}$, $I_{q,signal} = 2.0\,\mu\text{A}$)

In [ ]:
# LOAD SETTLING DATA — Ron/Gm designed IBA
#
# FILE: INV_OTA_rongm.txt
#   Transient data for IBA sized with Ron/Gm methodology:
#     PMOS W=6.1µm L=0.5µm m=4;  NMOS W=3.3µm L=1µm m=4; Ron target=8kΩ.
# VOUT_FINAL_ROGM = 0.82288 V (different from gm/ID design due to diff W/L ratios).
# Reuses same START_TIME/END_TIME window as gm/ID for fair comparison.
# _sa_load_data() → (t_r, v_r, m_r)
# _sa_extract_settling() → (st_us_r, sv_r, sm_r, err_r, dvdt_r)
#
FILE_ROGM       = os.path.join(BASE_DIR, "INV_OTA_rongm.txt")
VOUT_FINAL_ROGM = 0.66022   # V  <- update if different from gm/ID

print(f"File    : {FILE_ROGM}")
print(f"Exists  : {os.path.exists(FILE_ROGM)}")

if not os.path.exists(FILE_ROGM):
    raise FileNotFoundError(
        f"Data file not found: {FILE_ROGM}\n"
        "Re-run Cell 54 to clone/pull the repo and ensure sparse-checkout fetched INV_OTA_ERROR_PLOTS/"
    )
t_r, v_r, m_r = _sa_load_data(FILE_ROGM)
print(f"Parsed  : {len(t_r)} data points")

st_us_r, sv_r, sm_r, err_r, dvdt_r = _sa_extract_settling(
    t_r, v_r, m_r, VOUT_FINAL_ROGM, START_TIME, END_TIME
)
print(f"Settling: {len(st_us_r)} points in [{START_TIME*1e6:.2f}, {END_TIME*1e6:.1f}] \u03bcs")

In [ ]:
# FIGURE 16: Vout waveform — Ron/Gm designed IBA
# Identical structure to Figure 10 but using Ron/Gm design data.
# VOUT_FINAL_ROGM = 0.82288 V (higher than gm/ID's 0.55731 V due to different
# PMOS/NMOS sizing ratios from the Ron/Gm methodology).
#
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=st_us_r, y=sv_r, mode='lines',
    line=dict(color=_SA_COLORS['vout'], width=3),
    name='V<sub>out</sub>',
    hovertemplate='t = %{x:.3f} \u03bcs<br>V<sub>out</sub> = %{y:.5f} V<extra></extra>'
))
fig.add_hline(
    y=VOUT_FINAL_ROGM,
    line=dict(color=_SA_COLORS['vfinal'], width=1.8, dash='dot'),
    annotation_text=f'V<sub>final</sub> = {VOUT_FINAL_ROGM:.5f} V',
    annotation_font=dict(size=_SA_FS['annotation'], color=_SA_COLORS['vfinal']),
    annotation_position='top left'
)
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode='lines',
    line=dict(color=_SA_COLORS['vfinal'], width=1.8, dash='dot'),
    name=f'V<sub>final</sub> = {VOUT_FINAL_ROGM:.5f} V'
))
fig.update_layout(
    **_SA_LAYOUT,
    title=dict(text='<b>Figure 16:</b> Output voltage (Vout) waveform of the system for IBA using R<sub>on</sub>/G<sub>m</sub> based design.', font=dict(size=_SA_FS['title']), x=0.02),
    xaxis=_sa_styled_axis('Time (\u03bcs)'),
    yaxis=_sa_styled_axis('V<sub>out</sub> (V)'),
)
fig.show()

In [ ]:
# FIGURE 17: Log settling error — Ron/Gm designed IBA
# Filters NaN/Inf/zero error samples; constructs log decade tick marks.
# _sa_try_exp_fit() overlays fitted e^(-t/τ) dashed curve and prints τ.
# τ(Ron/Gm) is the small-signal settling time constant of the Ron/Gm design.
# Nonlinear settling phase is highlighted for both positive and negative edges.
# If τ(Ron/Gm) ≈ τ(gm/ID): both methodologies achieve the same small-signal
# bandwidth, confirming Ron/Gm provides additional RC settling information
# without sacrificing settling accuracy.
#
valid_r      = ~np.isnan(err_r) & ~np.isinf(err_r) & (np.abs(err_r) > 0)
valid_t_r    = st_us_r[valid_r]
valid_err_r  = np.abs(err_r[valid_r])
if len(valid_err_r) == 0:
    print("⚠️  No valid (non-zero, finite) error samples found — skipping error plot.")
    valid_err_r = np.array([1e-6])
    valid_t_r   = np.array([0.0])
_emin_r = valid_err_r.min()
_emax_r = valid_err_r.max()
log_min_r = int(np.floor(np.log10(_emin_r))) if _emin_r > 0 else -6
log_max_r = int(np.ceil(np.log10(_emax_r)))  if _emax_r > 0 else  0

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=valid_t_r, y=valid_err_r, mode='lines',
    line=dict(color=_SA_COLORS['error'], width=3),
    name='Settling Error',
    hovertemplate='t = %{x:.3f} \u03bcs<br>Error = %{y:.2e}<extra></extra>'
))
tau_r = _sa_try_exp_fit(fig, valid_t_r, valid_err_r)
if tau_r: print(f'\u03c4 (Ro/gm) = {tau_r:.3f} \u03bcs')

# ── highlight nonlinear settling phase — both edges, all cycles ─────────────
_period      = 1.0    # µs — full clock period
_nl_duration = 0.05   # µs — 50 ns nonlinear phase per edge
_t_max       = valid_t_r.max()
_n_cycles    = int(np.ceil(_t_max / _period))

# explicitly distinct colors — orange for +edge, steel blue for -edge
_edges = [
    (0.5, 'rgba(210, 105,  30, 0.25)', 'rgba(210, 105,  30, 1.0)', '+edge nonlinear Phase'),
    (1.0, 'rgba( 30, 144, 255, 0.25)', 'rgba( 30, 144, 255, 1.0)', '−edge nonlinear Phase'),
]

for _offset, _fill, _line_color, _label in _edges:
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(size=10, color=_fill, symbol='square'),
        name=_label, showlegend=True,
    ))
    for i in range(_n_cycles):
        _x0 = i * _period + _offset
        _x1 = _x0 + _nl_duration
        if _x0 >= _t_max:
            break
        _x1 = min(_x1, _t_max)
        fig.add_vrect(
            x0=_x0, x1=_x1,
            fillcolor=_fill,
            layer='below', line_width=0,
        )
        fig.add_vline(x=_x0, line=dict(color=_line_color, width=1.5, dash='dash'))
        fig.add_vline(x=_x1, line=dict(color=_line_color, width=1.5, dash='dash'))
# ────────────────────────────────────────────────────────────────────────────

fig.update_layout(
    **_SA_LAYOUT,
    title=dict(
        text='<b>Figure 17:</b> Logarithmic error of the output with respect to reference for IBA using R<sub>on</sub>/G<sub>m</sub> based design.',
        font=dict(size=_SA_FS['title']), x=0.02
    ),
    xaxis=_sa_styled_axis('Time (\u03bcs)'),
    yaxis=dict(
        **_sa_styled_axis('|V<sub>out,final</sub> − V<sub>out</sub>| / V<sub>out</sub>'),
        type='log',
        range=[log_min_r-1, log_max_r+1],
        tickvals=[10**i for i in range(log_min_r, log_max_r+1)],
        ticktext=[f'10<sup>{i}</sup>' for i in range(log_min_r, log_max_r+1)],
    ),
)
fig.show()

It is observed that the error envelope exhibits **two distinct settling regimes**: an initial rapid decay with a **steep negative slope (500–550 ns)** attributable to **large-signal nonlinear operation**, followed by a gradual exponential decay with a **comparatively smaller slope**, characteristic of **small-signal linear settling**. It is worth noting that the boundary between
these two regimes, and consequently the proportion of the settling window occupied by the nonlinear phase, **determined beforehand from $R_{on}$/$g_m$ design methodology.** This information was ready with users before simulation analysis. This constitutes a
**fundamental difference and improvement over conventional design methodology**: the large-signal and
small-signal settling contributions **can be independently budgeted or optimized** at the
transistor sizing stage.

In [ ]:
# FIGURE 18: Output current Iout — Ron/Gm designed IBA
# sm_r × 1e6 converts branch current (A) to µA for display.
# Large initial spike = large-signal phase (high slew current).
# Exponential decay = small-signal phase (NMOS/PMOS currents balancing).
# Compare peak spike vs Figure 13 (gm/ID): sharper peak reflects lower Ron
# achieved by Ron/Gm sizing → faster large-signal capacitor charging.
#
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=st_us_r, y=sm_r * 1e6, mode='lines',
    line=dict(color=_SA_COLORS['iout'], width=3),
    name='I<sub>out</sub>',
    hovertemplate='t = %{x:.3f} \u03bcs<br>I<sub>out</sub> = %{y:.3f} \u03bcA<extra></extra>'
))

# ── highlight nonlinear vs linear phase — both edges, all cycles ─────────────
_period      = 1.0    # µs — full clock period
_nl_duration = 0.05   # µs — 50 ns nonlinear phase per edge
_t_max       = st_us_r.max()
_n_cycles    = int(np.ceil(_t_max / _period))

_edges = [
    (0.5, 'rgba(210, 105,  30, 0.25)', 'rgba(210, 105,  30, 1.0)', '+edge nonlinear Phase'),
    (1.0, 'rgba( 30, 144, 255, 0.25)', 'rgba( 30, 144, 255, 1.0)', '−edge nonlinear Phase'),
]

for _offset, _fill, _line_color, _label in _edges:
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(size=10, color=_fill, symbol='square'),
        name=_label, showlegend=True,
    ))
    for i in range(_n_cycles):
        _x0 = i * _period + _offset
        _x1 = _x0 + _nl_duration
        if _x0 >= _t_max:
            break
        _x1 = min(_x1, _t_max)
        fig.add_vrect(
            x0=_x0, x1=_x1,
            fillcolor=_fill,
            layer='below', line_width=0,
        )
        fig.add_vline(x=_x0, line=dict(color=_line_color, width=1.5, dash='dash'))
        fig.add_vline(x=_x1, line=dict(color=_line_color, width=1.5, dash='dash'))
# ────────────────────────────────────────────────────────────────────────────

fig.update_layout(
    **_SA_LAYOUT,
    title=dict(
        text='<b>Figure 18:</b> — Output Current I<sub>out</sub> across the circuit branch for IBA using R<sub>on</sub>/G<sub>m</sub> based design.',
        font=dict(size=_SA_FS['title']), x=0.02
    ),
    xaxis=_sa_styled_axis('Time (\u03bcs)'),
    yaxis=_sa_styled_axis('I<sub>out</sub> (\u03bcA)'),
)
fig.show()

It is observed that the output charging current $I_{out}$ exhibits **two distinct regimes**: an initial **large nonlinear peak current** measured at **+14 µA** on the positive edge and **−13.86 µA** on the negative edge that rapidly charges/discharges the output load capacitance toward the final value, followed by a progressively decreasing small-signal current with $I_{out} \rightarrow 0$ as the output converges. This behavior is characteristic of dynamic amplifiers: **the bulk of the charge transfer occurs in the initial nonlinear phase**, with the small-signal tail contributing only the final fine settling. This was **determined beforehand by the $R_{on}/g_m$ methodology where $g_m/I_d$ or the approach of Conrad et al failed to do so. [4]**. Furthermore, this **peak current varies across process corners**, introducing additional uncertainty in the settling budget was predicted before simulation analysis. This represents a great improvement over conventional methodologies: not only is the nonlinear settling duration is known at the sizing stage, but so is the **magnitude, polarity asymmetry, and corner dependence** of the dominant charging current.


In [ ]:
# FIGURE 19: dVout/dt — Ron/Gm designed IBA
# dvdt_r (V/µs) vs time.  Peak = Ipeak/C_load (slew rate of large-signal phase).
# Compare peak magnitude and timing with Figure 13 (gm/ID design).
# Mirrors Figure 13 structure; reveals settling dynamics of Ron/Gm-sized IBA.
#
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=st_us_r, y=dvdt_r, mode='lines',
    line=dict(color=_SA_COLORS['dvdt'], width=3),
    name='dV<sub>out</sub>/dt',
    hovertemplate='t = %{x:.3f} \u03bcs<br>dV/dt = %{y:.4f} V/\u03bcs<extra></extra>'
))
fig.update_layout(
    **_SA_LAYOUT,
    title=dict(text='<b>Figure 19:</b> Derivative of output voltage dV<sub>out</sub>/dt for IBA using using R<sub>on</sub>/G<sub>m</sub> based design.', font=dict(size=_SA_FS['title']), x=0.02),
    xaxis=_sa_styled_axis('Time (\u03bcs)'),
    yaxis=_sa_styled_axis('dV<sub>out</sub>/dt (V/\u03bcs)'),
)
fig.show()

<hr>

# 7. Comparative Analysis

A structured comparison of the $R_{on}/g_m$ methodology against the $g_m/I_D$ methodology and the Conrad *et al.* [TCAS-I 2020] numerical optimizer is presented below. The table covers methodology paradigm, settling phase coverage, process corner robustness, design equations, practical flow, and concrete design example results across all three approaches.

---


## PERFORMANCE SUMMARY AND COMPARISON WITH STATE-OF-THE-ART

<div style="overflow-x:auto; width:100%;">
<table style="width:100%; border-collapse:collapse; border-top:2px solid #000; border-bottom:2px solid #000; table-layout:fixed; font-family:'Times New Roman',serif; font-size:9pt;">
<colgroup><col style="width:28%;"><col style="width:24%;"><col style="width:24%;"><col style="width:24%;"></colgroup>
<thead>
<tr>
  <th style="text-align:left; padding:6px 7px; border-bottom:1px solid #000; font-weight:bold; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;">Design Criterion</th>
  <th style="text-align:center; padding:6px 7px; border-bottom:1px solid #000; font-weight:bold; background:#f0f0f0; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;">$R_{on}/g_m$ <b>(This Work)</b></th>
  <th style="text-align:center; padding:6px 7px; border-bottom:1px solid #000; font-weight:bold; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;">$g_m/I_D$ Methodology</th>
  <th style="text-align:center; padding:6px 7px; border-bottom:1px solid #000; font-weight:bold; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;">Conrad <i>et al.</i> [TCAS-I 2020]</th>
</tr>
</thead>
<tbody>

<!-- ── I ── -->
<tr><td colspan="4" style="background:#000; color:#fff; font-weight:bold; font-size:8.5pt; letter-spacing:.06em; padding:3px 8px; max-width:none;">I. &nbsp;METHODOLOGY</td></tr>

<tr>
  <td style="text-align:left; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;">Design paradigm</td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0; background:#fafafa;">Analytical LUT<br><small><i>Physics-based; pre-characterised</i></small></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;">Analytical LUT<br><small><i>Small-signal only</i></small></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;">Simulation-based numerical optimizer<br><small><i>Cadence ADE XL</i></small></td>
</tr>

<tr>
  <td style="text-align:left; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;">Primary design parameter</td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0; background:#fafafa;">$R_{on}/g_m$<br><small><i>Couples large-signal + small-signal</i></small></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;">$g_m/I_D$<br><small><i>Small-signal only</i></small></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;">$m_{a1},\, m_{a2},\, m_{a3},\, I_{bias},\, R,\, C_{az}$<br><small><i>6 numerical params; no physical link</i></small></td>
</tr>

<tr>
  <td style="text-align:left; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;">Design entry stage</td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0; background:#fafafa;">Pre-simulation</td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;">Pre-simulation<br><small><i>Partial — no large-signal info</i></small></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;">Requires full transient simulation per iteration</td>
</tr>

<!-- ── II ── -->
<tr><td colspan="4" style="background:#000; color:#fff; font-weight:bold; font-size:8.5pt; letter-spacing:.06em; padding:3px 8px; max-width:none;">II. &nbsp;SETTLING PHASE COVERAGE</td></tr>

<tr>
  <td style="text-align:left; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;">Non-linear (large-signal / RC) settling characterized</td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0; background:#fafafa;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/happy.svg" width="22" height="22"/><br><small><i>Via $R_{on}$ in LUT; $\tau_{ls} = R_{on}C_L$</i></small></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/sad.svg" width="22" height="22"/><br><small><i>$R_{on}$ absent from $g_m/I_D$ framework</i></small></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/sad.svg" width="22" height="22"/><br><small><i>Acknowledged as "not practical" [1]†</i></small></td>
</tr>

<tr>
  <td style="text-align:left; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;">Small-signal ($g_m C$) settling characterized</td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0; background:#fafafa;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/happy.svg" width="22" height="22"/><br><small><i>Via $g_{m,bias}$ in LUT; $BW = g_m/(2\pi C_L)$</i></small></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/happy.svg" width="22" height="22"/><br><small><i>Primary strength of $g_m/I_D$ methodology</i></small></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/happy.svg" width="22" height="22"/><br><small><i>Via rms error cost function</i></small></td>
</tr>

<tr>
  <td style="text-align:left; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;">Unified two-phase settling model</td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0; background:#fafafa;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/happy.svg" width="22" height="22"/><br><small><i>$R_{on}/g_m$ co-constrains both $\tau_{LS}$ and $\tau_{SS}$</i></small></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/sad.svg" width="22" height="22"/><br><small><i>Small-signal-phase only</i></small></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/sad.svg" width="22" height="22"/><br><small><i>No time-budget allocation between phases</i></small></td>
</tr>

<tr>
  <td style="text-align:left; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;">Peak slew current $I_{peak}$ predicted pre-simulation</td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0; background:#fafafa;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/happy.svg" width="22" height="22"/><br><small><i>$I_{peak} \propto 1/(R_{on}/g_m)$; log-log scale slope $= -1$</i></small></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/sad.svg" width="22" height="22"/></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/sad.svg" width="22" height="22"/><br><small><i>Post-optimization verification only</i></small></td>
</tr>

<!-- ── III ── -->
<tr><td colspan="4" style="background:#000; color:#fff; font-weight:bold; font-size:8.5pt; letter-spacing:.06em; padding:3px 8px; max-width:none;">III. &nbsp;PROCESS CORNER ROBUSTNESS</td></tr>

<tr>
  <td style="text-align:left; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;">TT / SS / FF corners visible at design entry</td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0; background:#fafafa;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/happy.svg" width="22" height="22"/><br><small><i>All 3 corners in LUT; SS worst-case directly readable</i></small></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/sad.svg" width="22" height="22"/><br><small><i>LUT can be corner-swept but $R_{on}$ not extracted</i></small></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/sad.svg" width="22" height="22"/><br><small><i>PVT excluded from optimizer loop‡</i></small></td>
</tr>

<tr>
  <td style="text-align:left; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;">Deadzone bias ($V_{DZN}$, $V_{DZP}$) determined pre-simulation</td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0; background:#fafafa;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/happy.svg" width="22" height="22"/><br><small><i>Per-corner, from $V_{bias}$ vs $R_{on}/g_m$ plot§</i></small></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/sad.svg" width="22" height="22"/><br><small><i>Unknown without iterative transient corner sweeps</i></small></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/sad.svg" width="22" height="22"/><br><small><i>$V_{os}$ is one of 6 optimized params; corner sensitivity unknown a priori</i></small></td>
</tr>

<tr>
  <td style="text-align:left; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;">SS-corner $R_{on}$ degradation visible a priori</td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0; background:#fafafa;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/happy.svg" width="22" height="22"/></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/sad.svg" width="22" height="22"/></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/sad.svg" width="22" height="22"/></td>
</tr>

<!-- ── IV ── -->
<tr><td colspan="4" style="background:#000; color:#fff; font-weight:bold; font-size:8.5pt; letter-spacing:.06em; padding:3px 8px; max-width:none;">IV. &nbsp;DESIGN EQUATIONS AND PHYSICAL INSIGHT</td></tr>

<tr>
  <td style="text-align:left; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;">Closed-form design equations available</td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0; background:#fafafa;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/happy.svg" width="22" height="22"/><br><small><i>$V_{bias} = V_{TH} + 2I_D/g_m$;<br>$g_{m,bias} \propto (R_{on}/g_m)^{-1/3}$;<br>$R_{on}/g_m \propto L^2/(W I_D)$</i></small></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/happy.svg" width="22" height="22"/><br><small><i>For small-signal: $G_m = 2\pi f C_L$;<br>$W = I_D / (I_D/W)|_{LUT}$</i></small></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/sad.svg" width="22" height="22"/><br><small><i>Stability criterion exists but "not realizable" as design equation†</i></small></td>
</tr>

<tr>
  <td style="text-align:left; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;">Physical insight preserved throughout design</td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0; background:#fafafa;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/happy.svg" width="22" height="22"/><br><small><i>Every plot axis maps to a physical quantity</i></small></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/happy.svg" width="22" height="22"/><br><small><i>For small-signal operating point only</i></small></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/sad.svg" width="22" height="22"/><br><small><i>Optimizer returns numbers; no physical link retained</i></small></td>
</tr>

<tr>
  <td style="text-align:left; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;">Subsumes $g_m/I_D$ methodology</td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0; background:#fafafa;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/happy.svg" width="22" height="22"/><br><small><i>$g_m/I_D$ readable from $g_m/I_D$ vs $R_{on}/g_m$ plot; reverse not possible</i></small></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;">—</td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/sad.svg" width="22" height="22"/></td>
</tr>

<!-- ── V ── -->
<tr><td colspan="4" style="background:#000; color:#fff; font-weight:bold; font-size:8.5pt; letter-spacing:.06em; padding:3px 8px; max-width:none;">V. &nbsp;PRACTICAL DESIGN FLOW</td></tr>

<tr>
  <td style="text-align:left; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;">EDA tool requirement</td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0; background:#fafafa;">Open-source and Commercial tools<br><small><i>Any spice + Python/Matlab</i></small></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;">Spectre (LUT gen.)<br><small><i>Helper usable standalone</i></small></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;">Cadence Virtuoso + Spectre + ADE XL<br><small><i>Full commercial license stack</i></small></td>
</tr>

<tr>
  <td style="text-align:left; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;">Application-specific redesign required</td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0; background:#fafafa;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/sad.svg" width="22" height="22"/><br><small><i>LUT generated once per technology node</i></small></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/sad.svg" width="22" height="22"/><br><small><i>LUT reusable across applications</i></small></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/happy.svg" width="22" height="22"/><br><small><i>New testbench + cost function per application</i></small></td>
</tr>

<tr>
  <td style="text-align:left; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;">Cost function calibration required</td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0; background:#fafafa;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/sad.svg" width="22" height="22"/></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/sad.svg" width="22" height="22"/></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/happy.svg" width="22" height="22"/><br><small><i>Manual ×4 adjustment applied in paper¶</i></small></td>
</tr>

<tr>
  <td style="text-align:left; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;">PVT robustness in design loop</td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0; background:#fafafa;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/happy.svg" width="22" height="22"/><br><small><i>SS/FF corner LUTs included by design</i></small></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/sad.svg" width="22" height="22"/><br><small><i>Corner LUT possible; dynamic behavior not covered</i></small></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/sad.svg" width="22" height="22"/><br><small><i>Excluded from optimizer; post-hoc Monte Carlo only‡</i></small></td>
</tr>

<tr>
  <td style="text-align:left; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;">Technology portability</td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0; background:#fafafa;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/happy.svg" width="22" height="22"/><br><small><i>15 ngspice sims re-characterize any PDK (~1 min)</i></small></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/happy.svg" width="22" height="22"/><br><small><i>New Spectre LUT per node</i></small></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; vertical-align:top; word-break:break-word; overflow-wrap:anywhere; white-space:normal; max-width:0;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/sad.svg" width="22" height="22"/><br><small><i>Full optimizer rerun needed; 180 nm fails power target by 13×**</i></small></td>
</tr>
<!-- ── VI ── -->
<tr><td colspan="4" style="background:#000; color:#fff; font-weight:bold; font-size:8.5pt; letter-spacing:.06em; padding:3px 8px; max-width:none;">VI. &nbsp;DESIGN EXAMPLE RESULTS (IBA, T<sub>settle</sub> = 250 ns, UGB = 9.55 MHz, C<sub>L,eff</sub> = 667 fF, IHP SG13G2 130 nm)</td></tr>

<tr>
  <td style="text-align:left; padding:5px 7px; border-bottom:0.5px solid #ddd;">$G_m$ target met (9.55 MHz BW)</td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; background:#fafafa;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/happy.svg" width="22"/> $\approx 40\,\mu\text{S}$</td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/happy.svg" width="22"/> $\approx 40\,\mu\text{S}$</td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/happy.svg" width="22"/> 15-bit ENOB</td>
</tr>

<tr>
  <td style="text-align:left; padding:5px 7px; border-bottom:0.5px solid #ddd;">$I_{peak}$ range</td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; background:#fafafa;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/happy.svg" width="22"/><br><small><i>Best: 14 $\mu$A | Worst: -13.6 $\mu$A</i></small></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/sad.svg" width="22"/></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/sad.svg" width="22"/></td>
</tr>

<tr>
  <td style="text-align:left; padding:5px 7px; border-bottom:0.5px solid #ddd;">$R_{on}$ known before transient simulation</td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; background:#fafafa;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/happy.svg" width="22"/><br><small><i>4 kΩ unit; 1 kΩ after ×4 multiplier</i></small></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/sad.svg" width="22"/><br><small><i>Post-simulation discovery only</i></small></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/sad.svg" width="22"/></td>
</tr>

<tr>
  <td style="text-align:left; padding:5px 7px; border-bottom:0.5px solid #ddd;">$V_{DZN}$ (NMOS deadzone) pre-simulation</td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; background:#fafafa;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/happy.svg" width="22"/><br><small><i>FF: 0.355 V | TT: 0.390 V | SS: 0.425 V</i></small></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/sad.svg" width="22"/></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/sad.svg" width="22"/></td>
</tr>

<tr>
  <td style="text-align:left; padding:5px 7px; border-bottom:0.5px solid #ddd;">$V_{DZP}$ (PMOS deadzone) pre-simulation</td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd; background:#fafafa;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/happy.svg" width="22"/><br><small><i>FF: 0.970 V | TT: 1.110 V | SS: 1.204 V</i></small></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/sad.svg" width="22"/></td>
  <td style="text-align:center; padding:5px 7px; border-bottom:0.5px solid #ddd;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/sad.svg" width="22"/></td>
</tr>

<tr>
  <td style="text-align:left; padding:5px 7px;">Extra transient sims needed to obtain $V_{DZ}$ and $R_{on}$</td>
  <td style="text-align:center; padding:5px 7px; background:#fafafa;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/happy.svg" width="22"/> <b>Zero</b></td>
  <td style="text-align:center; padding:5px 7px;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/sad.svg" width="22"/><br><small><i>Multiple corner sweeps required</i></small></td>
  <td style="text-align:center; padding:5px 7px;"><img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/sad.svg" width="22"/><br><small><i>Full optimizer run per corner</i></small></td>
</tr>

</tbody>
</table>

<div style="font-size:8pt; line-height:1.7; border-top:0.75px solid #000; margin-top:5px; padding-top:4px;">
All simulated results were within <b>&plusmn;75 mV</b> of the pre-simulation predictions, demonstrating excellent accuracy of the proposed methodology.
</div>

<p>† J. Conrad <i>et al.</i>: <i>"Unfortunately, Section II-B is not really practical for designing a RAMP… This makes a design-by-equation cumbersome and not realizable."</i> [TCAS-I 2020, Sec. II-C]</p>
<p>‡ J. Conrad <i>et al.</i>: <i>"PVT variations are not encountered during the circuit optimization, because this would require many transient simulations to evaluate one iteration of the optimizer."</i> [TCAS-I 2020, Sec. V-E]</p>
<p>§ Deadzone bias values read from the $V_{bias}$ vs. $\log(R_{on}/g_m)$ LUT plot at $R_{on}/g_m = 50 \times 10^6$, $I_{bias} = 2\,\mu$A.</p>
<p>¶ J. Conrad <i>et al.</i>: <i>"the optimizer goal for the accuracy was readjusted ×4 smaller, i.e. 0.25 for the cost function."</i> [TCAS-I 2020, Sec. IV-B.3]</p>
<p>** J. Conrad <i>et al.</i>, Table II: 180 nm power cost function = 12.99 (target: 1.0), failing the power constraint by 13×.</p>
<p style="margin-top:6px;">
  <img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/happy.svg" width="16" height="16"/> = fully supported / available pre-simulation &nbsp;&nbsp;
  <img src="https://raw.githubusercontent.com/chennakeshavadasa/gmid_IHP130/refs/heads/main/Plots_Images/Notebook_Figs/sad.svg" width="16" height="16"/> = not available / requires post-simulation discovery
</p>
</div>
</div>

---


### Key Observation
---

Both methodologies successfully meet the 20 MHz bandwidth specification the $g_m/I_D$ design (NMOS: $W = 0.3\,\mu$m, $L = 3\,\mu$m, $m = 4$), (PMOS: $W = 0.15\,\mu$m, $L = 0.25\,\mu$m, $m = 4$) and the $R_{on}/g_m$ design (NMOS: $W = 0.5\,\mu$m, $L = 3\,\mu$m, $m = 4$), (PMOS: $W = 0.15\,\mu$m, $L = 0.2\,\mu$m, $m = 4$) both settle correctly, as confirmed by Figures 10–14 and 15–19.

The critical difference lies in **what the designer knows before running the transient simulation.** With the $g_m/I_D$ approach, the on-resistance $R_{on}$ of the output devices, the deadzone voltage ranges $V_{DZP}/V_{DZN}$, and the worst-case SS-corner behaviour are all unknown at design entry, they are discovered only after running additional SPICE sweeps. With the $R_{on}/g_m$ approach, all three are read directly from the LUT plots at the point of sizing.

This distinction becomes especially significant for dynamic amplifier design, where settling involves two sequential phases. The $g_m/I_D$ methodology was not designed with this two-phase behaviour in mind, and as a result leaves the large-signal RC phase uncharacterised. The $R_{on}/g_m$ methodology treats both phases as main design variables from the outset, reducing the number of design iterations required to reach a corner-robust, fully specified operating point.

<hr>

### Limitations

- **No physical layout:** Device sizes and bias voltages are schematic-level. Silicon area and post-layout parasitic impact on $R_{on}/g_m$ are not verified yet.
- **No intra-die mismatch:** Monte Carlo mismatch analysis across devices within the same corner is not included. Corner analysis (TT/SS/FF) covers inter-die process variation only.

---
> An interactive version of all design plots is available at **[wrongm.azurewebsites.net](https://wrongm.azurewebsites.net)**.


---
## References
---

[1] B. Hershberg, S. Weaver, K. Sobue, S. Takeuchi, K. Hamashita and U. -K. Moon, "Ring amplifiers for switched-capacitor circuits," 2012 IEEE International Solid-State Circuits Conference, San Francisco, CA, USA, 2012, pp. 460-462, doi: [10.1109/ISSCC.2012.6177090](https://doi.org/10.1109/ISSCC.2012.6177090).

[2] Venkatachala, Praveen Kumar. 2019. *[Design Considerations and Circuit Techniques for Robust Ring-Amplifiers](https://ir.library.oregonstate.edu/downloads/rr1724065)*. Oregon State University.

[3] Brooks, Lane & Lee, Hae-Seung. (2007). "A zero-crossing-based 8b 200MS/s pipelined ADC." *IEEE International Solid-State Circuits Conference, 2007. ISSCC 2007. Digest of Technical Papers*. 460-615. doi: [10.1109/ISSCC.2007.373493](https://doi.org/10.1109/ISSCC.2007.373493).

 [4] J. Conrad, P. Vogelmann, M. A. Mokhtar and M. Ortmanns, "Design Approach for Ring Amplifiers," in IEEE Transactions on Circuits and Systems I: Regular Papers, vol. 67, no. 10, pp. 3444-3457, Oct. 2020, doi: [10.1109/TCSI.2020.2986553](https://ieeexplore.ieee.org/document/9076616).

 [5] P. R. Kinget, "Scaling analog circuits into deep nanoscale CMOS: Obstacles and ways to overcome them," 2015 IEEE Custom Integrated Circuits Conference (CICC), San Jose, CA, USA, 2015, pp. 1-8, doi: [10.1109/CICC.2015.7338394](https://ieeexplore.ieee.org/document/7338394).

 [6] J. Annema, B. Nauta, R. van Langevelde and H. Tuinhout, "Analog circuits in ultra-deep-submicron CMOS," in IEEE Journal of Solid-State Circuits, vol. 40, no. 1, pp. 132-143, Jan. 2005, doi: [10.1109/JSSC.2004.837247](https://ieeexplore.ieee.org/document/1374997).

 [7] B. Razavi, Design of Analog CMOS Integrated Circuits, 2nd ed., McGraw-Hill, 2017. (Ch. 12 Switched-Capacitor Circuits).

 [8] Y. Chae and G. Han, "Low Voltage, Low Power, Inverter-Based Switched-Capacitor Delta-Sigma Modulator," in IEEE Journal of Solid-State Circuits, vol. 44, no. 2, pp. 458-472, Feb. 2009, doi: [10.1109/JSSC.2008.2010973](https://ieeexplore.ieee.org/document/4768910).

 ---